# DataRetail 360 · Pipeline ETL, modelo dimensional y almacenamiento analítico

**IDY1102 · Estructuras y Visualización de Datos · Evaluación Parcial 3 / 4**
Jennifer Albornoz · Francisca Blanchard · Sección 001_V

Este notebook ejecuta **todo el proceso en un solo flujo**, de principio a fin:

| Sección | Qué hace | Indicador |
|---|---|---|
| 1. Configuración | Parámetros, conexión a SQL Server (o SQLite de respaldo), logging | IL 2.3 |
| 2. Fuentes | Genera los archivos de los sistemas de origen (POS, e-commerce, CRM, inventario…) con errores de calidad reales | — |
| 3. Estructuras | Crea los esquemas `bronze`, `silver`, `gold`, `audit`, `sem` y todas las tablas | IL 2.1 · IL 2.4 |
| 4. Framework del pipeline | Registro de etapas, auditoría, rechazos, transacciones | IL 2.3 |
| 5. Bronze | Ingesta cruda y trazable | IL 2.4 |
| 6. Silver | Limpieza, validación, integración, seudonimización, regla de venta neta | IL 2.3 |
| 7. Gold · modelo dimensional | Dimensiones con SCD 1, 2 y 3 y dos tablas de hechos | IL 2.1 · IL 2.2 |
| 8. Orquestación | Ejecuta el lote 1 (ene–jun) y el lote 2 (jul–dic, con cambios en los maestros) | IL 2.3 |
| 9. Gold · data marts | 4 data marts por área + vista integrada + capa semántica para IA | IL 2.1 · IL 2.4 |
| 10. Controles de calidad | Reconciliación, unicidad, huérfanos, vigencias, cuadres, privacidad | IL 2.4 |
| 11. Evidencias | Linaje, efecto SCD, idempotencia, manejo de errores, consultas de negocio | IL 2.2 · IL 2.3 · IL 2.4 |

**Arquitectura Medallion en SQL Server**

```
Fuentes (CSV)  ──►  bronze.*  ──►  silver.*  ──►  gold.dim_* / gold.fact_*  ──►  gold.dm_* (4 data marts)  ──►  Power BI + Copilot
 POS, web,          crudo,        limpio,         modelo dimensional              Comercial · Marketing          sem.* (capa semántica
 CRM, stock…        trazable      validado        Kimball (SCD 1/2/3)             Finanzas · Operaciones          para consultas con IA)
                                                                   audit.*  (log de ejecución, rechazos, controles de calidad)
```

**Cómo ejecutarlo:** `Kernel → Restart & Run All`. La primera celda de configuración define si se usa SQL Server o SQLite.

## 1. Configuración, conexión y logging

In [ ]:
import os, io, re, json, time, uuid, hashlib, logging, warnings
from contextlib import contextmanager
from datetime import date, datetime, timedelta
from decimal import Decimal
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')

# --- Rutas del proyecto (estructura del repositorio) ---
BASE = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DIR_FUENTE = BASE / 'data' / 'fuente'
DIR_CONTROL = BASE / 'data' / 'control'
DIR_GOLD = BASE / 'data' / 'gold'
DIR_LOGS = BASE / 'logs'
for d in (DIR_FUENTE, DIR_GOLD, DIR_LOGS):
    d.mkdir(parents=True, exist_ok=True)


def leer_env(ruta):
    """Lee un archivo .env simple (CLAVE=valor). Las credenciales nunca se escriben en el código."""
    valores = {}
    if ruta.exists():
        for linea in ruta.read_text(encoding='utf-8').splitlines():
            linea = linea.strip()
            if linea and not linea.startswith('#') and '=' in linea:
                k, v = linea.split('=', 1)
                valores[k.strip()] = v.strip()
    return valores


ENV = {**leer_env(BASE / '.env'), **{k: v for k, v in os.environ.items() if k.startswith(('DR360_', 'MSSQL_'))}}

# --- Parámetros principales ---
MOTOR = ENV.get('DR360_MOTOR', 'mssql')          # 'mssql' (SQL Server) o 'sqlite' (respaldo)
SEMILLA = 360                                      # datos reproducibles: misma semilla, mismos datos
ESCALA = 2                                         # volumen: 2 ≈ 115 mil líneas de venta al año (1 = prueba rápida)
SAL_SEUDONIMO = ENV.get('DR360_SAL', 'cambiar-esta-sal-en-.env')   # sal secreta para el hash del RUT
REINICIAR_BD = True                                # True: borra y recrea la base (ejecución limpia de demostración)
REGENERAR_FUENTES = False                          # False: si el dataset ya existe en data/fuente se reutiliza (no se vuelve a crear)
RUN_ID = datetime.now().strftime('%Y%m%d-%H%M%S') + '-' + uuid.uuid4().hex[:6]

print('Carpeta del proyecto :', BASE)
print('Motor de base de datos:', MOTOR)
print('ID de esta ejecución  :', RUN_ID)

In [ ]:
# --- Logging: consola + archivo por ejecución ---
log = logging.getLogger('dr360')
log.setLevel(logging.INFO)
log.handlers.clear()
_fmt = logging.Formatter('%(asctime)s | %(levelname)-7s | %(message)s', '%H:%M:%S')
_h_consola = logging.StreamHandler()
_h_consola.setFormatter(_fmt)
_h_archivo = logging.FileHandler(DIR_LOGS / f'pipeline_{RUN_ID}.log', encoding='utf-8')
_h_archivo.setFormatter(logging.Formatter('%(asctime)s | %(levelname)-7s | %(message)s'))
log.addHandler(_h_consola)
log.addHandler(_h_archivo)
log.propagate = False
log.info('Inicio de ejecución %s · motor=%s', RUN_ID, MOTOR)

In [ ]:
# --- Conexión ---
def crear_engine():
    if MOTOR == 'mssql':
        servidor = ENV.get('MSSQL_SERVER', r'localhost\SQLEXPRESS')
        base = ENV.get('MSSQL_DATABASE', 'DataRetail360')
        driver = ENV.get('MSSQL_DRIVER', 'ODBC Driver 17 for SQL Server')
        usuario = ENV.get('MSSQL_USER', '')
        query = {'driver': driver, 'TrustServerCertificate': 'yes'}
        if not usuario:
            query['Trusted_Connection'] = 'yes'      # autenticación de Windows
        def url(db):
            return URL.create('mssql+pyodbc', username=usuario or None,
                              password=ENV.get('MSSQL_PASSWORD') or None,
                              host=servidor, database=db, query=query)
        # 1) crear la base de datos si no existe (desde master, en modo autocommit)
        maestro = create_engine(url('master'), isolation_level='AUTOCOMMIT')
        with maestro.connect() as c:
            c.execute(text(f"IF DB_ID('{base}') IS NULL CREATE DATABASE [{base}]"))
        maestro.dispose()
        # 2) conexión a la base del proyecto; fast_executemany acelera las cargas masivas
        return create_engine(url(base), fast_executemany=True)
    ruta = BASE / 'data' / 'dataretail360.db'
    if REINICIAR_BD and ruta.exists():
        ruta.unlink()
    return create_engine(f'sqlite:///{ruta}')


engine = crear_engine()
with engine.connect() as c:
    version = c.execute(text('SELECT @@VERSION' if MOTOR == 'mssql' else 'SELECT sqlite_version()')).scalar()
log.info('Conectado: %s', str(version).splitlines()[0])

ESQUEMAS = ('bronze', 'silver', 'gold', 'audit', 'sem')


def T(nombre):
    """Nombre físico de una tabla. En SQL Server se usan esquemas (gold.dim_producto);
    SQLite no tiene esquemas, así que se usa un prefijo (gold_dim_producto)."""
    return nombre if MOTOR == 'mssql' else nombre.replace('.', '_')


def Q(sql):
    """Adapta una consulta escrita con esquemas al motor activo."""
    if MOTOR == 'mssql':
        return sql
    return re.sub(r'\b(' + '|'.join(ESQUEMAS) + r')\.', r'\1_', sql)


def leer(sql, con=None, **params):
    df = pd.read_sql(text(Q(sql)), con or engine, params=params or None)
    # SQL Server entrega DECIMAL como objetos Decimal: se convierten a float para poder operar con pandas
    for c in df.columns:
        if df[c].dtype == object:
            no_nulos = df[c].dropna()
            if len(no_nulos) and isinstance(no_nulos.iloc[0], Decimal):
                df[c] = df[c].astype(float)
    return df


def ejecutar(sql, con=None, params=None):
    if con is not None:
        return con.execute(text(Q(sql)), params or {})
    with engine.begin() as c:
        return c.execute(text(Q(sql)), params or {})


def escribir(df, nombre, con, chunksize=2000):
    """Inserta un DataFrame en una tabla ya creada (append)."""
    esquema, tabla = nombre.split('.')
    if MOTOR == 'mssql':
        df.to_sql(tabla, con, schema=esquema, if_exists='append', index=False, chunksize=chunksize)
    else:
        df.to_sql(f'{esquema}_{tabla}', con, if_exists='append', index=False, chunksize=chunksize)


def contar(nombre, con=None):
    return int(leer(f'SELECT COUNT(*) AS n FROM {nombre}', con)['n'].iloc[0])


def a_fecha(serie):
    return pd.to_datetime(serie, errors='coerce').dt.date

## 2. Sistemas de origen (datos simulados del caso)

El caso DataRetail 360 no entrega archivos de datos, por lo que se generan con **semilla fija** (siempre los mismos datos) los archivos que exportaría cada sistema.
Siguiendo el enfoque del curso, la data **se basa en la guía 2.3.5** (y en los notebooks 2.1.4 y 2.2.3) y el equipo la **completa con datos propios** para cubrir todo el caso:

| Elemento | Origen |
|---|---|
| Catálogo de 16 productos (SKU, nombre, categoría, precio) | Guía 2.3.5 |
| Sucursales Providencia, Las Condes, Maipú, Ñuñoa, Concepción, Antofagasta, Viña del Mar, Temuco | Guía 2.3.5 |
| Canales Tienda física, E-commerce y App; estados OK / DEVUELTA; RUT sintéticos con dígito verificador | Guía 2.3.5 |
| Problemas de calidad (SKU en minúsculas, fechas con `/` e imposibles, canal mal escrito, sucursal vacía, monto negativo o `N/A`, SKU `XX-999`, duplicados) | Guía 2.3.5 |
| Sucursal Chillán, tienda online `WEB`, dimensión geográfica y cambio de la Región de Ñuble | Notebooks 2.1.4 / 2.2.3 y Guía 2.1.2 |
| Marcas, productos de Hogar, clientes y segmentos, campañas 2025, devoluciones parciales, inventario diario, costos, zonas comerciales, sistema legado 2018 | **Creados por el equipo** |

**18 archivos de entrada:** cada sistema entrega su propia exportación por semestre (S1 = enero–junio, S2 = julio–diciembre), más los 9 maestros y el histórico 2018.
Así se reproduce el problema central del caso (datos repartidos en varias fuentes con formatos distintos) y se pueden cargar los dos lotes.

**El dataset se genera una sola vez** y queda guardado en `data/fuente/` (y versionado en GitHub). En las ejecuciones siguientes se reutiliza tal cual, y antes de procesarlo
se verifica su integridad contra `manifest.json` (huella SHA-256 de cada archivo). Para recrearlo desde cero: `REGENERAR_FUENTES = True`.

Los datos imitan los problemas descritos en el caso y en el diagnóstico de calidad de la EP1, para que el pipeline tenga algo real que resolver:

| Sistema (área) | Archivos | Problemas de calidad sembrados a propósito |
|---|---|---|
| POS tiendas (Comercial) | `ventas/pos_ventas_2025_S1/S2.csv` | fechas en 3 formatos y fechas imposibles, SKU en minúsculas/espacios, canal escrito de 4 formas, precios con coma decimal, negativos o `N/A`, filas duplicadas, sucursal vacía, cantidades inválidas, SKU inexistente |
| E-commerce y App (Canal digital) | `ventas/web_pedidos_2025_S1/S2.csv` | otro separador (`;`), otros nombres de columna, fecha-hora ISO, descuento en **porcentaje** (no en monto), plataforma escrita de varias formas |
| Sistema legado | `ventas/legado_ventas_2018.csv` | ventas históricas de Chillán antes y después de la creación de la Región de Ñuble |
| Devoluciones (Comercial) | `ventas/devoluciones_2025_S1/S2.csv` | devoluciones parciales y totales asociadas a la línea vendida |
| CRM (Marketing) | `maestros/clientes_crm_v1/v2.csv` | RUT en 3 formatos, RUT con dígito verificador inválido, clientes duplicados, datos personales que el DW no debe guardar |
| Maestro de homologación (Comercial) | `maestros/equivalencias_productos.csv` | relaciona el código de cada sistema (POS, e-commerce, legado) con el SKU corporativo |
| Catálogo (Comercial) | `maestros/productos_v1/v2.csv` | v2 recategoriza el audio, consolida "Electrohogar" en "Hogar" y actualiza costos (SCD 2) |
| Sucursales (Operaciones) | `maestros/sucursales_v1/v2.csv` | v2 corrige un nombre (SCD 1) y reorganiza zonas comerciales (SCD 3) |
| Campañas (Marketing) | `maestros/campanas.csv` | calendario de campañas 2025 |
| División territorial (BCN) | `maestros/geografia_cut.csv` | comunas con vigencias (Ley 21.033, Región de Ñuble) |
| Inventario (Operaciones) | `inventario/stock_diario_2025_S1/S2.csv` | días sin registro en una sucursal, stock negativo |

In [ ]:
rng = np.random.default_rng(SEMILLA)

# ---------- División territorial (catálogo oficial con vigencias) ----------
GEO = pd.DataFrame([
    # codigo_comuna, codigo_cut_vigente, comuna, provincia, region, latitud, longitud, valido_desde, valido_hasta
    ('13101', '13101', 'Santiago',     'Santiago',     'Metropolitana', -33.4489, -70.6693, '1900-01-01', None),
    ('13123', '13123', 'Providencia',  'Santiago',     'Metropolitana', -33.4314, -70.6093, '1900-01-01', None),
    ('13114', '13114', 'Las Condes',   'Santiago',     'Metropolitana', -33.4080, -70.5670, '1900-01-01', None),
    ('13119', '13119', 'Maipú',        'Santiago',     'Metropolitana', -33.5110, -70.7580, '1900-01-01', None),
    ('13120', '13120', 'Ñuñoa',        'Santiago',     'Metropolitana', -33.4569, -70.5979, '1900-01-01', None),
    ('13201', '13201', 'Puente Alto',  'Cordillera',   'Metropolitana', -33.6110, -70.5750, '1900-01-01', None),
    ('05101', '05101', 'Valparaíso',   'Valparaíso',   'Valparaíso',    -33.0472, -71.6127, '1900-01-01', None),
    ('05109', '05109', 'Viña del Mar', 'Valparaíso',   'Valparaíso',    -33.0245, -71.5518, '1900-01-01', None),
    ('04101', '04101', 'La Serena',    'Elqui',        'Coquimbo',      -29.9027, -71.2519, '1900-01-01', None),
    ('02101', '02101', 'Antofagasta',  'Antofagasta',  'Antofagasta',   -23.6509, -70.3975, '1900-01-01', None),
    ('08101', '08101', 'Concepción',   'Concepción',   'Biobío',        -36.8270, -73.0503, '1900-01-01', None),
    ('08110', '08110', 'Talcahuano',   'Concepción',   'Biobío',        -36.7249, -73.1168, '1900-01-01', None),
    # Ley 21.033: desde el 2018-09-06 Chillán y Chillán Viejo pasan de la Región del Biobío a la nueva Región de Ñuble
    ('16101', '08401', 'Chillán',      'Ñuble',        'Biobío',        -36.6066, -72.1034, '1900-01-01', '2018-09-06'),
    ('16101', '16101', 'Chillán',      'Diguillín',    'Ñuble',         -36.6066, -72.1034, '2018-09-06', None),
    ('16103', '08406', 'Chillán Viejo', 'Ñuble',       'Biobío',        -36.6230, -72.1320, '1900-01-01', '2018-09-06'),
    ('16103', '16103', 'Chillán Viejo', 'Diguillín',   'Ñuble',         -36.6230, -72.1320, '2018-09-06', None),
    ('09101', '09101', 'Temuco',       'Cautín',       'La Araucanía',  -38.7359, -72.5904, '1900-01-01', None),
    ('10101', '10101', 'Puerto Montt', 'Llanquihue',   'Los Lagos',     -41.4693, -72.9424, '1900-01-01', None),
], columns=['codigo_comuna', 'codigo_cut_vigente', 'comuna', 'provincia', 'region', 'latitud', 'longitud', 'valido_desde', 'valido_hasta'])
GEO['pais'] = 'Chile'
COMUNAS = GEO['codigo_comuna'].unique().tolist()

# ---------- Sucursales (v1 y v2) ----------
# Códigos SUC-01, SUC-02, SUC-04 y WEB de los notebooks 2.1.4 / 2.2.3; nombres de sucursal de la guía 2.3.5
SUC_V1 = pd.DataFrame([
    ('SUC-01', 'Providencia',   'Mall',   '13123', 'Zona Centro'),
    ('SUC-02', 'Concepción',    'Mall',   '08101', 'Zona Centro-Sur'),
    ('SUC-03', 'Las Condes',    'Mall',   '13114', 'Zona Centro'),
    ('SUC-04', 'Chillán',       'Calle',  '16101', 'Zona Centro-Sur'),
    ('SUC-05', 'Maipu',         'Calle',  '13119', 'Zona Centro'),     # nombre sin tilde: se corrige en v2 (SCD 1)
    ('SUC-06', 'Ñuñoa',         'Calle',  '13120', 'Zona Centro'),
    ('SUC-07', 'Antofagasta',   'Mall',   '02101', 'Zona Norte'),
    ('SUC-08', 'Viña del Mar',  'Mall',   '05109', 'Zona Centro'),
    ('SUC-09', 'Temuco',        'Mall',   '09101', 'Zona Sur'),
    ('WEB',    'Tienda Online', 'Online', '13101', 'Zona Digital'),
], columns=['codigo_sucursal', 'sucursal', 'tipo_sucursal', 'cod_comuna', 'zona_comercial'])
SUC_V2 = SUC_V1.copy()
SUC_V2.loc[SUC_V2.codigo_sucursal == 'SUC-05', 'sucursal'] = 'Maipú'                              # SCD 1: corrección
SUC_V2.loc[SUC_V2.codigo_sucursal.isin(['SUC-02', 'SUC-04']), 'zona_comercial'] = 'Zona Sur'    # SCD 3: reorganización
FISICAS = [s for s in SUC_V1.codigo_sucursal if s != 'WEB']

# ---------- Productos (v1 y v2) ----------
# Los 16 primeros productos (SKU, nombre, categoría y precio) son el catálogo de la guía 2.3.5; marca y Hogar los agrega el equipo
_prod = [
    # sku, producto, subcategoria, categoria, marca, precio_lista
    ('NB-001', 'Notebook 14',          'Computación',   'Tecnología',   'Lenovo',    799000),
    ('AU-002', 'Audífonos',            'Audio',         'Accesorios',   'Sony',       30000),
    ('TV-003', 'Smart TV 55',          'Televisores',   'Tecnología',   'Samsung',   459000),
    ('CE-004', 'Celular',              'Telefonía',     'Tecnología',   'Samsung',   349000),
    ('TB-005', 'Tablet 10',            'Computación',   'Tecnología',   'Lenovo',    229000),
    ('MO-006', 'Monitor 27',           'Computación',   'Tecnología',   'LG',        189000),
    ('TE-007', 'Teclado',              'Periféricos',   'Accesorios',   'Logitech',   25000),
    ('MS-008', 'Mouse',                'Periféricos',   'Accesorios',   'Logitech',   15000),
    ('IM-009', 'Impresora',            'Computación',   'Tecnología',   'HP',        129000),
    ('PA-010', 'Parlante',             'Audio',         'Accesorios',   'JBL',        59000),
    ('SW-011', 'Smartwatch',           'Wearables',     'Tecnología',   'Xiaomi',    149000),
    ('CA-012', 'Cámara',               'Fotografía',    'Tecnología',   'Canon',     399000),
    ('RO-013', 'Router WiFi',          'Redes',         'Accesorios',   'TP-Link',    45000),
    ('DI-014', 'Disco SSD 1TB',        'Almacenamiento', 'Accesorios',  'Kingston',   69000),
    ('CO-015', 'Consola',              'Consolas',      'Gaming',       'Sony',      499000),
    ('MI-016', 'Microondas',           'Cocina',        'Electrohogar', 'LG',         89000),
    ('HG-017', 'Freidora de aire 5L',  'Cocina',        'Hogar',        'Oster',      79990),
    ('HG-018', 'Aspiradora robot',     'Limpieza',      'Hogar',        'Xiaomi',    189990),
    ('HG-019', 'Cafetera espresso',    'Cocina',        'Hogar',        'Oster',     129990),
    ('HG-020', 'Hervidor eléctrico',   'Cocina',        'Hogar',        'Philips',    29990),
    ('HG-021', 'Lámpara LED inteligente', 'Decoración', 'Hogar',        'Philips',    24990),
    ('HG-022', 'Purificador de aire',  'Climatización', 'Hogar',        'Philips',   149990),
]
PROD_V1 = pd.DataFrame(_prod, columns=['sku', 'producto', 'subcategoria', 'categoria', 'marca', 'precio_lista'])
PROD_V1['costo_unitario'] = (PROD_V1['precio_lista'] * rng.uniform(0.55, 0.72, len(PROD_V1)) / 10).round() * 10
PROD_V2 = PROD_V1.copy()
# SCD 2 · recategorización: el audio pasa de Accesorios a Tecnología
_audio = PROD_V2.subcategoria == 'Audio'
PROD_V2.loc[_audio, 'categoria'] = 'Tecnología'
# SCD 2 · homologación de categorías pedida por el caso: "Electrohogar" se consolida en "Hogar"
PROD_V2.loc[PROD_V2.categoria == 'Electrohogar', 'categoria'] = 'Hogar'
# SCD 2 · alza de costo de importación en televisores y celulares (+8%)
_imp = PROD_V2.sku.isin(['TV-003', 'CE-004'])
PROD_V2.loc[_imp, 'costo_unitario'] = (PROD_V2.loc[_imp, 'costo_unitario'] * 1.08 / 10).round() * 10
SKUS = PROD_V1.sku.tolist()

# Cada sistema identifica los productos a su manera (problema del caso: "los productos no siempre poseen códigos homologados").
# El POS usa el SKU corporativo; el e-commerce, su propio ID de catálogo web; el sistema legado, el SKU sin guion.
WEB_CODE = {k: f'WEB-{1001 + i}' for i, k in enumerate(SKUS)}
LEG_CODE = {k: k.replace('-', '') for k in SKUS}
EQUIV = pd.concat([
    pd.DataFrame({'sistema': 'POS', 'codigo_origen': SKUS, 'sku': SKUS}),
    pd.DataFrame({'sistema': 'ECOMMERCE', 'codigo_origen': [WEB_CODE[k] for k in SKUS], 'sku': SKUS}),
    pd.DataFrame({'sistema': 'LEGADO', 'codigo_origen': [LEG_CODE[k] for k in SKUS], 'sku': SKUS}),
], ignore_index=True)
_peso_cat = PROD_V1.categoria.map({'Tecnología': 1.0, 'Accesorios': 2.2, 'Gaming': 0.6, 'Electrohogar': 1.0, 'Hogar': 1.2}).to_numpy()
P_SKU = _peso_cat / _peso_cat.sum()
PRECIO = dict(zip(PROD_V1.sku, PROD_V1.precio_lista))
CATEG = dict(zip(PROD_V1.sku, PROD_V1.categoria))

# ---------- Campañas 2025 ----------
CAMP = pd.DataFrame([
    ('CMP-2501', 'Vuelta a clases',        'Estacional',  '2025-02-20', '2025-03-15', 'Todos'),
    ('CMP-2502', 'Día de la Madre',        'Estacional',  '2025-05-01', '2025-05-11', 'Todos'),
    ('CMP-2503', 'CyberDay',               'Evento online', '2025-06-02', '2025-06-04', 'E-commerce'),
    ('CMP-2504', 'Liquidación de invierno', 'Liquidación', '2025-07-07', '2025-07-27', 'Todos'),
    ('CMP-2505', 'Lanzamiento app',        'Fidelización', '2025-08-04', '2025-08-31', 'E-commerce'),
    ('CMP-2506', 'Fiestas Patrias',        'Estacional',  '2025-09-08', '2025-09-19', 'Todos'),
    ('CMP-2507', 'Black Friday',           'Evento online', '2025-11-27', '2025-11-30', 'Todos'),
    ('CMP-2508', 'Navidad',                'Estacional',  '2025-12-01', '2025-12-24', 'Todos'),
], columns=['codigo_campana', 'campana', 'tipo_campana', 'fecha_inicio', 'fecha_fin', 'canal_objetivo'])

# ---------- Clientes (CRM v1 y v2) ----------
def dv_rut(numero):
    s, m = 0, 2
    for d in reversed(str(numero)):
        s += int(d) * m
        m = 2 if m == 7 else m + 1
    r = 11 - s % 11
    return '0' if r == 11 else 'K' if r == 10 else str(r)


def formatear_rut(numero, dv, estilo):
    n = f'{numero:,}'.replace(',', '.')
    return {0: f'{n}-{dv}', 1: f'{numero}-{dv}', 2: f'{numero}{dv}'}[estilo]


N_CLIENTES = 2500
_nums = rng.choice(np.arange(8_000_000, 24_000_000), N_CLIENTES, replace=False)
_nombres = ['Ana', 'Bruno', 'Carla', 'Diego', 'Elena', 'Felipe', 'Gabriela', 'Hugo', 'Isidora', 'Javier', 'Karen', 'Luis',
            'Macarena', 'Nicolás', 'Olivia', 'Pablo', 'Rocío', 'Sebastián', 'Tamara', 'Vicente']
_apellidos = ['González', 'Muñoz', 'Rojas', 'Díaz', 'Pérez', 'Soto', 'Contreras', 'Silva', 'Martínez', 'Sepúlveda',
              'Morales', 'Rodríguez', 'López', 'Fuentes', 'Hernández', 'Torres', 'Araya', 'Flores']
_peso_comuna = np.array([6, 5, 5, 4, 4, 3, 3, 3, 2, 2, 3, 2, 2, 1, 2, 1], dtype=float)
_comunas_cli = [c for c in COMUNAS]
_peso_comuna = _peso_comuna[:len(_comunas_cli)] / _peso_comuna[:len(_comunas_cli)].sum()
REGION_DE = GEO.drop_duplicates('codigo_comuna', keep='last').set_index('codigo_comuna')['region'].to_dict()
CLI_V1 = pd.DataFrame({
    'rut_num': _nums,
    'nombre': [f'{rng.choice(_nombres)} {rng.choice(_apellidos)}' for _ in range(N_CLIENTES)],
    'segmento': rng.choice(['Nuevo', 'Ocasional', 'Frecuente', 'VIP'], N_CLIENTES, p=[0.30, 0.38, 0.25, 0.07]),
    'comuna': rng.choice(_comunas_cli, N_CLIENTES, p=_peso_comuna),
})
CLI_V1['dv'] = CLI_V1['rut_num'].map(dv_rut)
CLI_V1['email'] = [f'cliente{i:05d}@correo.cl' for i in range(N_CLIENTES)]
CLI_V1['region'] = CLI_V1['comuna'].map(REGION_DE)
CLI_V2 = CLI_V1.copy()
# SCD 2: 160 clientes cambian de segmento y 60 se mudan de región
_idx_seg = rng.choice(N_CLIENTES, 160, replace=False)
_sube = {'Nuevo': 'Ocasional', 'Ocasional': 'Frecuente', 'Frecuente': 'VIP', 'VIP': 'Frecuente'}
CLI_V2.loc[_idx_seg, 'segmento'] = CLI_V2.loc[_idx_seg, 'segmento'].map(_sube)
_idx_mud = rng.choice(np.setdiff1d(np.arange(N_CLIENTES), _idx_seg), 60, replace=False)
CLI_V2.loc[_idx_mud, 'comuna'] = rng.choice(['04101', '10101', '09101', '02101'], 60)
CLI_V2['region'] = CLI_V2['comuna'].map(REGION_DE)


def exportar_crm(df, ruta):
    out = df.copy()
    estilos = rng.choice([0, 1, 2], len(out), p=[0.5, 0.35, 0.15])
    out['rut'] = [formatear_rut(n, d, e) for n, d, e in zip(out.rut_num, out.dv, estilos)]
    # 2% de clientes duplicados: mismo RUT en otro formato y correo en mayúsculas
    dup = out.sample(frac=0.02, random_state=SEMILLA).copy()
    dup['rut'] = [formatear_rut(n, d, 2) for n, d in zip(dup.rut_num, dup.dv)]
    dup['email'] = dup['email'].str.upper()
    contar_problema('CRM_' + ruta.stem.split('_')[-1], 'cliente_duplicado_otro_formato_rut', len(dup))
    contar_problema('CRM_' + ruta.stem.split('_')[-1], 'rut_sin_puntos_o_sin_guion', (estilos != 0).sum())
    out = pd.concat([out, dup]).sample(frac=1, random_state=SEMILLA)
    out[['rut', 'nombre', 'email', 'segmento', 'comuna', 'region']].to_csv(ruta, index=False, encoding='utf-8')


# ---------- Ventas ----------
def es_campana(fecha):
    f = fecha.strftime('%Y-%m-%d')
    hit = CAMP[(CAMP.fecha_inicio <= f) & (CAMP.fecha_fin >= f)]
    return hit.iloc[0] if len(hit) else None


def generar_ventas(fechas, sucursales, p_suc, prefijo_pos, prefijo_web, clientes, p_identificado=0.72):
    """Genera líneas de venta 'limpias' (la verdad del negocio). Los errores se agregan al exportar."""
    filas, n_pos, n_web = [], 0, 0
    for f in fechas:
        camp = es_campana(f)
        base = ESCALA * (52 + 21 * (f.weekday() >= 5) + 30 * (f.month == 12) + (35 if camp is not None else 0))
        n_bol = rng.poisson(base)
        sucs = rng.choice(sucursales, n_bol, p=p_suc)
        for suc in sucs:
            web = suc == 'WEB'
            if web:
                n_web += 1; nro = f'{prefijo_web}{n_web:06d}'
            else:
                n_pos += 1; nro = f'{prefijo_pos}{n_pos:06d}'
            cli = clientes[rng.integers(len(clientes))] if (web or rng.random() < p_identificado) else None
            camp_ok = camp is not None and (camp.canal_objetivo == 'Todos' or web) and rng.random() < 0.65
            canal = ('App' if rng.random() < 0.28 else 'E-commerce') if web else 'Tienda física'
            for linea in range(1, int(rng.choice([1, 1, 1, 2, 2, 3, 4])) + 1):
                sku = rng.choice(SKUS, p=P_SKU)
                cant = int(rng.choice([1, 1, 1, 2, 2, 3])) if CATEG[sku] == 'Accesorios' else int(rng.choice([1, 1, 1, 1, 2]))
                precio = PRECIO[sku]
                pct = rng.choice([0.10, 0.15, 0.20, 0.30]) if camp_ok else (0.05 if rng.random() < 0.12 else 0.0)
                filas.append(dict(nro_boleta=nro, linea=linea, fecha=f, sku=sku, codigo_sucursal=suc, cliente=cli,
                                  canal=canal,
                                  cod_comuna=None, cantidad=cant, precio_unitario=precio,
                                  pct_desc=pct, descuento=round(precio * cant * pct),
                                  codigo_campana=camp.codigo_campana if camp_ok else None,
                                  estado=rng.choice(['OK', 'DEVUELTA', 'ANULADA'], p=[0.975, 0.017, 0.008])))
    df = pd.DataFrame(filas)
    comuna_suc = SUC_V1.set_index('codigo_sucursal')['cod_comuna']
    df['cod_comuna'] = df['codigo_sucursal'].map(comuna_suc)
    return df


PROBLEMAS = {}          # errores insertados por fuente: se guardan en manifest.json


def contar_problema(fuente, error, n):
    PROBLEMAS.setdefault(fuente, {})
    PROBLEMAS[fuente][error] = PROBLEMAS[fuente].get(error, 0) + int(n)


def ensuciar_pos(df, fuente='POS'):
    """Inyecta los problemas de calidad del export del POS (mismas tasas de la guía 2.3.5, más los del caso)."""
    out = df.copy().reset_index(drop=True)
    n = len(out)
    pos = lambda p: out.index[rng.choice(n, size=max(int(n * p), 1), replace=False)]
    fecha_txt = out['fecha'].map(lambda d: d.strftime('%Y-%m-%d'))
    alt = rng.random(n) < 0.30                                       # el POS antiguo exporta DD/MM/AAAA
    fecha_txt[alt] = out.loc[alt, 'fecha'].map(lambda d: d.strftime('%d/%m/%Y'))
    contar_problema(fuente, 'fecha_DD/MM/AAAA', alt.sum())
    out['fecha'] = fecha_txt
    i = pos(.03); out.loc[i, 'fecha'] = out.loc[i, 'fecha'].str.replace('-', '/')     # guía 2.3.5: fecha con '/'
    contar_problema(fuente, 'fecha_con_slash', len(i))
    i = pos(.05); out.loc[i, 'sku'] = out.loc[i, 'sku'].str.lower() + ' '               # guía 2.3.5: SKU en minúsculas
    contar_problema(fuente, 'sku_minusculas_espacios', len(i))
    out['canal'] = rng.choice(['Tienda física', 'Tienda fisica', 'TIENDA FISICA', 'tienda física'], n, p=[0.7, 0.15, 0.1, 0.05])
    contar_problema(fuente, 'canal_mal_escrito', (out['canal'] != 'Tienda física').sum())
    precio = out['precio_unitario'].astype(int).astype(str)
    m = rng.random(n) < 0.10
    precio[m] = precio[m] + ',00'                                                         # coma decimal
    contar_problema(fuente, 'precio_coma_decimal', m.sum())
    out['precio_unitario'] = precio
    out['rut_cliente'] = [None if c is None else formatear_rut(c[0], c[1], int(rng.choice([0, 0, 1, 2])))
                          for c in out['cliente']]
    m = (rng.random(n) < 0.015) & out.rut_cliente.notna()                                 # RUT con dígito verificador mal digitado
    out.loc[m, 'rut_cliente'] = out.loc[m, 'rut_cliente'].str[:-1] + 'X'
    contar_problema(fuente, 'rut_dv_invalido', m.sum())
    out['cantidad'] = out['cantidad'].astype(str)
    i = pos(.002); out.loc[i, 'cantidad'] = '0'
    contar_problema(fuente, 'cantidad_cero', len(i))
    i = pos(.001); out.loc[i, 'cantidad'] = 'dos'
    contar_problema(fuente, 'cantidad_no_numerica', len(i))
    i = pos(.008); out.loc[i, 'codigo_sucursal'] = None                                   # guía 2.3.5: sucursal vacía
    contar_problema(fuente, 'sucursal_vacia', len(i))
    i = pos(.004); out.loc[i, 'precio_unitario'] = '-' + out.loc[i, 'precio_unitario']    # guía 2.3.5: monto negativo
    contar_problema(fuente, 'monto_negativo', len(i))
    i = pos(.001); out.loc[i, 'precio_unitario'] = 'N/A'                                  # guía 2.3.5: monto no numérico
    contar_problema(fuente, 'monto_no_numerico', len(i))
    i = pos(.002); out.loc[i, 'sku'] = 'XX-999'                                           # guía 2.3.5: SKU inexistente
    contar_problema(fuente, 'sku_inexistente', len(i))
    i = pos(.001); out.loc[i, 'fecha'] = '2025-02-30'                                     # guía 2.3.5: fecha imposible
    contar_problema(fuente, 'fecha_imposible', len(i))
    i = pos(.001); out.loc[i, 'fecha'] = '31/02/2025'
    contar_problema(fuente, 'fecha_imposible', len(i))
    dups = out.sample(frac=0.01, random_state=SEMILLA)                                    # guía 2.3.5: 1% duplicadas
    contar_problema(fuente, 'fila_duplicada', len(dups))
    contar_problema(fuente, 'filas_exportadas', n + len(dups))
    out = pd.concat([out, dups]).sort_values(['nro_boleta', 'linea'], kind='stable')
    return out[['nro_boleta', 'linea', 'fecha', 'sku', 'codigo_sucursal', 'rut_cliente', 'canal', 'cantidad',
                'precio_unitario', 'descuento', 'codigo_campana', 'estado']]


def exportar_web(df):
    """El e-commerce exporta con otro formato: ';' , coma decimal, fecha-hora ISO y descuento en %."""
    out = pd.DataFrame({
        'id_pedido': df['nro_boleta'], 'item': df['linea'],
        'fecha_hora': [f.strftime('%Y-%m-%d') + f'T{rng.integers(8, 23):02d}:{rng.integers(0, 60):02d}:00' for f in df['fecha']],
        'codigo_producto': [('WEB-9999' if rng.random() < 0.0015 else WEB_CODE[k]) for k in df['sku']],   # ID propio del e-commerce
        'rut': [None if c is None else formatear_rut(c[0], c[1], 1) for c in df['cliente']],
        'comuna_despacho': [c[2] if c is not None else rng.choice(COMUNAS) for c in df['cliente']],
        'unidades': df['cantidad'],
        'precio': df['precio_unitario'].map(lambda v: f'{v:.2f}'.replace('.', ',')),
        'descuento_pct': (df['pct_desc'] * 100).round(0).astype(int),
        'cupon_campana': df['codigo_campana'],
        'plataforma': [('APP' if c == 'App' else 'WEB') if rng.random() > 0.04 else ('app' if c == 'App' else 'ecommerce')
                       for c in df['canal']],
        'estado_pedido': df['estado'].map({'OK': 'ENTREGADO', 'ANULADA': 'CANCELADO', 'DEVUELTA': 'DEVUELTO'}),
    })
    contar_problema('ECOMMERCE', 'codigo_producto_propio_WEB', (out['codigo_producto'] != 'WEB-9999').sum())
    contar_problema('ECOMMERCE', 'sku_inexistente_WEB-9999', (out['codigo_producto'] == 'WEB-9999').sum())
    contar_problema('ECOMMERCE', 'fecha_hora_ISO_y_coma_decimal', len(out))
    contar_problema('ECOMMERCE', 'descuento_en_porcentaje', (out['descuento_pct'] > 0).sum())
    contar_problema('ECOMMERCE', 'plataforma_mal_escrita', out['plataforma'].isin(['app', 'ecommerce']).sum())
    contar_problema('ECOMMERCE', 'filas_exportadas', len(out))
    return out


def generar_devoluciones(df):
    validas = df[df.estado == 'OK']
    dev = validas.sample(frac=0.025, random_state=int(rng.integers(1e6))).copy()
    dev['cantidad_devuelta'] = [int(rng.integers(1, c + 1)) for c in dev['cantidad']]
    dev['fecha_devolucion'] = [(f + timedelta(days=int(rng.integers(1, 25)))).strftime('%Y-%m-%d') for f in dev['fecha']]
    dev['motivo'] = rng.choice(['Producto defectuoso', 'No cumple expectativas', 'Error en el pedido', 'Retracto (compra web)'],
                               len(dev), p=[0.35, 0.35, 0.15, 0.15])
    return dev[['nro_boleta', 'linea', 'fecha_devolucion', 'cantidad_devuelta', 'motivo']]


_SUCS_INV = SUC_V1.codigo_sucursal.tolist()
INV_ESTADO = {'stock': {(s, k): int(rng.integers(40, 90)) for s in _SUCS_INV for k in SKUS},
              'pedidos': {}, 'pendiente': set(),
              'reorden': {(s, k): int(rng.integers(6, 18)) for s in _SUCS_INV for k in SKUS}}


rng_inv = np.random.default_rng(SEMILLA + 1)       # generador propio del inventario (no altera las ventas)


def preparar_reposicion(ventas_anio):
    """Ajusta la política de reposición a la demanda real de cada sucursal y producto:
    punto de reorden ≈ 7 días de venta y lote ≈ 14–22 días de venta (mínimos de 6 y 30 unidades)."""
    dem = (ventas_anio[ventas_anio.estado == 'OK'].groupby(['codigo_sucursal', 'sku'])['cantidad'].sum() / 365).to_dict()
    INV_ESTADO['demanda'] = dem
    for (s, k) in INV_ESTADO['stock']:
        d = dem.get((s, k), 0)
        INV_ESTADO['reorden'][(s, k)] = max(INV_ESTADO['reorden'][(s, k)], int(np.ceil(d * 7)))
        INV_ESTADO['stock'][(s, k)] = max(INV_ESTADO['stock'][(s, k)], int(d * 20))


def generar_inventario(ventas, fechas):
    """Snapshot diario de stock por producto y sucursal, coherente con las ventas (el stock se arrastra entre meses)."""
    v = ventas[ventas.estado == 'OK'].groupby(['fecha', 'codigo_sucursal', 'sku'])['cantidad'].sum().to_dict()
    st_, ped, pend, reo = INV_ESTADO['stock'], INV_ESTADO['pedidos'], INV_ESTADO['pendiente'], INV_ESTADO['reorden']
    dem = INV_ESTADO.get('demanda', {})
    filas = []
    for f in fechas:
        for s in _SUCS_INV:
            for k in SKUS:
                rec = ped.pop((f, s, k), 0)
                if rec:
                    pend.discard((s, k))
                st = max(st_[(s, k)] + rec - int(v.get((f, s, k), 0)), 0)
                st_[(s, k)] = st
                if st <= reo[(s, k)] and (s, k) not in pend:
                    lote = max(int(rng_inv.integers(30, 70)), int(round(dem.get((s, k), 0) * rng_inv.uniform(14, 22))))
                    ped[(f + timedelta(days=int(rng_inv.integers(2, 10))), s, k)] = lote
                    pend.add((s, k))
                filas.append((f, k, s, st, rec))
    inv = pd.DataFrame(filas, columns=['fecha', 'sku', 'codigo_sucursal', 'stock_cierre', 'unidades_recibidas'])
    return inv

In [ ]:
def generar_fuentes():
    """Genera los archivos de los sistemas de origen y el manifiesto (solo se ejecuta la primera vez)."""
    global CLIENTES_TUPLA, P_SUC
    PROBLEMAS.clear()
    t0 = time.time()
    for sub in ('ventas', 'maestros', 'inventario'):
        (DIR_FUENTE / sub).mkdir(parents=True, exist_ok=True)

    # Maestros
    GEO.to_csv(DIR_FUENTE / 'maestros' / 'geografia_cut.csv', index=False)
    SUC_V1.to_csv(DIR_FUENTE / 'maestros' / 'sucursales_v1.csv', index=False)
    SUC_V2.to_csv(DIR_FUENTE / 'maestros' / 'sucursales_v2.csv', index=False)
    PROD_V1.to_csv(DIR_FUENTE / 'maestros' / 'productos_v1.csv', index=False)
    PROD_V2.to_csv(DIR_FUENTE / 'maestros' / 'productos_v2.csv', index=False)
    CAMP.to_csv(DIR_FUENTE / 'maestros' / 'campanas.csv', index=False)
    EQUIV.to_csv(DIR_FUENTE / 'maestros' / 'equivalencias_productos.csv', index=False)
    exportar_crm(CLI_V1, DIR_FUENTE / 'maestros' / 'clientes_crm_v1.csv')
    exportar_crm(CLI_V2, DIR_FUENTE / 'maestros' / 'clientes_crm_v2.csv')

    # Clientes como tuplas (rut_num, dv, comuna) para asignarlos a las ventas
    CLIENTES_TUPLA = list(zip(CLI_V1.rut_num, CLI_V1.dv, CLI_V1.comuna))
    P_SUC = np.array([0.12, 0.08, 0.11, 0.05, 0.08, 0.07, 0.06, 0.07, 0.05, 0.31])   # orden de SUC_V1

    # Ventas 2025 + devoluciones + inventario: se generan día a día y se exporta UN archivo por semestre y sistema
    todas = []
    pos_s, web_s, dev_s, inv_s = {1: [], 2: []}, {1: [], 2: []}, {1: [], 2: []}, {1: [], 2: []}
    for mes in range(1, 13):
        sem = 1 if mes <= 6 else 2
        ini = date(2025, mes, 1)
        fin = date(2025 + (mes == 12), mes % 12 + 1, 1)
        fechas = [ini + timedelta(days=i) for i in range((fin - ini).days)]
        vt = generar_ventas(fechas, SUC_V1.codigo_sucursal.tolist(), P_SUC, f'B{mes:02d}-', f'W{mes:02d}-', CLIENTES_TUPLA)
        todas.append(vt)
        pos_s[sem].append(ensuciar_pos(vt[vt.codigo_sucursal != 'WEB']))
        web_s[sem].append(exportar_web(vt[vt.codigo_sucursal == 'WEB']))
        dev_s[sem].append(generar_devoluciones(vt))
    # Inventario: se genera después de conocer la demanda del año, para que la reposición sea coherente con lo que vende cada sucursal
    preparar_reposicion(pd.concat(todas, ignore_index=True))
    for mes in range(1, 13):
        sem = 1 if mes <= 6 else 2
        ini = date(2025, mes, 1)
        fin = date(2025 + (mes == 12), mes % 12 + 1, 1)
        fechas = [ini + timedelta(days=i) for i in range((fin - ini).days)]
        vt = todas[mes - 1]
        inv = generar_inventario(vt, fechas)
        # Problemas de calidad en inventario: Antofagasta no informó stock del 10 al 23 de marzo; algunos negativos
        if mes == 3:
            antes = len(inv)
            inv = inv[~((inv.codigo_sucursal == 'SUC-07') & inv.fecha.between(date(2025, 3, 10), date(2025, 3, 23)))]
            contar_problema('INVENTARIO', 'filas_sin_registro_SUC-07_marzo', antes - len(inv))
        inv.loc[inv.sample(n=3, random_state=mes).index, 'stock_cierre'] = -5
        contar_problema('INVENTARIO', 'stock_negativo', 3)
        inv['fecha'] = inv['fecha'].map(lambda d: d.strftime('%Y-%m-%d'))
        inv_s[sem].append(inv)
    for sem in (1, 2):
        pd.concat(pos_s[sem]).to_csv(DIR_FUENTE / 'ventas' / f'pos_ventas_2025_S{sem}.csv', index=False)
        pd.concat(web_s[sem]).to_csv(DIR_FUENTE / 'ventas' / f'web_pedidos_2025_S{sem}.csv', sep=';', index=False, encoding='utf-8')
        pd.concat(dev_s[sem]).to_csv(DIR_FUENTE / 'ventas' / f'devoluciones_2025_S{sem}.csv', index=False)
        pd.concat(inv_s[sem]).to_csv(DIR_FUENTE / 'inventario' / f'stock_diario_2025_S{sem}.csv', index=False)

    # Ventas históricas del sistema legado: Chillán, jun–dic 2018 (cruza la creación de la Región de Ñuble)
    f18 = [date(2018, 6, 1) + timedelta(days=i) for i in range(214)]
    leg = generar_ventas(f18, ['SUC-04'], [1.0], 'L18-', 'LW18-', CLIENTES_TUPLA)
    leg = leg.sample(frac=0.06, random_state=SEMILLA).sort_values(['nro_boleta', 'linea'])
    leg['codigo_campana'] = None
    leg['descuento'] = 0
    leg['sku'] = leg['sku'].map(LEG_CODE)                         # el sistema legado usa el SKU sin guion
    contar_problema('LEGADO', 'codigo_producto_sin_guion', len(leg))
    ensuciar_pos(leg, fuente='LEGADO').to_csv(DIR_FUENTE / 'ventas' / 'legado_ventas_2018.csv', index=False)

    # Manifiesto: evidencia de qué fuente se procesó (nombre, filas, huella SHA-256)
    manifiesto = []
    for p in sorted(DIR_FUENTE.rglob('*.csv')):
        contenido = p.read_bytes()
        manifiesto.append({'archivo': str(p.relative_to(DIR_FUENTE)).replace('\\', '/'),
                           'filas': contenido.count(b'\n') - 1,
                           'sha256': hashlib.sha256(contenido).hexdigest()})
    parametros = {'semilla': SEMILLA, 'escala': ESCALA, 'periodo_ventas': '2025-01-01 a 2025-12-31 (+ histórico Chillán jun–dic 2018)',
                  'clientes': N_CLIENTES, 'productos': len(PROD_V1), 'sucursales': len(SUC_V1),
                  'vigencia_maestros_v1': '2025-01-01', 'vigencia_maestros_v2': '2025-07-01'}
    (DIR_FUENTE / 'manifest.json').write_text(json.dumps({'semilla': SEMILLA, 'generado': datetime.now().isoformat(timespec='seconds'),
                                                           'parametros': parametros, 'problemas_insertados': PROBLEMAS,
                                                           'archivos': manifiesto}, indent=2, ensure_ascii=False, default=int), encoding='utf-8')
    # Ventas "verdaderas" (antes de ensuciarlas): referencia de control para la validación final (sección 11.5)
    referencia = pd.concat(todas, ignore_index=True)[['nro_boleta', 'linea', 'estado', 'cantidad', 'precio_unitario', 'descuento']]
    DIR_CONTROL.mkdir(parents=True, exist_ok=True)
    referencia.to_csv(DIR_CONTROL / 'ventas_referencia_2025.csv', index=False)
    return manifiesto


def verificar_fuentes():
    """Integridad: cada archivo de origen debe tener la misma huella SHA-256 registrada en el manifiesto."""
    man = json.loads((DIR_FUENTE / 'manifest.json').read_text(encoding='utf-8'))
    filas = []
    for item in man['archivos']:
        ruta = DIR_FUENTE / item['archivo']
        actual = hashlib.sha256(ruta.read_bytes()).hexdigest() if ruta.exists() else None
        filas.append({**item, 'estado': 'OK' if actual == item['sha256'] else ('NO EXISTE' if actual is None else 'MODIFICADO')})
    rev = pd.DataFrame(filas)
    malos = rev[rev.estado != 'OK']
    if len(malos):
        raise ValueError(f'Las fuentes no coinciden con el manifiesto ({len(malos)} archivos): '
                         f'{", ".join(malos.archivo.head(5))}. Use REGENERAR_FUENTES = True para recrearlas.')
    return rev


t0 = time.time()
if REGENERAR_FUENTES or not (DIR_FUENTE / 'manifest.json').exists():
    generar_fuentes()
    log.info('Fuentes generadas y guardadas en %s (%.1f s)', DIR_FUENTE, time.time() - t0)
else:
    log.info('Se usan las fuentes ya guardadas en %s (no se regeneran)', DIR_FUENTE)
revision = verificar_fuentes()
VERDAD = pd.read_csv(DIR_CONTROL / 'ventas_referencia_2025.csv', dtype={'nro_boleta': str})
log.info('Integridad de las fuentes: %d de %d archivos coinciden con el manifiesto (SHA-256)', (revision.estado == 'OK').sum(), len(revision))
revision.assign(sha256=lambda d: d.sha256.str[:16] + '…')[['archivo', 'filas', 'sha256', 'estado']]

### Errores de calidad insertados a propósito

Cada error tiene una proporción fija (la misma de la guía 2.3.5 cuando la guía lo define). En la sección 10 se compara lo **insertado** con lo que el
pipeline **detectó**, para demostrar que la limpieza funciona.

In [ ]:
ERRORES_INSERTADOS = pd.DataFrame([
    # error, fuente, proporción, origen del criterio, cómo lo detecta el pipeline (tipo: clave), tratamiento en Silver
    ('SKU en minúsculas o con espacios', 'POS', '5%', 'Guía 2.3.5', 'correccion:SKU con minúsculas o espacios normalizado', 'Se corrige'),
    ('Fecha con "/": AAAA/MM/DD (guía) o DD/MM/AAAA (POS antiguo)', 'POS', '3% + 30%', 'Guía 2.3.5 + equipo', 'correccion:Fecha DD/MM/AAAA o AAAA/MM/DD convertida a fecha estándar', 'Se corrige'),
    ('Fecha imposible (2025-02-30, 31/02/2025)', 'POS', '0,2%', 'Guía 2.3.5', 'rechazo:FECHA_INVALIDA', 'Cuarentena'),
    ('Canal mal escrito (Tienda fisica, TIENDA FISICA…)', 'POS', '30%', 'Guía 2.3.5', 'correccion:Canal escrito de otra forma homologado (Tienda física / E-commerce / App)', 'Se corrige'),
    ('Sucursal vacía', 'POS', '0,8%', 'Guía 2.3.5', 'correccion:Venta sin sucursal asignada a SIN_ASIGNAR (se conserva)', 'Se conserva como SIN_ASIGNAR'),
    ('Monto negativo o "N/A"', 'POS', '0,5%', 'Guía 2.3.5', 'rechazo:PRECIO_INVALIDO', 'Cuarentena'),
    ('SKU inexistente (XX-999 / WEB-9999)', 'POS y e-commerce', '0,2% / 0,15%', 'Guía 2.3.5', 'rechazo:SKU_INEXISTENTE', 'Cuarentena'),
    ('Fila duplicada', 'POS', '1%', 'Guía 2.3.5', 'rechazo:DUPLICADO', 'Se elimina la copia'),
    ('Cantidad 0 o no numérica ("dos")', 'POS', '0,3%', 'Equipo', 'rechazo:CANTIDAD_INVALIDA', 'Cuarentena'),
    ('Precio con coma decimal', 'POS', '10%', 'Equipo', 'correccion:Precio con coma decimal convertido a número', 'Se corrige'),
    ('RUT con dígito verificador inválido', 'POS', '1,5% de las filas con RUT', 'Equipo', 'correccion:RUT con dígito verificador inválido → cliente no identificado', 'Cliente no identificado'),
    ('Código de producto propio del e-commerce (WEB-1001…)', 'E-commerce', '100%', 'Caso / EP1', 'correccion:Código propio del sistema homologado a SKU corporativo (tabla de equivalencias)', 'Tabla de equivalencias'),
    ('Código de producto sin guion (NB001)', 'Legado 2018', '100%', 'Caso / EP1', 'correccion:Código propio del sistema homologado a SKU corporativo (tabla de equivalencias)', 'Tabla de equivalencias'),
    ('Descuento en porcentaje (no en monto)', 'E-commerce', 'según venta', 'Equipo', 'correccion:Descuento en porcentaje convertido a monto', 'Se convierte'),
    ('Cliente duplicado (mismo RUT en otro formato)', 'CRM', '2%', 'Caso', 'rechazo:CLIENTE_DUPLICADO', 'Se conserva un registro'),
    ('Stock negativo', 'Inventario', '3 filas por mes', 'Equipo', 'rechazo:STOCK_NEGATIVO', 'Cuarentena'),
    ('Sucursal sin registro de stock 14 días', 'Inventario', 'SUC-07, marzo', 'Caso', 'control:Oportunidad', 'Se informa (no se inventa)'),
], columns=['error', 'fuente', 'proporcion_insertada', 'criterio', 'deteccion', 'tratamiento'])
ERRORES_INSERTADOS[['error', 'fuente', 'proporcion_insertada', 'criterio', 'tratamiento']]

In [ ]:
print('Muestra del export del POS (con sus errores):')
display(pd.read_csv(DIR_FUENTE / 'ventas' / 'pos_ventas_2025_S1.csv', dtype=str).head(8))
print('Muestra del export del e-commerce (otro formato):')
display(pd.read_csv(DIR_FUENTE / 'ventas' / 'web_pedidos_2025_S1.csv', sep=';', dtype=str).head(5))

## 3. Estructuras de almacenamiento (esquemas y tablas)

Se crean cinco esquemas en la base `DataRetail360`:

* **bronze**: copia fiel de cada archivo de origen, todo como texto, con columnas de trazabilidad (`_archivo`, `_fila`, `_run_id`, `_cargado_en`).
* **silver**: datos limpios, tipados, validados e integrados (POS + e-commerce + legado), sin RUT ni nombres.
* **gold**: modelo dimensional Kimball (dimensiones conformadas y dos tablas de hechos) y los 4 data marts por área.
* **audit**: log de ejecución, registros rechazados (cuarentena) y controles de calidad.
* **sem**: capa semántica para consultas con IA (diccionario de datos y definición de KPIs).

**Modelo dimensional (Gold).** Esquema **estrella**: dos tablas de hechos que comparten dimensiones conformadas. Todas las dimensiones son tablas planas (desnormalizadas) conectadas **directo** a los hechos; no hay dimensiones que dependan de otras dimensiones.

| Tabla | Tipo | Grano | SCD |
|---|---|---|---|
| `fact_ventas` | Hechos transaccional | 1 fila = 1 línea de boleta | — |
| `fact_inventario` | Hechos snapshot periódico | 1 fila = 1 producto × 1 sucursal × 1 día | — |
| `dim_fecha` | Dimensión conformada | 1 día | 0 (estática) |
| `dim_producto` | Dimensión conformada | 1 versión de producto | **2** |
| `dim_cliente` | Dimensión | 1 versión de cliente | **2** |
| `dim_geografia` | Dimensión espacial conformada | 1 versión de comuna | **2** |
| `dim_sucursal` | Dimensión conformada | 1 sucursal | **1** (nombre, formato) + **3** (zona comercial) |
| `dim_canal` | Dimensión | 1 canal | 1 |
| `dim_campana` | Dimensión | 1 campaña | 1 |

In [ ]:
DDL = {
# ------------------------------ AUDIT ------------------------------
'audit.etl_log': """
    run_id VARCHAR(40), lote_id VARCHAR(20), etapa VARCHAR(60), objeto VARCHAR(80),
    filas_entrada INT, filas_salida INT, filas_rechazadas INT, estado VARCHAR(10), mensaje NVARCHAR(400),
    inicio DATETIME2, fin DATETIME2, segundos DECIMAL(10,2)""",
'audit.rechazos': """
    run_id VARCHAR(40), lote_id VARCHAR(20), entidad VARCHAR(40), archivo VARCHAR(120), fila INT,
    motivo VARCHAR(40), detalle NVARCHAR(300), registro NVARCHAR(1000)""",
'audit.correcciones': """
    run_id VARCHAR(40), lote_id VARCHAR(20), fuente VARCHAR(20), regla NVARCHAR(150), dimension_calidad VARCHAR(30),
    capa VARCHAR(10), filas INT""",
'audit.controles_calidad': """
    run_id VARCHAR(40), lote_id VARCHAR(20), control NVARCHAR(150), dimension_calidad VARCHAR(30),
    resultado NVARCHAR(150), ok INT""",
# ------------------------------ BRONZE (todo texto) ------------------------------
'bronze.pos_ventas': """
    nro_boleta NVARCHAR(30), linea NVARCHAR(10), fecha NVARCHAR(30), sku NVARCHAR(30), codigo_sucursal NVARCHAR(20),
    rut_cliente NVARCHAR(20), canal NVARCHAR(30), cantidad NVARCHAR(10), precio_unitario NVARCHAR(20), descuento NVARCHAR(20),
    codigo_campana NVARCHAR(20), estado NVARCHAR(15),
    _archivo VARCHAR(120), _fila INT, _run_id VARCHAR(40), _cargado_en DATETIME2""",
'bronze.web_pedidos': """
    id_pedido NVARCHAR(30), item NVARCHAR(10), fecha_hora NVARCHAR(30), codigo_producto NVARCHAR(30), rut NVARCHAR(20),
    comuna_despacho NVARCHAR(10), unidades NVARCHAR(10), precio NVARCHAR(20), descuento_pct NVARCHAR(10),
    cupon_campana NVARCHAR(20), plataforma NVARCHAR(15), estado_pedido NVARCHAR(15),
    _archivo VARCHAR(120), _fila INT, _run_id VARCHAR(40), _cargado_en DATETIME2""",
'bronze.devoluciones': """
    nro_boleta NVARCHAR(30), linea NVARCHAR(10), fecha_devolucion NVARCHAR(30), cantidad_devuelta NVARCHAR(10), motivo NVARCHAR(60),
    _archivo VARCHAR(120), _fila INT, _run_id VARCHAR(40), _cargado_en DATETIME2""",
'bronze.stock_diario': """
    fecha NVARCHAR(30), sku NVARCHAR(30), codigo_sucursal NVARCHAR(20), stock_cierre NVARCHAR(10), unidades_recibidas NVARCHAR(10),
    _archivo VARCHAR(120), _fila INT, _run_id VARCHAR(40), _cargado_en DATETIME2""",
'bronze.clientes_crm': """
    rut NVARCHAR(20), nombre NVARCHAR(80), email NVARCHAR(80), segmento NVARCHAR(20), comuna NVARCHAR(10), region NVARCHAR(40),
    _archivo VARCHAR(120), _fila INT, _run_id VARCHAR(40), _cargado_en DATETIME2""",
'bronze.productos': """
    sku NVARCHAR(30), producto NVARCHAR(80), subcategoria NVARCHAR(40), categoria NVARCHAR(40), marca NVARCHAR(40),
    precio_lista NVARCHAR(20), costo_unitario NVARCHAR(20),
    _archivo VARCHAR(120), _fila INT, _run_id VARCHAR(40), _cargado_en DATETIME2""",
'bronze.sucursales': """
    codigo_sucursal NVARCHAR(20), sucursal NVARCHAR(60), tipo_sucursal NVARCHAR(20), cod_comuna NVARCHAR(10), zona_comercial NVARCHAR(40),
    _archivo VARCHAR(120), _fila INT, _run_id VARCHAR(40), _cargado_en DATETIME2""",
'bronze.campanas': """
    codigo_campana NVARCHAR(20), campana NVARCHAR(60), tipo_campana NVARCHAR(30), fecha_inicio NVARCHAR(20), fecha_fin NVARCHAR(20),
    canal_objetivo NVARCHAR(20),
    _archivo VARCHAR(120), _fila INT, _run_id VARCHAR(40), _cargado_en DATETIME2""",
'bronze.equivalencias_productos': """
    sistema NVARCHAR(20), codigo_origen NVARCHAR(30), sku NVARCHAR(30),
    _archivo VARCHAR(120), _fila INT, _run_id VARCHAR(40), _cargado_en DATETIME2""",
'bronze.geografia_cut': """
    codigo_comuna NVARCHAR(10), codigo_cut_vigente NVARCHAR(10), comuna NVARCHAR(60), provincia NVARCHAR(60), region NVARCHAR(60),
    latitud NVARCHAR(20), longitud NVARCHAR(20), valido_desde NVARCHAR(20), valido_hasta NVARCHAR(20), pais NVARCHAR(30),
    _archivo VARCHAR(120), _fila INT, _run_id VARCHAR(40), _cargado_en DATETIME2""",
# ------------------------------ SILVER (limpio y tipado) ------------------------------
'silver.ventas': """
    nro_boleta VARCHAR(30) NOT NULL, linea_boleta INT NOT NULL, fecha DATE, sku VARCHAR(20), codigo_sucursal VARCHAR(20),
    cliente_hash CHAR(64), canal NVARCHAR(30), cod_comuna_venta VARCHAR(10), codigo_campana VARCHAR(20),
    cantidad INT, precio_unitario DECIMAL(14,2), monto_bruto DECIMAL(14,2), monto_descuento DECIMAL(14,2),
    cantidad_devuelta INT, monto_devuelto DECIMAL(14,2), monto_neto DECIMAL(14,2), estado VARCHAR(15), origen VARCHAR(12),
    lote_id VARCHAR(20), archivo_origen VARCHAR(120), fila_origen INT, run_id VARCHAR(40),
    CONSTRAINT pk_silver_ventas PRIMARY KEY (nro_boleta, linea_boleta)""",
'silver.inventario': """
    fecha DATE NOT NULL, sku VARCHAR(20) NOT NULL, codigo_sucursal VARCHAR(20) NOT NULL, stock_cierre INT, unidades_recibidas INT,
    lote_id VARCHAR(20), archivo_origen VARCHAR(120), fila_origen INT, run_id VARCHAR(40),
    CONSTRAINT pk_silver_inventario PRIMARY KEY (fecha, sku, codigo_sucursal)""",
'silver.clientes': """
    cliente_hash CHAR(64) NOT NULL, segmento NVARCHAR(20), comuna VARCHAR(10), region NVARCHAR(40),
    lote_id VARCHAR(20), run_id VARCHAR(40), CONSTRAINT pk_silver_clientes PRIMARY KEY (cliente_hash, lote_id)""",
# ------------------------------ GOLD · dimensiones ------------------------------
'gold.dim_fecha': """
    fecha_key INT NOT NULL PRIMARY KEY, fecha DATE, dia_semana NVARCHAR(12), es_fin_de_semana INT, semana INT, mes INT,
    nombre_mes NVARCHAR(12), trimestre INT, anio INT, es_feriado INT""",
'gold.dim_geografia': """
    geo_key INT NOT NULL PRIMARY KEY, codigo_comuna VARCHAR(10), codigo_cut_vigente VARCHAR(10), comuna NVARCHAR(60),
    provincia NVARCHAR(60), region NVARCHAR(60), pais NVARCHAR(30), latitud DECIMAL(9,6), longitud DECIMAL(9,6),
    valido_desde DATE, valido_hasta DATE, es_actual INT""",
'gold.dim_sucursal': """
    sucursal_key INT NOT NULL PRIMARY KEY, codigo_sucursal VARCHAR(20), sucursal NVARCHAR(60), tipo_sucursal NVARCHAR(20),
    codigo_comuna VARCHAR(10), comuna NVARCHAR(60), region NVARCHAR(60),
    zona_comercial_actual NVARCHAR(40), zona_comercial_anterior NVARCHAR(40), fecha_cambio_zona DATE""",
'gold.dim_producto': """
    producto_key INT NOT NULL PRIMARY KEY, sku VARCHAR(20), producto NVARCHAR(80), subcategoria NVARCHAR(40), categoria NVARCHAR(40),
    marca NVARCHAR(40), precio_lista DECIMAL(14,2), costo_unitario DECIMAL(14,2), valido_desde DATE, valido_hasta DATE, es_actual INT""",
'gold.dim_cliente': """
    cliente_key INT NOT NULL PRIMARY KEY, cliente_hash CHAR(64), segmento NVARCHAR(40), comuna NVARCHAR(60), region NVARCHAR(60),
    valido_desde DATE, valido_hasta DATE, es_actual INT""",
'gold.dim_canal': """
    canal_key INT NOT NULL PRIMARY KEY, canal NVARCHAR(30), tipo_canal NVARCHAR(20)""",
'gold.dim_campana': """
    campana_key INT NOT NULL PRIMARY KEY, codigo_campana VARCHAR(20), campana NVARCHAR(60), tipo_campana NVARCHAR(30),
    fecha_inicio DATE, fecha_fin DATE, canal_objetivo NVARCHAR(20)""",
# ------------------------------ GOLD · hechos ------------------------------
'gold.fact_ventas': """
    nro_boleta VARCHAR(30) NOT NULL, linea_boleta INT NOT NULL,
    fecha_key INT NOT NULL, producto_key INT NOT NULL, sucursal_key INT NOT NULL, cliente_key INT NOT NULL,
    canal_key INT NOT NULL, geo_key INT NOT NULL, campana_key INT NOT NULL,
    cantidad INT, precio_unitario DECIMAL(14,2), monto_bruto DECIMAL(14,2), monto_descuento DECIMAL(14,2),
    cantidad_devuelta INT, monto_devuelto DECIMAL(14,2), monto_neto DECIMAL(14,2),
    costo_unitario DECIMAL(14,2), costo_neto DECIMAL(14,2), margen_bruto DECIMAL(14,2), devolucion INT,
    lote_id VARCHAR(20), archivo_origen VARCHAR(120), fila_origen INT, run_id VARCHAR(40),
    CONSTRAINT pk_fact_ventas PRIMARY KEY (nro_boleta, linea_boleta),
    CONSTRAINT fk_fv_fecha FOREIGN KEY (fecha_key) REFERENCES gold.dim_fecha(fecha_key),
    CONSTRAINT fk_fv_producto FOREIGN KEY (producto_key) REFERENCES gold.dim_producto(producto_key),
    CONSTRAINT fk_fv_sucursal FOREIGN KEY (sucursal_key) REFERENCES gold.dim_sucursal(sucursal_key),
    CONSTRAINT fk_fv_cliente FOREIGN KEY (cliente_key) REFERENCES gold.dim_cliente(cliente_key),
    CONSTRAINT fk_fv_canal FOREIGN KEY (canal_key) REFERENCES gold.dim_canal(canal_key),
    CONSTRAINT fk_fv_geo FOREIGN KEY (geo_key) REFERENCES gold.dim_geografia(geo_key),
    CONSTRAINT fk_fv_campana FOREIGN KEY (campana_key) REFERENCES gold.dim_campana(campana_key)""",
'gold.fact_inventario': """
    fecha_key INT NOT NULL, producto_key INT NOT NULL, sucursal_key INT NOT NULL, geo_key INT NOT NULL,
    stock_cierre INT, unidades_recibidas INT, en_quiebre INT,
    lote_id VARCHAR(20), archivo_origen VARCHAR(120), fila_origen INT, run_id VARCHAR(40),
    CONSTRAINT pk_fact_inventario PRIMARY KEY (fecha_key, producto_key, sucursal_key),
    CONSTRAINT fk_fi_fecha FOREIGN KEY (fecha_key) REFERENCES gold.dim_fecha(fecha_key),
    CONSTRAINT fk_fi_producto FOREIGN KEY (producto_key) REFERENCES gold.dim_producto(producto_key),
    CONSTRAINT fk_fi_sucursal FOREIGN KEY (sucursal_key) REFERENCES gold.dim_sucursal(sucursal_key),
    CONSTRAINT fk_fi_geo FOREIGN KEY (geo_key) REFERENCES gold.dim_geografia(geo_key)""",
# ------------------------------ SEM (capa semántica) ------------------------------
'sem.kpi': """
    kpi NVARCHAR(60), data_mart NVARCHAR(30), definicion NVARCHAR(300), formula_sql NVARCHAR(300), grano NVARCHAR(120)""",
'sem.mapa_fuentes': """
    sistema_origen NVARCHAR(60), area NVARCHAR(40), archivos NVARCHAR(200), formato NVARCHAR(150), tabla_bronze NVARCHAR(60),
    limpieza_silver NVARCHAR(400), integracion NVARCHAR(250), destino_gold NVARCHAR(200)""",
'sem.diccionario_datos': """
    capa NVARCHAR(10), tabla NVARCHAR(80), columna NVARCHAR(60), tipo_dato NVARCHAR(40), rol NVARCHAR(30), descripcion NVARCHAR(400)""",
'sem.diccionario': """
    objeto NVARCHAR(60), columna NVARCHAR(60), descripcion NVARCHAR(300), sinonimos NVARCHAR(150), regla NVARCHAR(200)""",
}

VISTAS = ['gold.vw_tablero_integrado', 'gold.vw_ia_ventas']
TABLAS_AGG = ['gold.agg_ventas_mes_producto_sucursal', 'gold.agg_inventario_mes_producto_sucursal']
TABLAS_DM = ['gold.dm_comercial_ventas', 'gold.dm_marketing_campanas', 'gold.dm_finanzas_margen', 'gold.dm_operaciones_inventario']


def crear_estructuras():
    with engine.begin() as c:
        if MOTOR == 'mssql':
            for e in ESQUEMAS:
                c.execute(text(f"IF SCHEMA_ID('{e}') IS NULL EXEC('CREATE SCHEMA {e}')"))
        for v in VISTAS:
            c.execute(text(Q(f'DROP VIEW IF EXISTS {v}')))
        for t in TABLAS_DM + TABLAS_AGG + list(reversed(list(DDL))):   # primero hechos, luego dimensiones (por las FK)
            c.execute(text(Q(f'DROP TABLE IF EXISTS {t}')))
        for t, cols in DDL.items():
            c.execute(text(Q(f'CREATE TABLE {t} ({cols})')))
        # Miembros especiales (-1): permiten conservar la venta aunque falte el dato, sin perder integridad referencial
        c.execute(text(Q("INSERT INTO gold.dim_geografia VALUES (-1,'-1','-1','Sin comuna','Sin provincia','Sin región','Chile',NULL,NULL,'1900-01-01',NULL,1)")))
        c.execute(text(Q("INSERT INTO gold.dim_sucursal VALUES (-1,'SIN_ASIGNAR','Sin sucursal asignada','Sin dato','-1','Sin comuna','Sin región','Sin zona',NULL,NULL)")))
        c.execute(text(Q("INSERT INTO gold.dim_cliente VALUES (-1,NULL,'Cliente no identificado','Sin dato','Sin dato','1900-01-01',NULL,1)")))
        c.execute(text(Q("INSERT INTO gold.dim_campana VALUES (-1,'SIN_CAMPANA','Sin campaña','Sin campaña',NULL,NULL,'Todos')")))
    log.info('Estructuras creadas: %d tablas en %s', len(DDL), ', '.join(ESQUEMAS))


if REINICIAR_BD:
    crear_estructuras()
pd.DataFrame([(t.split('.')[0], t.split('.')[1]) for t in DDL], columns=['esquema', 'tabla']).groupby('esquema')['tabla'].apply(', '.join).to_frame()

## 4. Framework del pipeline: etapas, auditoría, rechazos y manejo de errores

Cada etapa se ejecuta dentro de `etapa(...)`, que:
1. registra inicio, fin, duración y filas procesadas en `audit.etl_log` y en el archivo de log;
2. si ocurre un error, lo registra con estado `ERROR` y lo vuelve a lanzar, para que la **transacción del lote se revierta completa** (todo o nada).

Los registros que no cumplen las reglas no se pierden: van a `audit.rechazos` con su motivo y el registro original (cuarentena).

In [ ]:
_log_buffer = []


@contextmanager
def etapa(lote_id, nombre, objeto=''):
    info = {'filas_entrada': None, 'filas_salida': None, 'filas_rechazadas': None, 'mensaje': ''}
    inicio = datetime.now()
    log.info('▶ [%s] %s %s', lote_id, nombre, objeto)
    try:
        yield info
        estado = 'OK'
    except Exception as e:
        estado, info['mensaje'] = 'ERROR', f'{type(e).__name__}: {e}'[:400]
        log.error('✖ [%s] %s falló: %s', lote_id, nombre, info['mensaje'])
        raise
    finally:
        fin = datetime.now()
        _log_buffer.append(dict(run_id=RUN_ID, lote_id=lote_id, etapa=nombre, objeto=objeto,
                                filas_entrada=info['filas_entrada'], filas_salida=info['filas_salida'],
                                filas_rechazadas=info['filas_rechazadas'], estado=estado, mensaje=info['mensaje'],
                                inicio=inicio, fin=fin, segundos=round((fin - inicio).total_seconds(), 2)))
        if estado == 'OK':
            log.info('  ✔ entrada=%s salida=%s rechazadas=%s (%.1f s)', info['filas_entrada'], info['filas_salida'],
                     info['filas_rechazadas'], (fin - inicio).total_seconds())


def guardar_log():
    """El log de auditoría se escribe en una conexión aparte: sobrevive aunque el lote se revierta."""
    if _log_buffer:
        with engine.begin() as c:
            escribir(pd.DataFrame(_log_buffer), 'audit.etl_log', c)
        _log_buffer.clear()


def registrar_rechazos(con, lote_id, entidad, df, motivo_col='motivo', detalle_col='detalle'):
    if df.empty:
        return 0
    # El registro original se guarda sin datos personales (RUT, nombre, correo) ni columnas auxiliares
    cols = [c for c in df.columns if not c.startswith('_') and c not in (motivo_col, detalle_col)
            and not re.search(r'rut|nombre|email|hash', c) and not c.endswith(('_n', '_d'))]
    rech = pd.DataFrame({
        'run_id': RUN_ID, 'lote_id': lote_id, 'entidad': entidad,
        'archivo': df['_archivo'].to_numpy(), 'fila': df['_fila'].astype(int).to_numpy(),
        'motivo': df[motivo_col].to_numpy(),
        'detalle': df[detalle_col].fillna('').astype(str).str[:300].to_numpy() if detalle_col in df else '',
        'registro': [json.dumps(r, ensure_ascii=False, default=str)[:1000] for r in df[cols].to_dict('records')],
    })
    escribir(rech, 'audit.rechazos', con)
    return len(rech)

## 5. Capa Bronze: ingesta cruda y trazable

Se copia cada archivo **tal como llega**, todo como texto, sin corregir nada. Cada fila queda con el archivo y el número de fila de origen,
el identificador de la ejecución y la hora de carga. Esto permite volver a procesar desde Bronze sin pedir los archivos otra vez y rastrear cualquier cifra hasta su origen.

### Mapa de fuentes y linaje

Qué sistema genera cada archivo, cómo entra al pipeline, en qué capa se limpia y se integra, y dónde termina. Se guarda también en `sem.mapa_fuentes`.

In [ ]:
MAPA_FUENTES = pd.DataFrame([
    ('POS de tiendas físicas', 'Comercial', 'ventas/pos_ventas_2025_S1/S2.csv', 'CSV con coma; fechas DD/MM/AAAA y AAAA-MM-DD; descuento en pesos; SKU corporativo',
     'bronze.pos_ventas', 'Duplicados, SKU, canal, fechas, precios, cantidades, sucursal vacía, RUT → hash',
     'Se une con e-commerce y legado en silver.ventas (mismo esquema y SKU corporativo)', 'fact_ventas → Comercial, Marketing, Finanzas'),
    ('E-commerce y App', 'Canal digital', 'ventas/web_pedidos_2025_S1/S2.csv', 'CSV con punto y coma; coma decimal; fecha-hora ISO; descuento en %; código de producto WEB-####',
     'bronze.web_pedidos', 'Código WEB → SKU (equivalencias), plataforma → canal, % → monto, fecha-hora → fecha, RUT → hash',
     'silver.ventas (sucursal WEB; comuna de despacho como lugar de venta)', 'fact_ventas → Comercial, Marketing, Finanzas'),
    ('Sistema legado (2018)', 'Comercial', 'ventas/legado_ventas_2018.csv', 'Formato del POS; SKU sin guion',
     'bronze.pos_ventas', 'Código legado → SKU (equivalencias) y mismas reglas del POS',
     'silver.ventas (origen LEGADO)', 'fact_ventas (histórico; SCD 2 geográfico de Chillán)'),
    ('Registro de devoluciones', 'Comercial', 'ventas/devoluciones_2025_S1/S2.csv', 'CSV por línea de boleta',
     'bronze.devoluciones', 'Se agregan por línea y se descuentan con la regla única de venta neta',
     'silver.ventas (cantidad_devuelta, monto_devuelto)', 'fact_ventas → Finanzas (devoluciones)'),
    ('Sistema de operaciones', 'Operaciones', 'inventario/stock_diario_2025_S1/S2.csv', 'CSV diario por producto y sucursal',
     'bronze.stock_diario', 'Stock negativo a cuarentena; detección de días sin registro',
     'silver.inventario', 'fact_inventario → Operaciones'),
    ('CRM de clientes', 'Marketing', 'maestros/clientes_crm_v1/v2.csv', 'CSV; RUT en 3 formatos; incluye nombre y correo',
     'bronze.clientes_crm', 'RUT validado y deduplicado; nombre y correo descartados; RUT → hash',
     'silver.clientes', 'dim_cliente (SCD 2)'),
    ('Catálogo de productos', 'Comercial', 'maestros/productos_v1/v2.csv', 'CSV; v2 recategoriza y cambia costos',
     'bronze.productos', 'SKU normalizado', 'Catálogo único de SKU corporativos', 'dim_producto (SCD 2)'),
    ('Maestro de homologación', 'Comercial', 'maestros/equivalencias_productos.csv', 'CSV sistema–código–SKU',
     'bronze.equivalencias_productos', 'Se usa para traducir códigos de e-commerce y legado', 'Resuelve los códigos distintos entre sistemas', '(insumo de silver.ventas)'),
    ('Maestro de sucursales', 'Operaciones', 'maestros/sucursales_v1/v2.csv', 'CSV; v2 corrige nombre y cambia zonas',
     'bronze.sucursales', '—', 'Comuna y región copiadas como atributos planos (estrella)', 'dim_sucursal (SCD 1 + SCD 3)'),
    ('Calendario de campañas', 'Marketing', 'maestros/campanas.csv', 'CSV', 'bronze.campanas', 'Fechas tipadas', '—', 'dim_campana (SCD 1)'),
    ('División territorial (BCN)', 'Externo', 'maestros/geografia_cut.csv', 'CSV con vigencias (Ley 21.033)',
     'bronze.geografia_cut', 'Coordenadas y fechas tipadas', '—', 'dim_geografia (SCD 2, carga histórica)'),
], columns=['sistema_origen', 'area', 'archivos', 'formato', 'tabla_bronze', 'limpieza_silver', 'integracion', 'destino_gold'])
MAPA_FUENTES[['sistema_origen', 'area', 'archivos', 'tabla_bronze', 'destino_gold']]

In [ ]:
FUENTES_LOTE = {
    'L1-2025H1': {'ventas_pos': ['ventas/pos_ventas_2025_S1.csv', 'ventas/legado_ventas_2018.csv'],
                  'ventas_web': ['ventas/web_pedidos_2025_S1.csv'],
                  'devoluciones': ['ventas/devoluciones_2025_S1.csv'],
                  'inventario': ['inventario/stock_diario_2025_S1.csv'],
                  'clientes': 'maestros/clientes_crm_v1.csv', 'productos': 'maestros/productos_v1.csv',
                  'sucursales': 'maestros/sucursales_v1.csv', 'campanas': 'maestros/campanas.csv',
                  'geografia': 'maestros/geografia_cut.csv', 'equivalencias': 'maestros/equivalencias_productos.csv',
                  'vigencia_maestros': date(2025, 1, 1)},
    'L2-2025H2': {'ventas_pos': ['ventas/pos_ventas_2025_S2.csv'],
                  'ventas_web': ['ventas/web_pedidos_2025_S2.csv'],
                  'devoluciones': ['ventas/devoluciones_2025_S2.csv'],
                  'inventario': ['inventario/stock_diario_2025_S2.csv'],
                  'clientes': 'maestros/clientes_crm_v2.csv', 'productos': 'maestros/productos_v2.csv',
                  'sucursales': 'maestros/sucursales_v2.csv', 'campanas': 'maestros/campanas.csv',
                  'geografia': 'maestros/geografia_cut.csv', 'equivalencias': 'maestros/equivalencias_productos.csv',
                  'vigencia_maestros': date(2025, 7, 1)},
}

COLUMNAS_ESPERADAS = {
    'pos_ventas': ['nro_boleta', 'linea', 'fecha', 'sku', 'codigo_sucursal', 'rut_cliente', 'canal', 'cantidad',
                   'precio_unitario', 'descuento', 'codigo_campana', 'estado'],
    'web_pedidos': ['id_pedido', 'item', 'fecha_hora', 'codigo_producto', 'rut', 'comuna_despacho', 'unidades', 'precio',
                    'descuento_pct', 'cupon_campana', 'plataforma', 'estado_pedido'],
}


def leer_fuente(rel, sep=','):
    """Extract: lee un archivo de origen como texto y le agrega columnas de linaje."""
    ruta = DIR_FUENTE / rel
    if not ruta.exists():
        raise FileNotFoundError(f'No existe el archivo de origen {rel}')
    df = pd.read_csv(ruta, sep=sep, dtype=str, keep_default_na=False, na_values=[''], encoding='utf-8')
    df.columns = [c.strip() for c in df.columns]
    df['_archivo'] = rel
    df['_fila'] = np.arange(2, len(df) + 2)       # fila 1 = encabezado
    df['_run_id'] = RUN_ID
    df['_cargado_en'] = datetime.now().replace(microsecond=0)
    return df


def bronze(con, lote_id, tabla, archivos, sep=','):
    archivos = archivos if isinstance(archivos, list) else [archivos]
    with etapa(lote_id, 'Bronze · ingesta', f'bronze.{tabla}') as info:
        df = pd.concat([leer_fuente(a, sep) for a in archivos], ignore_index=True)
        if tabla in COLUMNAS_ESPERADAS:
            faltan = set(COLUMNAS_ESPERADAS[tabla]) - set(df.columns)
            if faltan:
                raise ValueError(f'Estructura inválida en {tabla}: faltan columnas {sorted(faltan)}')
        # Idempotencia en Bronze: si el mismo archivo ya se había cargado, se reemplaza su copia (no se duplica)
        borradas = 0
        for arch in df['_archivo'].unique():
            borradas += con.execute(text(Q(f'DELETE FROM bronze.{tabla} WHERE _archivo = :a')), {'a': arch}).rowcount or 0
        escribir(df, f'bronze.{tabla}', con)
        info['filas_entrada'] = info['filas_salida'] = len(df)
        if borradas:
            info['mensaje'] = f'copia anterior reemplazada: {borradas} filas'
    return df

## 6. Capa Silver: limpieza, validación e integración

Cada regla corresponde a una dimensión de calidad del diagnóstico de la EP1:

| # | Regla | Dimensión de calidad | Tratamiento |
|---|---|---|---|
| 1 | SKU en mayúsculas y sin espacios; canal homologado | Consistencia | Se corrige |
| 2 | Fechas `DD/MM/AAAA`, `AAAA/MM/DD`, `AAAA-MM-DD` e ISO con hora → fecha única | Validez | Se corrige; si es imposible (31/02) → rechazo |
| 3 | Montos con coma decimal → número | Validez | Se corrige |
| 4 | Filas idénticas repetidas en la exportación | Unicidad | Se elimina la copia → rechazo `DUPLICADO` |
| 5 | Cantidad no numérica o ≤ 0; precio negativo, cero o `N/A` | Validez | Rechazo |
| 6 | SKU que no existe en el catálogo | Integridad referencial | Rechazo |
| 7 | Venta sin sucursal | Completitud | Se conserva con sucursal `SIN_ASIGNAR` (miembro -1) |
| 8 | RUT en 3 formatos; dígito verificador inválido | Validez / Unicidad | Se normaliza y valida (módulo 11); inválido → cliente no identificado |
| 9 | RUT y nombre del cliente | Seguridad (Ley 19.628 / 21.719) | El RUT se reemplaza por `cliente_hash` (SHA-256 con sal); el nombre y el correo no pasan de Bronze |
| 10 | Descuento web en % y POS en monto | Consistencia | Se lleva todo a monto |
| 11 | Devoluciones y anulaciones | Precisión | **Regla única:** `monto_neto = monto_bruto − monto_descuento − monto_devuelto` |

In [ ]:
MAPA_CANAL = {'tienda fisica': 'Tienda física', 'tienda física': 'Tienda física', 'e-commerce': 'E-commerce', 'ecommerce': 'E-commerce',
              'web': 'E-commerce', 'app': 'App'}


def normalizar_rut(valor):
    """Devuelve el RUT normalizado (sin puntos ni guion, DV en mayúscula) solo si el dígito verificador es válido."""
    if valor is None or (isinstance(valor, float) and np.isnan(valor)):
        return None
    limpio = re.sub(r'[^0-9kK]', '', str(valor)).upper()
    if len(limpio) < 2 or not limpio[:-1].isdigit():
        return None
    cuerpo, dv = limpio[:-1], limpio[-1]
    return f'{int(cuerpo)}{dv}' if dv_rut(int(cuerpo)) == dv else None


def seudonimizar(rut_normalizado):
    """SHA-256 con sal: estable para el mismo RUT (permite analizar recurrencia) e irreversible sin la sal."""
    if rut_normalizado is None:
        return None
    return hashlib.sha256(f'{SAL_SEUDONIMO}|{rut_normalizado}'.encode('utf-8')).hexdigest()


def a_numero(serie):
    return pd.to_numeric(serie.astype(str).str.strip().str.replace(',', '.', regex=False), errors='coerce')


def parsear_fecha(serie):
    s = serie.astype(str).str.strip().str[:10]
    iso = pd.to_datetime(s, format='%Y-%m-%d', errors='coerce')
    ymd = pd.to_datetime(s, format='%Y/%m/%d', errors='coerce')       # guía 2.3.5: AAAA/MM/DD
    dmy = pd.to_datetime(s, format='%d/%m/%Y', errors='coerce')
    return iso.fillna(ymd).fillna(dmy).dt.date


def registrar_correcciones(con, lote_id, filas):
    df = pd.DataFrame(filas, columns=['fuente', 'regla', 'dimension_calidad', 'filas'])
    df = df[df['filas'] > 0].assign(run_id=RUN_ID, lote_id=lote_id, capa='Silver')
    if len(df):
        escribir(df[['run_id', 'lote_id', 'fuente', 'regla', 'dimension_calidad', 'capa', 'filas']], 'audit.correcciones', con)


def silver_ventas(con, lote_id, pos, web, dev, catalogo_skus, equivalencias):
    with etapa(lote_id, 'Silver · ventas', 'silver.ventas') as info:
        info['filas_entrada'] = len(pos) + len(web)
        # --- Integración: el e-commerce se lleva al mismo esquema que el POS ---
        w = pd.DataFrame({
            'nro_boleta': web['id_pedido'], 'linea': web['item'], 'fecha': web['fecha_hora'], 'sku': web['codigo_producto'],
            'codigo_sucursal': 'WEB', 'rut_cliente': web['rut'], 'canal': web['plataforma'], 'cod_comuna_venta': web['comuna_despacho'],
            'cantidad': web['unidades'], 'precio_unitario': web['precio'], 'descuento_pct': web['descuento_pct'],
            'descuento': None, 'codigo_campana': web['cupon_campana'],
            'estado': web['estado_pedido'].map({'ENTREGADO': 'OK', 'CANCELADO': 'ANULADA', 'DEVUELTO': 'DEVUELTA'}), 'origen': 'ECOMMERCE',
            '_archivo': web['_archivo'], '_fila': web['_fila']})
        p = pos.drop(columns=['_run_id', '_cargado_en']).copy()
        p['origen'] = np.where(p['_archivo'].str.contains('legado'), 'LEGADO', 'POS')
        p['descuento_pct'] = None
        df = pd.concat([p, w], ignore_index=True)

        # Regla 4 · Unicidad: filas idénticas (mismo contenido de negocio) en la exportación
        claves_negocio = ['nro_boleta', 'linea', 'fecha', 'sku', 'codigo_sucursal', 'rut_cliente', 'cantidad', 'precio_unitario']
        dup = df.duplicated(subset=claves_negocio, keep='first')
        rech = [df[dup].assign(motivo='DUPLICADO', detalle='Fila repetida en la exportación del origen')]
        df = df[~dup].copy()

        # ---- Registro de correcciones: cuántas filas corrige cada regla, por fuente (evidencia del linaje) ----
        corr = []
        def anotar(regla, dimension, mascara):
            for fuente, n in mascara.groupby(df['origen']).sum().items():
                corr.append((fuente, regla, dimension, int(n)))

        # Regla 1 · Consistencia: códigos de producto distintos entre sistemas → SKU corporativo (tabla de equivalencias)
        crudo = df['sku'].astype(str)
        df['sku'] = crudo.str.strip().str.upper()
        anotar('SKU con minúsculas o espacios normalizado', 'Consistencia', crudo != df['sku'])
        eq = equivalencias.assign(codigo_origen=equivalencias['codigo_origen'].str.strip().str.upper())
        homologado = pd.Series(False, index=df.index)
        for sistema in ('ECOMMERCE', 'LEGADO'):
            mapa = dict(zip(eq.loc[eq.sistema == sistema, 'codigo_origen'], eq.loc[eq.sistema == sistema, 'sku']))
            m = (df['origen'] == sistema) & df['sku'].isin(mapa)
            df.loc[m, 'sku'] = df.loc[m, 'sku'].map(mapa)
            homologado |= m
        anotar('Código propio del sistema homologado a SKU corporativo (tabla de equivalencias)', 'Consistencia', homologado)

        # Reglas 1, 2, 3, 10 · Estandarización
        canal_crudo = df['canal'].astype(str)
        df['canal'] = canal_crudo.str.strip().str.lower().map(MAPA_CANAL).fillna(df['canal'])
        anotar('Canal escrito de otra forma homologado (Tienda física / E-commerce / App)', 'Consistencia', canal_crudo != df['canal'])
        f_cruda = df['fecha'].astype(str).str.strip()
        anotar('Fecha DD/MM/AAAA o AAAA/MM/DD convertida a fecha estándar', 'Validez', f_cruda.str.contains('/'))
        anotar('Fecha-hora ISO del e-commerce convertida a fecha', 'Validez', f_cruda.str.contains('T'))
        anotar('Precio con coma decimal convertido a número', 'Validez', df['precio_unitario'].astype(str).str.contains(','))
        df['fecha_d'] = parsear_fecha(df['fecha'])
        df['cantidad_n'] = a_numero(df['cantidad'])
        df['precio_n'] = a_numero(df['precio_unitario'])
        df['linea_n'] = a_numero(df['linea'])
        bruto = df['cantidad_n'] * df['precio_n']
        desc_monto = a_numero(df['descuento'])
        desc_pct = a_numero(df['descuento_pct'])
        df['monto_descuento'] = desc_monto.fillna((bruto * desc_pct / 100).round(0)).fillna(0)
        anotar('Descuento en porcentaje convertido a monto', 'Consistencia', desc_pct.fillna(0) > 0)

        # Reglas 2, 5, 6 · Validaciones con rechazo
        motivos = [
            ('BOLETA_NULA', df['nro_boleta'].isna() | df['linea_n'].isna(), 'Sin número de boleta o de línea'),
            ('FECHA_INVALIDA', df['fecha_d'].isna(), 'Fecha inexistente o con formato no reconocido'),
            ('CANTIDAD_INVALIDA', df['cantidad_n'].isna() | (df['cantidad_n'] <= 0), 'Cantidad no numérica o menor o igual a cero'),
            ('PRECIO_INVALIDO', df['precio_n'].isna() | (df['precio_n'] <= 0), 'Precio no numérico o menor o igual a cero'),
            ('SKU_INEXISTENTE', ~df['sku'].isin(catalogo_skus), 'El código no existe en el catálogo ni en la tabla de equivalencias'),
        ]
        malo = pd.Series(False, index=df.index)
        for motivo, mask, detalle in motivos:
            nuevo = mask & ~malo                       # cada registro se rechaza por su primer motivo
            rech.append(df[nuevo].assign(motivo=motivo, detalle=detalle))
            malo |= mask
        df = df[~malo].copy()

        # Regla 7 · Completitud: venta sin sucursal → SIN_ASIGNAR (se conserva para no alterar la venta neta)
        anotar('Venta sin sucursal asignada a SIN_ASIGNAR (se conserva)', 'Completitud', df['codigo_sucursal'].isna())
        df['codigo_sucursal'] = df['codigo_sucursal'].fillna('SIN_ASIGNAR')
        comuna_suc = SUC_V1.set_index('codigo_sucursal')['cod_comuna']
        df['cod_comuna_venta'] = df['cod_comuna_venta'].fillna(df['codigo_sucursal'].map(comuna_suc))

        # Regla 8 y 9 · RUT: validar, normalizar y seudonimizar. El RUT no sigue en el pipeline.
        rut_norm = df['rut_cliente'].map(normalizar_rut)
        df['rut_invalido'] = df['rut_cliente'].notna() & rut_norm.isna()
        df['cliente_hash'] = rut_norm.map(seudonimizar)
        anotar('RUT con puntos o sin guion normalizado', 'Validez', rut_norm.notna() & df['rut_cliente'].astype(str).str.contains(r'\.|^[0-9]+[0-9kK]$', regex=True))
        anotar('RUT con dígito verificador inválido → cliente no identificado', 'Validez', df['rut_invalido'])
        anotar('RUT reemplazado por cliente_hash (SHA-256 con sal)', 'Seguridad', rut_norm.notna())

        # Devoluciones: se agregan por línea vendida
        d = dev.copy()
        d['cantidad_devuelta'] = a_numero(d['cantidad_devuelta']).fillna(0)
        d['linea_n'] = a_numero(d['linea'])
        d = d.groupby(['nro_boleta', 'linea_n'], as_index=False)['cantidad_devuelta'].sum()
        df = df.merge(d, on=['nro_boleta', 'linea_n'], how='left')
        df['cantidad_devuelta'] = df['cantidad_devuelta'].fillna(0)
        df['estado'] = df['estado'].fillna('OK').str.strip().str.upper()
        anotar('Anulación o devolución total descontada de la venta neta', 'Precisión', df['estado'].isin(['ANULADA', 'DEVUELTA']))
        df.loc[df['estado'].isin(['ANULADA', 'DEVUELTA']), 'cantidad_devuelta'] = df['cantidad_n']   # anulación o devolución total
        df['cantidad_devuelta'] = np.minimum(df['cantidad_devuelta'], df['cantidad_n'])

        # Regla 11 · Venta neta única
        df['monto_bruto'] = (df['cantidad_n'] * df['precio_n']).round(2)
        precio_efectivo = (df['monto_bruto'] - df['monto_descuento']) / df['cantidad_n']
        df['monto_devuelto'] = (precio_efectivo * df['cantidad_devuelta']).round(0)      # pesos chilenos: sin decimales
        df['monto_neto'] = (df['monto_bruto'] - df['monto_descuento'] - df['monto_devuelto']).round(2)

        silver = pd.DataFrame({
            'nro_boleta': df['nro_boleta'], 'linea_boleta': df['linea_n'].astype(int), 'fecha': df['fecha_d'],
            'sku': df['sku'], 'codigo_sucursal': df['codigo_sucursal'], 'cliente_hash': df['cliente_hash'],
            'canal': df['canal'], 'cod_comuna_venta': df['cod_comuna_venta'], 'codigo_campana': df['codigo_campana'],
            'cantidad': df['cantidad_n'].astype(int), 'precio_unitario': df['precio_n'], 'monto_bruto': df['monto_bruto'],
            'monto_descuento': df['monto_descuento'], 'cantidad_devuelta': df['cantidad_devuelta'].astype(int),
            'monto_devuelto': df['monto_devuelto'], 'monto_neto': df['monto_neto'], 'estado': df['estado'], 'origen': df['origen'],
            'lote_id': lote_id, 'archivo_origen': df['_archivo'], 'fila_origen': df['_fila'].astype(int), 'run_id': RUN_ID})
        con.execute(text(Q('DELETE FROM silver.ventas WHERE lote_id = :l')), {'l': lote_id})   # re-ejecución idempotente
        escribir(silver, 'silver.ventas', con)
        rech_df = pd.concat(rech, ignore_index=True).drop(columns=['fecha_d', 'cantidad_n', 'precio_n', 'linea_n'], errors='ignore')
        info['filas_rechazadas'] = registrar_rechazos(con, lote_id, 'ventas', rech_df)
        anotar('Devolución parcial descontada de la venta neta', 'Precisión', (df['cantidad_devuelta'] > 0) & ~df['estado'].isin(['ANULADA', 'DEVUELTA']))
        registrar_correcciones(con, lote_id, corr)
        info['filas_salida'] = len(silver)
        info['mensaje'] = f'RUT inválidos tratados como cliente no identificado: {int(df["rut_invalido"].sum())}'
    return silver


def silver_inventario(con, lote_id, inv, catalogo_skus):
    with etapa(lote_id, 'Silver · inventario', 'silver.inventario') as info:
        info['filas_entrada'] = len(inv)
        df = inv.copy()
        df['fecha_d'] = parsear_fecha(df['fecha'])
        df['stock_n'] = a_numero(df['stock_cierre'])
        df['rec_n'] = a_numero(df['unidades_recibidas']).fillna(0)
        df['sku'] = df['sku'].str.strip().str.upper()
        motivos = [
            ('FECHA_INVALIDA', df['fecha_d'].isna(), 'Fecha no reconocida'),
            ('STOCK_NEGATIVO', df['stock_n'].isna() | (df['stock_n'] < 0), 'Stock negativo o no numérico'),
            ('SKU_INEXISTENTE', ~df['sku'].isin(catalogo_skus), 'SKU no existe en el catálogo'),
        ]
        malo, rech = pd.Series(False, index=df.index), []
        for motivo, mask, det in motivos:
            rech.append(df[mask & ~malo].assign(motivo=motivo, detalle=det))
            malo |= mask
        df = df[~malo]
        silver = pd.DataFrame({'fecha': df['fecha_d'], 'sku': df['sku'], 'codigo_sucursal': df['codigo_sucursal'],
                               'stock_cierre': df['stock_n'].astype(int), 'unidades_recibidas': df['rec_n'].astype(int),
                               'lote_id': lote_id, 'archivo_origen': df['_archivo'], 'fila_origen': df['_fila'].astype(int),
                               'run_id': RUN_ID})
        con.execute(text(Q('DELETE FROM silver.inventario WHERE lote_id = :l')), {'l': lote_id})
        escribir(silver, 'silver.inventario', con, chunksize=5000)
        info['filas_rechazadas'] = registrar_rechazos(con, lote_id, 'inventario',
                                                      pd.concat(rech).drop(columns=['fecha_d', 'stock_n', 'rec_n']))
        info['filas_salida'] = len(silver)
    return silver


def silver_clientes(con, lote_id, crm):
    """Minimización: del CRM solo pasan el hash del RUT y atributos analíticos. Nombre y correo se descartan."""
    with etapa(lote_id, 'Silver · clientes', 'silver.clientes') as info:
        info['filas_entrada'] = len(crm)
        df = crm.copy()
        df['rut_norm'] = df['rut'].map(normalizar_rut)
        invalidos = df[df['rut_norm'].isna()].assign(motivo='RUT_INVALIDO', detalle='RUT con dígito verificador inválido')
        df = df[df['rut_norm'].notna()]
        dup = df.duplicated('rut_norm', keep='first')            # el mismo cliente en distintos formatos
        duplicados = df[dup].assign(motivo='CLIENTE_DUPLICADO', detalle='Mismo RUT registrado en otro formato')
        df = df[~dup]
        silver = pd.DataFrame({'cliente_hash': df['rut_norm'].map(seudonimizar), 'segmento': df['segmento'].str.strip(),
                               'comuna': df['comuna'].str.zfill(5), 'region': df['region'].str.strip(),
                               'lote_id': lote_id, 'run_id': RUN_ID})
        con.execute(text(Q('DELETE FROM silver.clientes WHERE lote_id = :l')), {'l': lote_id})
        escribir(silver, 'silver.clientes', con)
        rech = pd.concat([invalidos, duplicados]).drop(columns=['rut_norm', 'nombre', 'email', 'rut'])  # sin datos personales
        info['filas_rechazadas'] = registrar_rechazos(con, lote_id, 'clientes', rech)
        info['filas_salida'] = len(silver)
    return silver

## 7. Capa Gold: modelo dimensional con SCD

| Técnica | Dónde se aplica | Por qué |
|---|---|---|
| **SCD 0** | `dim_fecha` | Un día del calendario nunca cambia. |
| **SCD 1** (sobrescribir) | `dim_sucursal.sucursal`, `tipo_sucursal`, `comuna`, `region`; `dim_canal`; `dim_campana` | Son correcciones de datos, no cambios del negocio: "Maipu" estaba mal escrito y no tiene sentido conservar el error. |
| **SCD 2** (nueva versión) | `dim_producto`, `dim_cliente`, `dim_geografia` | El análisis histórico depende del valor vigente en la fecha de la venta: si los audífonos pasan de Accesorios a Tecnología, las ventas de enero deben seguir sumando en Accesorios. |
| **SCD 3** (columna anterior) | `dim_sucursal.zona_comercial_actual / _anterior` | Al reorganizar zonas, el negocio quiere comparar "zona nueva vs zona anterior" en el mismo reporte, sin multiplicar versiones. |

Convención de vigencia: `valido_desde` inclusivo, `valido_hasta` exclusivo; la versión vigente tiene `valido_hasta = NULL` y `es_actual = 1`.

In [ ]:
FIN_DE_LOS_TIEMPOS = date(9999, 12, 31)


def siguiente_clave(con, tabla, sk):
    v = leer(f'SELECT MAX({sk}) AS m FROM {tabla}', con)['m'].iloc[0]
    return 1 if pd.isna(v) or v < 1 else int(v) + 1


def _iguales(a, b):
    """Compara un valor del origen con el del DW (SQL Server devuelve Decimal y date; SQLite, texto)."""
    na, nb = a is None or (isinstance(a, float) and np.isnan(a)), b is None or (isinstance(b, float) and np.isnan(b))
    if na or nb:
        return na and nb
    try:
        return abs(float(a) - float(b)) < 0.005
    except (TypeError, ValueError):
        return str(a)[:10] == str(b)[:10] if isinstance(a, (date, datetime)) or isinstance(b, (date, datetime)) else str(a) == str(b)


def cargar_scd2(con, lote_id, tabla, sk, nk, atributos, origen, vigencia):
    """SCD Tipo 2: si cambia un atributo, se cierra la versión vigente y se inserta una nueva."""
    with etapa(lote_id, 'Gold · SCD 2', tabla) as info:
        info['filas_entrada'] = len(origen)
        actual = leer(f'SELECT * FROM {tabla} WHERE es_actual = 1 AND {sk} > 0', con)
        m = origen.merge(actual[[sk, nk] + atributos], on=nk, how='left', suffixes=('', '_dw'))
        nuevos = m[m[sk].isna()]
        existentes = m[m[sk].notna()]
        cambia = existentes.apply(lambda r: any(not _iguales(r[a], r[f'{a}_dw']) for a in atributos), axis=1) \
            if len(existentes) else pd.Series(dtype=bool)
        cambiados = existentes[cambia] if len(existentes) else existentes
        # 1) cerrar versiones vigentes que cambiaron
        if len(cambiados):
            con.execute(text(Q(f'UPDATE {tabla} SET valido_hasta = :f, es_actual = 0 WHERE {sk} = :k')),
                        [{'f': vigencia, 'k': int(k)} for k in cambiados[sk]])
        # 2) insertar nuevas filas (clientes/productos nuevos y nuevas versiones)
        insertar = pd.concat([nuevos, cambiados])[[nk] + atributos].copy()
        if len(insertar):
            desde = np.where(insertar[nk].isin(cambiados[nk]), vigencia, date(1900, 1, 1))
            insertar.insert(0, sk, range(siguiente_clave(con, tabla, sk), siguiente_clave(con, tabla, sk) + len(insertar)))
            insertar['valido_desde'] = desde
            insertar['valido_hasta'] = None
            insertar['es_actual'] = 1
            escribir(insertar, tabla, con)
        info['filas_salida'] = len(insertar)
        info['mensaje'] = f'nuevos={len(nuevos)} · nuevas versiones={len(cambiados)} · sin cambios={len(existentes) - len(cambiados)}'
        log.info('    %s', info['mensaje'])
    return len(nuevos), len(cambiados)


def cargar_scd1(con, lote_id, tabla, sk, nk, atributos, origen):
    """SCD Tipo 1: los cambios se sobrescriben (corrección de datos, sin historia)."""
    with etapa(lote_id, 'Gold · SCD 1', tabla) as info:
        info['filas_entrada'] = len(origen)
        actual = leer(f'SELECT * FROM {tabla} WHERE {sk} > 0', con)
        m = origen.merge(actual[[sk, nk] + atributos], on=nk, how='left', suffixes=('', '_dw'))
        nuevos = m[m[sk].isna()][[nk] + atributos].copy()
        cambios = 0
        for _, r in m[m[sk].notna()].iterrows():
            difs = [a for a in atributos if not _iguales(r[a], r[f'{a}_dw'])]
            if difs:
                cambios += 1
                sets = ', '.join(f'{a} = :{a}' for a in difs)
                con.execute(text(Q(f'UPDATE {tabla} SET {sets} WHERE {sk} = :k')), {**{a: r[a] for a in difs}, 'k': int(r[sk])})
                log.info('    SCD1 %s=%s: %s', nk, r[nk], {a: (r[f"{a}_dw"], r[a]) for a in difs})
        if len(nuevos):
            nuevos.insert(0, sk, range(siguiente_clave(con, tabla, sk), siguiente_clave(con, tabla, sk) + len(nuevos)))
            escribir(nuevos, tabla, con)
        info['filas_salida'] = len(nuevos) + cambios
        info['mensaje'] = f'nuevos={len(nuevos)} · sobrescritos={cambios}'
    return len(nuevos), cambios


def cargar_dim_sucursal(con, lote_id, suc, vigencia):
    """SCD 1 para nombre, formato y ubicación + SCD 3 para la zona comercial.
    Estrella: la comuna y la región se copian como atributos planos (no hay FK de dim_sucursal hacia dim_geografia)."""
    geo_actual = leer('SELECT codigo_comuna, comuna, region FROM gold.dim_geografia WHERE es_actual = 1 AND geo_key > 0', con)
    origen = suc.rename(columns={'zona_comercial': 'zona_nueva'}).merge(
        geo_actual, left_on='cod_comuna', right_on='codigo_comuna', how='left')
    origen['codigo_comuna'] = origen['cod_comuna']
    cargar_scd1(con, lote_id, 'gold.dim_sucursal', 'sucursal_key', 'codigo_sucursal',
                ['sucursal', 'tipo_sucursal', 'codigo_comuna', 'comuna', 'region'],
                origen[['codigo_sucursal', 'sucursal', 'tipo_sucursal', 'codigo_comuna', 'comuna', 'region']])
    with etapa(lote_id, 'Gold · SCD 3', 'gold.dim_sucursal (zona comercial)') as info:
        actual = leer('SELECT sucursal_key, codigo_sucursal, zona_comercial_actual FROM gold.dim_sucursal WHERE sucursal_key > 0', con)
        m = actual.merge(origen[['codigo_sucursal', 'zona_nueva']], on='codigo_sucursal')
        primera = m[m['zona_comercial_actual'].isna()]
        cambio = m[m['zona_comercial_actual'].notna() & (m['zona_comercial_actual'] != m['zona_nueva'])]
        if len(primera):
            con.execute(text(Q('UPDATE gold.dim_sucursal SET zona_comercial_actual = :z WHERE sucursal_key = :k')),
                        [{'z': z, 'k': int(k)} for k, z in zip(primera.sucursal_key, primera.zona_nueva)])
        if len(cambio):
            con.execute(text(Q('UPDATE gold.dim_sucursal SET zona_comercial_anterior = zona_comercial_actual, '
                               'zona_comercial_actual = :z, fecha_cambio_zona = :f WHERE sucursal_key = :k')),
                        [{'z': z, 'f': vigencia, 'k': int(k)} for k, z in zip(cambio.sucursal_key, cambio.zona_nueva)])
            for _, r in cambio.iterrows():
                log.info('    SCD3 %s: %s → %s', r.codigo_sucursal, r.zona_comercial_actual, r.zona_nueva)
        info['filas_entrada'] = len(m)
        info['filas_salida'] = len(primera) + len(cambio)
        info['mensaje'] = f'zonas iniciales={len(primera)} · cambios de zona={len(cambio)}'


def cargar_geografia_inicial(con, lote_id, geo):
    """Carga histórica de una dimensión SCD 2: el catálogo oficial ya trae las vigencias de cada versión."""
    with etapa(lote_id, 'Gold · SCD 2 (carga histórica)', 'gold.dim_geografia') as info:
        if contar('gold.dim_geografia', con) > 1:
            info['mensaje'] = 'ya cargada'
            info['filas_entrada'] = info['filas_salida'] = 0
            return
        g = geo.copy()
        g['geo_key'] = range(1, len(g) + 1)
        g['latitud'] = a_numero(g['latitud'])
        g['longitud'] = a_numero(g['longitud'])
        g['valido_desde'] = a_fecha(g['valido_desde'])
        g['valido_hasta'] = a_fecha(g['valido_hasta'])
        g['es_actual'] = g['valido_hasta'].isna().astype(int)
        cols = ['geo_key', 'codigo_comuna', 'codigo_cut_vigente', 'comuna', 'provincia', 'region', 'pais',
                'latitud', 'longitud', 'valido_desde', 'valido_hasta', 'es_actual']
        escribir(g[cols], 'gold.dim_geografia', con)
        info['filas_entrada'] = info['filas_salida'] = len(g)


DIAS = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
MESES = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']
FERIADOS = {'01-01', '04-18', '04-19', '05-01', '05-21', '06-20', '06-29', '07-16', '08-15', '09-18', '09-19',
            '10-12', '10-31', '11-01', '12-08', '12-25'}


def cargar_dim_fecha(con, lote_id, fechas):
    """SCD 0: se generan solo los días que aún no existen (carga incremental)."""
    with etapa(lote_id, 'Gold · SCD 0', 'gold.dim_fecha') as info:
        f = pd.to_datetime(pd.Series(sorted(set(fechas))))
        existentes = set(leer('SELECT fecha_key FROM gold.dim_fecha', con)['fecha_key'])
        d = pd.DataFrame({'fecha_key': (f.dt.year * 10000 + f.dt.month * 100 + f.dt.day).astype(int), 'fecha': f.dt.date,
                          'dia_semana': f.dt.dayofweek.map(lambda i: DIAS[i]), 'es_fin_de_semana': (f.dt.dayofweek >= 5).astype(int),
                          'semana': f.dt.isocalendar().week.astype(int).to_numpy(), 'mes': f.dt.month,
                          'nombre_mes': f.dt.month.map(lambda m: MESES[m - 1]), 'trimestre': f.dt.quarter, 'anio': f.dt.year,
                          'es_feriado': f.dt.strftime('%m-%d').isin(FERIADOS).astype(int)})
        d = d[~d['fecha_key'].isin(existentes)]
        escribir(d, 'gold.dim_fecha', con)
        info['filas_entrada'] = len(f)
        info['filas_salida'] = len(d)


def buscar_scd2(df, con, tabla, sk, nk_df, nk_dim, columna_fecha, extra=()):
    """Lookup 'point-in-time': asigna la versión de la dimensión vigente en la fecha del hecho."""
    dim = leer(f'SELECT {sk}, {nk_dim}, valido_desde, valido_hasta{"".join(", " + e for e in extra)} FROM {tabla} WHERE {sk} > 0', con)
    dim['valido_desde'] = pd.to_datetime(dim['valido_desde']).dt.date
    dim['valido_hasta'] = pd.to_datetime(dim['valido_hasta']).dt.date
    dim['valido_hasta'] = dim['valido_hasta'].where(dim['valido_hasta'].notna(), FIN_DE_LOS_TIEMPOS)
    base = df[[nk_df, columna_fecha]].reset_index().rename(columns={'index': '_i'})
    m = base.merge(dim, left_on=nk_df, right_on=nk_dim, how='left')
    ok = (m['valido_desde'] <= m[columna_fecha]) & (m[columna_fecha] < m['valido_hasta'])
    m = m[ok].drop_duplicates('_i').set_index('_i')
    claves = m[sk].reindex(df.index).fillna(-1).astype(int)
    extras = {e: m[e].reindex(df.index) for e in extra}
    return claves, extras


def cargar_fact_ventas(con, lote_id, sv):
    with etapa(lote_id, 'Gold · hechos', 'gold.fact_ventas') as info:
        info['filas_entrada'] = len(sv)
        # Idempotencia: si el lote ya estaba cargado, se reemplaza (no se duplica)
        borradas = con.execute(text(Q('DELETE FROM gold.fact_ventas WHERE lote_id = :l')), {'l': lote_id}).rowcount
        f = sv.reset_index(drop=True).copy()
        f['fecha'] = pd.to_datetime(f['fecha']).dt.date
        f['producto_key'], ext = buscar_scd2(f, con, 'gold.dim_producto', 'producto_key', 'sku', 'sku', 'fecha', extra=('costo_unitario',))
        f['costo_unitario'] = pd.to_numeric(ext['costo_unitario'])
        f['cliente_key'], _ = buscar_scd2(f, con, 'gold.dim_cliente', 'cliente_key', 'cliente_hash', 'cliente_hash', 'fecha')
        f['geo_key'], _ = buscar_scd2(f, con, 'gold.dim_geografia', 'geo_key', 'cod_comuna_venta', 'codigo_comuna', 'fecha')
        suc = leer('SELECT sucursal_key, codigo_sucursal FROM gold.dim_sucursal', con)
        f['sucursal_key'] = f['codigo_sucursal'].map(dict(zip(suc.codigo_sucursal, suc.sucursal_key))).fillna(-1).astype(int)
        can = leer('SELECT canal_key, canal FROM gold.dim_canal', con)
        f['canal_key'] = f['canal'].map(dict(zip(can.canal, can.canal_key))).astype(int)
        cmp_ = leer('SELECT campana_key, codigo_campana FROM gold.dim_campana', con)
        f['campana_key'] = f['codigo_campana'].map(dict(zip(cmp_.codigo_campana, cmp_.campana_key))).fillna(-1).astype(int)
        f['fecha_key'] = pd.to_datetime(f['fecha']).dt.strftime('%Y%m%d').astype(int)
        f['costo_neto'] = (f['costo_unitario'] * (f['cantidad'] - f['cantidad_devuelta'])).round(2)
        f['margen_bruto'] = (f['monto_neto'] - f['costo_neto']).round(2)
        f['devolucion'] = (f['cantidad_devuelta'] > 0).astype(int)
        cols = ['nro_boleta', 'linea_boleta', 'fecha_key', 'producto_key', 'sucursal_key', 'cliente_key', 'canal_key', 'geo_key',
                'campana_key', 'cantidad', 'precio_unitario', 'monto_bruto', 'monto_descuento', 'cantidad_devuelta', 'monto_devuelto',
                'monto_neto', 'costo_unitario', 'costo_neto', 'margen_bruto', 'devolucion', 'lote_id', 'archivo_origen',
                'fila_origen', 'run_id']
        escribir(f[cols], 'gold.fact_ventas', con)
        info['filas_salida'] = len(f)
        info['mensaje'] = f'filas reemplazadas por re-ejecución: {borradas}'


def cargar_fact_inventario(con, lote_id, si):
    with etapa(lote_id, 'Gold · hechos', 'gold.fact_inventario') as info:
        info['filas_entrada'] = len(si)
        con.execute(text(Q('DELETE FROM gold.fact_inventario WHERE lote_id = :l')), {'l': lote_id})
        f = si.reset_index(drop=True).copy()
        f['fecha'] = pd.to_datetime(f['fecha']).dt.date
        f['producto_key'], _ = buscar_scd2(f, con, 'gold.dim_producto', 'producto_key', 'sku', 'sku', 'fecha')
        suc = leer('SELECT sucursal_key, codigo_sucursal, codigo_comuna FROM gold.dim_sucursal', con)
        f['sucursal_key'] = f['codigo_sucursal'].map(dict(zip(suc.codigo_sucursal, suc.sucursal_key))).fillna(-1).astype(int)
        # Geografía conectada directo al hecho: comuna de la sucursal, versión SCD 2 vigente en la fecha del stock
        f['cod_comuna'] = f['codigo_sucursal'].map(dict(zip(suc.codigo_sucursal, suc.codigo_comuna)))
        f['geo_key'], _ = buscar_scd2(f, con, 'gold.dim_geografia', 'geo_key', 'cod_comuna', 'codigo_comuna', 'fecha')
        f['fecha_key'] = pd.to_datetime(f['fecha']).dt.strftime('%Y%m%d').astype(int)
        f['en_quiebre'] = (f['stock_cierre'] == 0).astype(int)
        cols = ['fecha_key', 'producto_key', 'sucursal_key', 'geo_key', 'stock_cierre', 'unidades_recibidas', 'en_quiebre',
                'lote_id', 'archivo_origen', 'fila_origen', 'run_id']
        escribir(f[cols], 'gold.fact_inventario', con, chunksize=5000)
        info['filas_salida'] = len(f)

## 8. Orquestación: ejecución de los lotes

Un lote se procesa **completo dentro de una sola transacción**: si cualquier etapa falla, SQL Server revierte todo lo que se cargó en ese lote
y el error queda registrado en `audit.etl_log`. Se ejecutan dos lotes:

* **Lote 1 (ene–jun 2025 + histórico 2018):** maestros v1.
* **Lote 2 (jul–dic 2025):** maestros v2 con cambios, que activan SCD 1, SCD 2 y SCD 3.

In [ ]:
def ejecutar_lote(lote_id):
    cfg = FUENTES_LOTE[lote_id]
    vig = cfg['vigencia_maestros']
    log.info('═' * 70)
    log.info('LOTE %s · vigencia de maestros %s', lote_id, vig)
    t = time.time()
    try:
        with engine.begin() as con:                    # ← transacción: todo o nada
            # Idempotencia de la auditoría: al re-ejecutar un lote se reemplazan sus rechazos y correcciones (el etl_log sí se conserva)
            for tabla_audit in ('audit.rechazos', 'audit.correcciones'):
                con.execute(text(Q(f'DELETE FROM {tabla_audit} WHERE lote_id = :l')), {'l': lote_id})
            # ---------- BRONZE ----------
            b_pos = bronze(con, lote_id, 'pos_ventas', cfg['ventas_pos'])
            b_web = bronze(con, lote_id, 'web_pedidos', cfg['ventas_web'], sep=';')
            b_dev = bronze(con, lote_id, 'devoluciones', cfg['devoluciones'])
            b_inv = bronze(con, lote_id, 'stock_diario', cfg['inventario'])
            b_cli = bronze(con, lote_id, 'clientes_crm', cfg['clientes'])
            b_pro = bronze(con, lote_id, 'productos', cfg['productos'])
            b_suc = bronze(con, lote_id, 'sucursales', cfg['sucursales'])
            b_cmp = bronze(con, lote_id, 'campanas', cfg['campanas'])
            b_geo = bronze(con, lote_id, 'geografia_cut', cfg['geografia'])
            b_eq = bronze(con, lote_id, 'equivalencias_productos', cfg['equivalencias'])
            # ---------- SILVER ----------
            productos = b_pro[['sku', 'producto', 'subcategoria', 'categoria', 'marca']].copy()
            productos['sku'] = productos['sku'].str.strip().str.upper()
            productos['precio_lista'] = a_numero(b_pro['precio_lista'])
            productos['costo_unitario'] = a_numero(b_pro['costo_unitario'])
            s_ven = silver_ventas(con, lote_id, b_pos, b_web, b_dev, set(productos['sku']), b_eq)
            s_inv = silver_inventario(con, lote_id, b_inv, set(productos['sku']))
            s_cli = silver_clientes(con, lote_id, b_cli)
            # ---------- GOLD · dimensiones ----------
            # Reproceso histórico: si ya se aplicaron maestros más nuevos, este lote NO los pisa
            # (las dimensiones ya tienen esa historia); solo se recargan sus hechos con lookup por fecha.
            ultima = leer("SELECT MAX(valido_desde) AS f FROM gold.dim_producto WHERE producto_key > 0", con)['f'].iloc[0]
            ultima = pd.to_datetime(ultima).date() if ultima is not None and not pd.isna(ultima) else date(1900, 1, 1)
            reproceso = vig < ultima
            if reproceso:
                log.info('  Reproceso histórico: maestros vigentes desde %s > %s → no se modifican las dimensiones', ultima, vig)
            cargar_geografia_inicial(con, lote_id, b_geo)
            cargar_dim_fecha(con, lote_id, list(s_ven['fecha']) + list(s_inv['fecha']))
            cargar_scd1(con, lote_id, 'gold.dim_canal', 'canal_key', 'canal', ['tipo_canal'],
                        pd.DataFrame({'canal': ['Tienda física', 'E-commerce', 'App'], 'tipo_canal': ['Presencial', 'Digital', 'Digital']}))
            if not reproceso:
                cargar_maestros(con, lote_id, b_cmp, b_suc, productos, s_cli, vig)
            # ---------- GOLD · hechos ----------
            cargar_fact_ventas(con, lote_id, s_ven)
            cargar_fact_inventario(con, lote_id, s_inv)
        log.info('LOTE %s completado en %.1f s', lote_id, time.time() - t)
    except Exception:
        log.error('LOTE %s revertido completo (rollback). Revise audit.etl_log.', lote_id)
        raise
    finally:
        guardar_log()


def cargar_maestros(con, lote_id, b_cmp, b_suc, productos, s_cli, vig):
    """Dimensiones que vienen de los maestros (SCD 1, 2 y 3)."""
    camp = b_cmp.drop(columns=['_archivo', '_fila', '_run_id', '_cargado_en']).copy()
    camp['fecha_inicio'], camp['fecha_fin'] = a_fecha(camp['fecha_inicio']), a_fecha(camp['fecha_fin'])
    cargar_scd1(con, lote_id, 'gold.dim_campana', 'campana_key', 'codigo_campana',
                ['campana', 'tipo_campana', 'fecha_inicio', 'fecha_fin', 'canal_objetivo'], camp)
    cargar_dim_sucursal(con, lote_id, b_suc[['codigo_sucursal', 'sucursal', 'tipo_sucursal', 'cod_comuna', 'zona_comercial']], vig)
    cargar_scd2(con, lote_id, 'gold.dim_producto', 'producto_key', 'sku',
                ['producto', 'subcategoria', 'categoria', 'marca', 'precio_lista', 'costo_unitario'], productos, vig)
    geo_nombre = leer('SELECT codigo_comuna, comuna FROM gold.dim_geografia WHERE es_actual = 1', con)
    cli = s_cli.merge(geo_nombre, left_on='comuna', right_on='codigo_comuna', how='left')
    cli['comuna'] = cli['comuna_y'].fillna(cli['comuna_x'])
    cargar_scd2(con, lote_id, 'gold.dim_cliente', 'cliente_key', 'cliente_hash', ['segmento', 'comuna', 'region'],
                cli[['cliente_hash', 'segmento', 'comuna', 'region']], vig)

In [ ]:
ejecutar_lote('L1-2025H1')

In [ ]:
ejecutar_lote('L2-2025H2')

In [ ]:
print('Filas por capa después de los dos lotes:')
capas = ['bronze.pos_ventas', 'bronze.web_pedidos', 'bronze.devoluciones', 'bronze.stock_diario', 'bronze.clientes_crm',
         'silver.ventas', 'silver.inventario', 'silver.clientes',
         'gold.dim_fecha', 'gold.dim_producto', 'gold.dim_cliente', 'gold.dim_geografia', 'gold.dim_sucursal', 'gold.dim_canal',
         'gold.dim_campana', 'gold.fact_ventas', 'gold.fact_inventario', 'audit.rechazos', 'audit.etl_log']
pd.DataFrame({'tabla': capas, 'filas': [contar(t) for t in capas]}).assign(capa=lambda d: d.tabla.str.split('.').str[0])[['capa', 'tabla', 'filas']]

## 9. Capa Gold: los 4 data marts por área de negocio

Los cuatro data marts **comparten las mismas dimensiones conformadas** (fecha, producto, sucursal, geografía), por lo que sus cifras se pueden cruzar sin contradicciones.

| Data mart | Área usuaria | Hecho de origen | Grano del data mart | Preguntas que responde |
|---|---|---|---|---|
| `dm_comercial_ventas` | Comercial / Gerencia | `fact_ventas` | mes × sucursal × canal × categoría | ¿Qué sucursales, canales y categorías explican la venta neta? ¿Cuál es el ticket promedio? |
| `dm_marketing_campanas` | Marketing | `fact_ventas` + `dim_campana` + `dim_cliente` | mes × campaña × segmento × canal × sucursal × categoría | ¿Qué campañas generan más venta? ¿Cuánto descuento se entregó y a qué segmentos? |
| `dm_finanzas_margen` | Finanzas | `fact_ventas` (costo, margen, devoluciones) | mes × sucursal × categoría | ¿Cuál es el margen por categoría? ¿Cuánto se pierde por devoluciones? |
| `dm_operaciones_inventario` | Operaciones | `fact_inventario` + `fact_ventas` | mes × sucursal × producto | ¿Dónde hay quiebres de stock? ¿Qué productos rotan menos? |

**Granularidad de cada tabla** (los dos hechos tienen granos distintos, por eso nunca se cruzan directamente):

| Tabla | Grano: una fila representa… |
|---|---|
| `fact_ventas` | una línea de boleta (un producto en una venta) |
| `fact_inventario` | la foto del stock de un producto en una sucursal al cierre de un día |
| `agg_ventas_mes_producto_sucursal` | las ventas de un producto en una sucursal en un mes |
| `agg_inventario_mes_producto_sucursal` | el stock (promedio, días en quiebre) de un producto en una sucursal en un mes |
| `vw_tablero_integrado` | un producto en una sucursal en un mes, con ventas **y** stock |

**Regla para unir procesos distintos:** una venta no es lo mismo que una foto diaria del stock. Si se cruzaran `fact_ventas` y `fact_inventario`
fila a fila, cada venta se repetiría por cada día de stock y las cifras se duplicarían. Por eso **primero se agrega cada hecho al mismo grano
(producto–sucursal–mes)** en dos tablas `agg_*`, y recién ahí se unen. El control de calidad de la sección 10 comprueba que la vista integrada
suma exactamente lo mismo que cada hecho por separado.

**Bus Matrix (dimensiones conformadas):** los 4 data marts usan **las mismas tablas de dimensión y las mismas llaves sustitutas**.

| Proceso / data mart | Fecha | Producto | Sucursal | Geografía | Canal | Cliente | Campaña |
|---|---|---|---|---|---|---|---|
| Ventas → Comercial | ✔ | ✔ | ✔ | ✔ | ✔ | | |
| Ventas → Marketing | ✔ | ✔ | ✔ | | ✔ | ✔ | ✔ |
| Ventas → Finanzas | ✔ | ✔ | ✔ | | | | |
| Inventario → Operaciones | ✔ | ✔ | ✔ | ✔ | | | |
| Tablas FK en los hechos | `fecha_key` | `producto_key` | `sucursal_key` | `geo_key` | `canal_key` | `cliente_key` | `campana_key` |

In [ ]:
SQL_DM = {
'gold.dm_comercial_ventas': """
SELECT f.anio, f.mes, s.codigo_sucursal, s.sucursal, s.zona_comercial_actual AS zona_comercial, g.region AS region_venta,
       c.canal, p.categoria,
       SUM(v.cantidad - v.cantidad_devuelta) AS unidades_netas,
       COUNT(DISTINCT v.nro_boleta)          AS boletas,
       SUM(v.monto_bruto)                    AS venta_bruta,
       SUM(v.monto_descuento)                AS descuentos,
       SUM(v.monto_devuelto)                 AS devoluciones,
       SUM(v.monto_neto)                     AS venta_neta
FROM gold.fact_ventas v
JOIN gold.dim_fecha f      ON f.fecha_key = v.fecha_key
JOIN gold.dim_sucursal s   ON s.sucursal_key = v.sucursal_key
JOIN gold.dim_geografia g  ON g.geo_key = v.geo_key
JOIN gold.dim_canal c      ON c.canal_key = v.canal_key
JOIN gold.dim_producto p   ON p.producto_key = v.producto_key
GROUP BY f.anio, f.mes, s.codigo_sucursal, s.sucursal, s.zona_comercial_actual, g.region, c.canal, p.categoria""",
'gold.dm_marketing_campanas': """
SELECT f.anio, f.mes, cp.codigo_campana, cp.campana, cp.tipo_campana, cl.segmento, c.canal, s.codigo_sucursal, p.categoria,
       COUNT(DISTINCT v.nro_boleta)   AS boletas,
       COUNT(DISTINCT v.cliente_key)  AS clientes_distintos,
       SUM(v.monto_bruto)             AS venta_bruta,
       SUM(v.monto_descuento)         AS descuento_otorgado,
       SUM(v.monto_neto)              AS venta_neta,
       SUM(v.margen_bruto)            AS margen_bruto
FROM gold.fact_ventas v
JOIN gold.dim_fecha f     ON f.fecha_key = v.fecha_key
JOIN gold.dim_campana cp  ON cp.campana_key = v.campana_key
JOIN gold.dim_cliente cl  ON cl.cliente_key = v.cliente_key
JOIN gold.dim_canal c     ON c.canal_key = v.canal_key
JOIN gold.dim_sucursal s  ON s.sucursal_key = v.sucursal_key
JOIN gold.dim_producto p  ON p.producto_key = v.producto_key
GROUP BY f.anio, f.mes, cp.codigo_campana, cp.campana, cp.tipo_campana, cl.segmento, c.canal, s.codigo_sucursal, p.categoria""",
'gold.dm_finanzas_margen': """
SELECT f.anio, f.mes, f.trimestre, s.codigo_sucursal, s.sucursal, p.categoria,
       SUM(v.monto_bruto)     AS venta_bruta,
       SUM(v.monto_descuento) AS descuentos,
       SUM(v.monto_devuelto)  AS monto_devuelto,
       SUM(v.monto_neto)      AS venta_neta,
       SUM(v.costo_neto)      AS costo_venta,
       SUM(v.margen_bruto)    AS margen_bruto,
       SUM(v.devolucion)      AS lineas_con_devolucion,
       COUNT(*)               AS lineas
FROM gold.fact_ventas v
JOIN gold.dim_fecha f    ON f.fecha_key = v.fecha_key
JOIN gold.dim_sucursal s ON s.sucursal_key = v.sucursal_key
JOIN gold.dim_producto p ON p.producto_key = v.producto_key
GROUP BY f.anio, f.mes, f.trimestre, s.codigo_sucursal, s.sucursal, p.categoria""",
'gold.dm_operaciones_inventario': """
SELECT i.anio, i.mes, i.codigo_sucursal, i.sucursal, i.sku, i.producto, i.categoria,
       i.dias_con_registro, i.stock_promedio, i.dias_en_quiebre, i.unidades_recibidas,
       COALESCE(v.unidades_vendidas, 0) AS unidades_vendidas
FROM gold.agg_inventario_mes_producto_sucursal i
LEFT JOIN gold.agg_ventas_mes_producto_sucursal v
       ON v.anio = i.anio AND v.mes = i.mes AND v.codigo_sucursal = i.codigo_sucursal AND v.sku = i.sku""",
}

# Paso previo: cada hecho se agrega por separado al MISMO grano (producto–sucursal–mes) antes de unirlos
SQL_AGG = {
'gold.agg_ventas_mes_producto_sucursal': """
SELECT f.anio, f.mes, s.codigo_sucursal, MAX(s.sucursal) AS sucursal, MAX(s.zona_comercial_actual) AS zona_comercial,
       p.sku, MAX(p.producto) AS producto, MAX(p.categoria) AS categoria,
       SUM(v.cantidad - v.cantidad_devuelta) AS unidades_vendidas,
       SUM(v.monto_bruto)     AS venta_bruta,
       SUM(v.monto_descuento) AS descuento_otorgado,
       SUM(v.monto_devuelto)  AS monto_devuelto,
       SUM(v.monto_neto)      AS venta_neta,
       SUM(v.costo_neto)      AS costo_venta,
       SUM(v.margen_bruto)    AS margen_bruto,
       SUM(CASE WHEN v.campana_key <> -1 THEN v.monto_neto ELSE 0 END) AS venta_neta_con_campana,
       COUNT(*)               AS lineas_vendidas
FROM gold.fact_ventas v
JOIN gold.dim_fecha f    ON f.fecha_key = v.fecha_key
JOIN gold.dim_sucursal s ON s.sucursal_key = v.sucursal_key
JOIN gold.dim_producto p ON p.producto_key = v.producto_key
GROUP BY f.anio, f.mes, s.codigo_sucursal, p.sku""",
'gold.agg_inventario_mes_producto_sucursal': """
SELECT f.anio, f.mes, s.codigo_sucursal, MAX(s.sucursal) AS sucursal, MAX(s.zona_comercial_actual) AS zona_comercial,
       p.sku, MAX(p.producto) AS producto, MAX(p.categoria) AS categoria,
       COUNT(*) AS dias_con_registro,
       AVG(CAST(fi.stock_cierre AS DECIMAL(12,2))) AS stock_promedio,
       SUM(fi.en_quiebre) AS dias_en_quiebre,
       SUM(fi.unidades_recibidas) AS unidades_recibidas
FROM gold.fact_inventario fi
JOIN gold.dim_fecha f    ON f.fecha_key = fi.fecha_key
JOIN gold.dim_sucursal s ON s.sucursal_key = fi.sucursal_key
JOIN gold.dim_producto p ON p.producto_key = fi.producto_key
GROUP BY f.anio, f.mes, s.codigo_sucursal, p.sku""",
}


MONEDA = re.compile(r'(venta|margen_bruto|descuento|devuelto|devoluciones|costo|monto|ticket)')


def tipos_sql(df):
    """Tipos explícitos para SQL Server: dinero en DECIMAL(18,2) (sin errores de redondeo), conteos en INT,
    índices y promedios en DECIMAL(18,4) y textos en NVARCHAR(120)."""
    from sqlalchemy.types import NVARCHAR, DECIMAL, Integer
    tipos = {}
    for c in df.columns:
        if not pd.api.types.is_numeric_dtype(df[c]):
            tipos[c] = NVARCHAR(120)
        elif MONEDA.search(c) and not c.endswith('_pct'):
            tipos[c] = DECIMAL(18, 2)
        elif pd.api.types.is_integer_dtype(df[c]) or (df[c].dropna() % 1 == 0).all() and c in ('anio', 'mes', 'trimestre'):
            tipos[c] = Integer()
        else:
            tipos[c] = DECIMAL(18, 4)
    return tipos


def construir_data_marts():
    with engine.begin() as con:
        for v in VISTAS:
            con.execute(text(Q(f'DROP VIEW IF EXISTS {v}')))
        for tabla, sql in list(SQL_AGG.items()) + list(SQL_DM.items()):
            with etapa('GOLD', 'Gold · agregado' if '.agg_' in tabla else 'Gold · data mart', tabla) as info:
                df = leer(sql, con)
                if tabla == 'gold.dm_comercial_ventas':
                    df['ticket_promedio'] = (df['venta_neta'] / df['boletas']).round(0)
                if tabla == 'gold.dm_finanzas_margen':
                    df['margen_pct'] = (df['margen_bruto'] / df['venta_neta'].where(df['venta_neta'] != 0)).round(4)
                    df['tasa_devolucion_pct'] = (df['monto_devuelto'] / df['venta_bruta']).round(4)
                if 'stock_promedio' in df:
                    df['stock_promedio'] = df['stock_promedio'].astype(float).round(2)
                if tabla == 'gold.dm_operaciones_inventario':
                    df['rotacion'] = (df['unidades_vendidas'] / df['stock_promedio'].where(df['stock_promedio'] > 0)).round(3)
                    df['cobertura_dias'] = (df['stock_promedio'] / (df['unidades_vendidas'] / df['dias_con_registro']).where(
                        df['unidades_vendidas'] > 0)).round(1)
                esquema, nombre = tabla.split('.')
                con.execute(text(Q(f'DROP TABLE IF EXISTS {tabla}')))
                if MOTOR == 'mssql':
                    df.to_sql(nombre, con, schema=esquema, index=False, chunksize=2000, dtype=tipos_sql(df))
                else:
                    df.to_sql(f'{esquema}_{nombre}', con, index=False)
                info['filas_salida'] = len(df)
        # Vista integrada (grano producto–sucursal–mes): une los dos agregados, NUNCA los hechos fila a fila.
        # Las llaves se toman de ambos lados (UNION) para no perder productos con stock sin ventas ni ventas sin registro de stock.
        con.execute(text(Q("""
CREATE VIEW gold.vw_tablero_integrado AS
SELECT k.anio, k.mes, k.codigo_sucursal,
       COALESCE(v.sucursal, i.sucursal) AS sucursal, COALESCE(v.zona_comercial, i.zona_comercial) AS zona_comercial,
       k.sku, COALESCE(v.producto, i.producto) AS producto, COALESCE(v.categoria, i.categoria) AS categoria,
       COALESCE(v.unidades_vendidas, 0)      AS unidades_vendidas,
       COALESCE(v.venta_neta, 0)             AS venta_neta,
       COALESCE(v.margen_bruto, 0)           AS margen_bruto,
       COALESCE(v.monto_devuelto, 0)         AS monto_devuelto,
       COALESCE(v.venta_neta_con_campana, 0) AS venta_neta_con_campana,
       COALESCE(v.descuento_otorgado, 0)     AS descuento_otorgado,
       i.stock_promedio, i.dias_en_quiebre, i.dias_con_registro,
       CASE WHEN i.stock_promedio > 0 THEN COALESCE(v.unidades_vendidas, 0) / i.stock_promedio END AS rotacion
FROM (SELECT anio, mes, codigo_sucursal, sku FROM gold.agg_ventas_mes_producto_sucursal
      UNION
      SELECT anio, mes, codigo_sucursal, sku FROM gold.agg_inventario_mes_producto_sucursal) k
LEFT JOIN gold.agg_ventas_mes_producto_sucursal v
       ON v.anio = k.anio AND v.mes = k.mes AND v.codigo_sucursal = k.codigo_sucursal AND v.sku = k.sku
LEFT JOIN gold.agg_inventario_mes_producto_sucursal i
       ON i.anio = k.anio AND i.mes = k.mes AND i.codigo_sucursal = k.codigo_sucursal AND i.sku = k.sku""")))
        # Vista para consultas en lenguaje natural (Copilot / Vanna.ai): nombres de negocio y sin datos personales
        con.execute(text(Q("""
CREATE VIEW gold.vw_ia_ventas AS
SELECT f.fecha, f.anio, f.mes, f.nombre_mes, f.semana, s.sucursal, s.zona_comercial_actual AS zona_comercial,
       g.comuna AS comuna_venta, g.region AS region_venta, c.canal, p.sku, p.producto, p.categoria, p.subcategoria, p.marca,
       cp.campana, cl.segmento AS segmento_cliente,
       v.nro_boleta, v.cantidad, v.monto_bruto, v.monto_descuento, v.monto_devuelto, v.monto_neto AS venta_neta, v.margen_bruto
FROM gold.fact_ventas v
JOIN gold.dim_fecha f     ON f.fecha_key = v.fecha_key
JOIN gold.dim_sucursal s  ON s.sucursal_key = v.sucursal_key
JOIN gold.dim_geografia g ON g.geo_key = v.geo_key
JOIN gold.dim_canal c     ON c.canal_key = v.canal_key
JOIN gold.dim_producto p  ON p.producto_key = v.producto_key
JOIN gold.dim_campana cp  ON cp.campana_key = v.campana_key
JOIN gold.dim_cliente cl  ON cl.cliente_key = v.cliente_key""")))
    guardar_log()


construir_data_marts()
for t in TABLAS_AGG + TABLAS_DM:
    print(f'{t}: {contar(t):,} filas')

In [ ]:
print('DATA MART COMERCIAL · venta neta y ticket promedio por canal (año 2025)')
display(leer("""SELECT canal, SUM(venta_neta) AS venta_neta, SUM(boletas) AS boletas, SUM(venta_neta) / SUM(boletas) AS ticket_promedio
                FROM gold.dm_comercial_ventas WHERE anio = 2025 GROUP BY canal ORDER BY venta_neta DESC"""))
print('DATA MART MARKETING · campañas 2025')
display(leer("""SELECT campana, tipo_campana, SUM(boletas) AS boletas, SUM(venta_neta) AS venta_neta, SUM(descuento_otorgado) AS descuento,
                       SUM(descuento_otorgado) * 100.0 / SUM(venta_bruta) AS descuento_pct
                FROM gold.dm_marketing_campanas WHERE codigo_campana <> 'SIN_CAMPANA' GROUP BY campana, tipo_campana ORDER BY venta_neta DESC"""))
print('DATA MART FINANZAS · margen por categoría 2025')
display(leer("""SELECT categoria, SUM(venta_neta) AS venta_neta, SUM(costo_venta) AS costo, SUM(margen_bruto) AS margen,
                       SUM(margen_bruto) * 100.0 / SUM(venta_neta) AS margen_pct, SUM(monto_devuelto) AS devuelto
                FROM gold.dm_finanzas_margen WHERE anio = 2025 GROUP BY categoria ORDER BY margen DESC"""))
print('DATA MART OPERACIONES · sucursales con más días de quiebre (producto-día)')
display(leer("""SELECT codigo_sucursal, sucursal, SUM(dias_en_quiebre) AS dias_quiebre, SUM(unidades_vendidas) AS unidades_vendidas,
                       AVG(rotacion) AS rotacion_promedio
                FROM gold.dm_operaciones_inventario GROUP BY codigo_sucursal, sucursal ORDER BY dias_quiebre DESC"""))

In [ ]:
print('VISTA INTEGRADA · grano producto–sucursal–mes (muestra: diciembre 2025, Providencia)')
display(leer("""SELECT sku, producto, categoria, unidades_vendidas, venta_neta, margen_bruto, venta_neta_con_campana,
                       stock_promedio, dias_en_quiebre, rotacion
                FROM gold.vw_tablero_integrado WHERE anio = 2025 AND mes = 12 AND codigo_sucursal = 'SUC-01' ORDER BY venta_neta DESC""").head(10))
print('Cómo la ve la gerencia: la misma vista resumida por sucursal (diciembre 2025, Tecnología)')
leer("""SELECT sucursal, zona_comercial, SUM(venta_neta) AS venta_neta, SUM(margen_bruto) AS margen_bruto,
               SUM(venta_neta_con_campana) AS venta_con_campana, SUM(dias_en_quiebre) AS dias_en_quiebre
        FROM gold.vw_tablero_integrado WHERE anio = 2025 AND mes = 12 AND categoria = 'Tecnología'
        GROUP BY sucursal, zona_comercial ORDER BY venta_neta DESC""")

### Capa semántica para consultas con IA (Power BI + Copilot / Vanna.ai)

La IA no debe adivinar qué significa "venta neta" ni consultar las tablas de origen. Por eso se publica en `sem` la definición oficial de cada KPI
y un diccionario de datos con descripciones y sinónimos de negocio. Estas tablas son el contexto que se entrega a Copilot (descripciones del modelo semántico)
o a Vanna.ai (entrenamiento con documentación). Las consultas de IA se hacen solo sobre `gold.vw_ia_ventas` y los data marts, nunca sobre Bronze ni Silver.

In [ ]:
KPIS = pd.DataFrame([
    ('Venta neta', 'Comercial', 'Monto vendido descontando descuentos, anulaciones y devoluciones.',
     'SUM(monto_neto)  donde monto_neto = monto_bruto - monto_descuento - monto_devuelto', 'línea de boleta'),
    ('Ticket promedio', 'Comercial', 'Venta neta promedio por boleta.', 'SUM(monto_neto) / COUNT(DISTINCT nro_boleta)', 'boleta'),
    ('Margen bruto', 'Finanzas', 'Venta neta menos el costo de las unidades no devueltas.', 'SUM(monto_neto) - SUM(costo_neto)', 'línea de boleta'),
    ('Margen %', 'Finanzas', 'Margen bruto sobre venta neta.', 'SUM(margen_bruto) / SUM(monto_neto)', 'mes x sucursal x categoría'),
    ('Tasa de devolución', 'Finanzas', 'Proporción del monto bruto que fue devuelto o anulado.', 'SUM(monto_devuelto) / SUM(monto_bruto)', 'mes x sucursal x categoría'),
    ('Venta con campaña', 'Marketing', 'Venta neta de boletas asociadas a una campaña.', "SUM(monto_neto) WHERE codigo_campana <> 'SIN_CAMPANA'", 'mes x campaña'),
    ('Descuento otorgado %', 'Marketing', 'Descuento entregado sobre la venta bruta.', 'SUM(monto_descuento) / SUM(monto_bruto)', 'mes x campaña'),
    ('Días en quiebre', 'Operaciones', 'Días en que un producto terminó con stock 0 en una sucursal.', 'SUM(en_quiebre)', 'producto x sucursal x día'),
    ('Rotación', 'Operaciones', 'Unidades vendidas sobre stock promedio del mes.', 'SUM(unidades_vendidas) / AVG(stock_cierre)', 'mes x sucursal x producto'),
], columns=['kpi', 'data_mart', 'definicion', 'formula_sql', 'grano'])

DICC = pd.DataFrame([
    ('gold.vw_ia_ventas', 'venta_neta', 'Monto efectivamente vendido de la línea, ya descontados descuento y devoluciones.', 'ventas netas, ingresos, facturación neta', 'No volver a filtrar devoluciones: ya están descontadas.'),
    ('gold.vw_ia_ventas', 'region_venta', 'Región donde ocurrió la venta (sucursal o comuna de despacho web), según la división vigente en la fecha.', 'zona, región', 'No confundir con la región donde vive el cliente.'),
    ('gold.vw_ia_ventas', 'segmento_cliente', 'Segmento del cliente vigente en la fecha de la venta.', 'tipo de cliente', 'Cliente no identificado = venta sin RUT válido.'),
    ('gold.vw_ia_ventas', 'categoria', 'Categoría del producto vigente en la fecha de la venta (historia SCD 2).', 'línea, familia', 'Para la categoría actual usar dim_producto con es_actual = 1.'),
    ('gold.vw_ia_ventas', 'canal', 'Canal de venta: Tienda física, E-commerce o App.', 'canal, online, tienda, app, web', 'E-commerce y App son el canal digital (tipo_canal).'),
    ('gold.vw_ia_ventas', 'campana', 'Campaña comercial asociada a la boleta.', 'promoción, evento', "Sin campaña = venta regular."),
    ('gold.vw_ia_ventas', 'zona_comercial', 'Zona comercial actual de la sucursal (SCD 3).', 'zona', 'La zona anterior está en dim_sucursal.zona_comercial_anterior.'),
    ('gold.vw_ia_ventas', 'nro_boleta', 'Número de boleta (dimensión degenerada).', 'boleta, transacción, ticket', 'Para contar compras usar COUNT(DISTINCT nro_boleta).'),
    ('gold.dm_operaciones_inventario', 'stock_promedio', 'Stock promedio diario del mes. Es semiaditivo: no se suma en el tiempo.', 'inventario, existencias', 'Promediar, no sumar, entre meses.'),
    ('gold.dm_operaciones_inventario', 'dias_en_quiebre', 'Días del mes en que el producto terminó con stock 0.', 'quiebre, sin stock', ''),
], columns=['objeto', 'columna', 'descripcion', 'sinonimos', 'regla'])

with engine.begin() as con:
    ejecutar('DELETE FROM sem.kpi', con)
    ejecutar('DELETE FROM sem.diccionario', con)
    ejecutar('DELETE FROM sem.mapa_fuentes', con)
    escribir(KPIS, 'sem.kpi', con)
    escribir(DICC, 'sem.diccionario', con)
    escribir(MAPA_FUENTES, 'sem.mapa_fuentes', con)
leer('SELECT kpi, data_mart, formula_sql FROM sem.kpi')

### Diccionario de datos

Describe **todas las tablas nuevas** de Silver, Gold (dimensiones, hechos, agregados, data marts y vistas), auditoría y capa semántica:
tipo de dato, rol (PK, FK, clave natural, medida, atributo, control SCD, linaje) y significado de negocio.
Se guarda en `sem.diccionario_datos` y se exporta a `data/salida/diccionario_datos.xlsx`.

In [ ]:
DESC = {
    # claves y tiempo
    'fecha_key': 'Clave de fecha AAAAMMDD', 'fecha': 'Fecha calendario', 'dia_semana': 'Día de la semana', 'es_fin_de_semana': '1 si es sábado o domingo',
    'semana': 'Semana ISO del año', 'mes': 'Mes (1–12)', 'nombre_mes': 'Nombre del mes', 'trimestre': 'Trimestre (1–4)', 'anio': 'Año', 'es_feriado': '1 si es feriado en Chile',
    'producto_key': 'Clave sustituta del producto (versión SCD 2)', 'sku': 'SKU corporativo del producto (clave natural)', 'producto': 'Nombre del producto',
    'subcategoria': 'Subcategoría del producto', 'categoria': 'Categoría del producto vigente en la fecha', 'marca': 'Marca', 'precio_lista': 'Precio de lista (CLP, sin IVA)',
    'costo_unitario': 'Costo unitario (CLP) vigente en la fecha', 'cliente_key': 'Clave sustituta del cliente (versión SCD 2); -1 = no identificado',
    'cliente_hash': 'RUT seudonimizado con SHA-256 y sal (clave natural, sin dato personal)', 'segmento': 'Segmento del cliente (Nuevo, Ocasional, Frecuente, VIP)',
    'comuna': 'Comuna', 'region': 'Región', 'provincia': 'Provincia', 'pais': 'País', 'geo_key': 'Clave sustituta de la comuna (versión SCD 2)',
    'codigo_comuna': 'Código CUT actual de la comuna (clave natural)', 'codigo_cut_vigente': 'Código CUT que tenía la comuna en esa versión',
    'latitud': 'Latitud del centroide de la comuna', 'longitud': 'Longitud del centroide de la comuna',
    'sucursal_key': 'Clave sustituta de la sucursal; -1 = sin asignar', 'codigo_sucursal': 'Código de la sucursal (clave natural)', 'sucursal': 'Nombre de la sucursal',
    'tipo_sucursal': 'Formato del local (Mall, Calle, Online)', 'zona_comercial_actual': 'Zona comercial actual (SCD 3)', 'zona_comercial_anterior': 'Zona comercial anterior (SCD 3)',
    'fecha_cambio_zona': 'Fecha del último cambio de zona (SCD 3)', 'zona_comercial': 'Zona comercial actual de la sucursal',
    'canal_key': 'Clave del canal', 'canal': 'Canal de venta (Tienda física, E-commerce, App)', 'tipo_canal': 'Presencial o digital',
    'campana_key': 'Clave de la campaña; -1 = sin campaña', 'codigo_campana': 'Código de la campaña', 'campana': 'Nombre de la campaña',
    'tipo_campana': 'Tipo de campaña', 'fecha_inicio': 'Inicio de la campaña', 'fecha_fin': 'Fin de la campaña', 'canal_objetivo': 'Canal al que apunta la campaña',
    'valido_desde': 'Inicio de vigencia de la versión (inclusivo)', 'valido_hasta': 'Fin de vigencia (exclusivo); vacío = vigente', 'es_actual': '1 = versión vigente',
    # hechos
    'nro_boleta': 'Número de boleta o pedido (dimensión degenerada)', 'linea_boleta': 'Número de línea dentro de la boleta',
    'cantidad': 'Unidades vendidas en la línea', 'precio_unitario': 'Precio unitario cobrado (CLP)', 'monto_bruto': 'cantidad × precio_unitario',
    'monto_descuento': 'Descuento aplicado a la línea (CLP)', 'cantidad_devuelta': 'Unidades devueltas o anuladas', 'monto_devuelto': 'Monto devuelto o anulado (CLP)',
    'monto_neto': 'Venta neta de la línea = monto_bruto − monto_descuento − monto_devuelto', 'costo_neto': 'Costo de las unidades no devueltas',
    'margen_bruto': 'monto_neto − costo_neto', 'devolucion': '1 si la línea tuvo devolución o anulación',
    'stock_cierre': 'Unidades en stock al cierre del día (semiaditivo)', 'unidades_recibidas': 'Unidades recibidas por reposición', 'en_quiebre': '1 si el stock cerró en 0',
    # linaje y auditoría
    'lote_id': 'Lote de carga', 'archivo_origen': 'Archivo de origen de la fila (linaje)', 'fila_origen': 'Fila del archivo de origen (linaje)', 'run_id': 'Identificador de la ejecución',
    'origen': 'Sistema de origen (POS, ECOMMERCE, LEGADO)', 'estado': 'Estado de la venta (OK, ANULADA, DEVUELTA)', 'cod_comuna_venta': 'Comuna donde ocurrió la venta',
    # agregados y marts
    'unidades_vendidas': 'Unidades vendidas netas de devoluciones', 'unidades_netas': 'Unidades vendidas netas de devoluciones', 'boletas': 'Boletas distintas',
    'venta_bruta': 'Suma de monto_bruto', 'descuentos': 'Suma de descuentos', 'descuento_otorgado': 'Suma de descuentos', 'devoluciones': 'Suma de montos devueltos',
    'venta_neta': 'Suma de monto_neto (regla única)', 'costo_venta': 'Suma de costo_neto', 'ticket_promedio': 'venta_neta / boletas', 'margen_pct': 'margen_bruto / venta_neta',
    'tasa_devolucion_pct': 'monto_devuelto / venta_bruta', 'lineas': 'Líneas de venta', 'lineas_vendidas': 'Líneas de venta', 'lineas_con_devolucion': 'Líneas con devolución',
    'clientes_distintos': 'Clientes distintos (incluye no identificado)', 'venta_neta_con_campana': 'Venta neta asociada a una campaña',
    'region_venta': 'Región donde ocurrió la venta', 'comuna_venta': 'Comuna donde ocurrió la venta', 'segmento_cliente': 'Segmento del cliente en la fecha de la venta',
    'dias_con_registro': 'Días con registro de stock en el mes', 'stock_promedio': 'Stock promedio diario del mes (no sumar entre meses)',
    'dias_en_quiebre': 'Días del mes con stock 0', 'rotacion': 'unidades_vendidas / stock_promedio', 'cobertura_dias': 'Días que alcanza el stock al ritmo de venta del mes',
}
# Descripciones que dependen de la tabla (tienen prioridad sobre DESC)
DESC_TABLA_COL = {
    # auditoría
    'audit.etl_log.etapa': 'Etapa del pipeline (p. ej. Bronze · ingesta, Silver · ventas, Gold · SCD 2)',
    'audit.etl_log.objeto': 'Tabla que procesó la etapa',
    'audit.etl_log.filas_entrada': 'Filas que recibió la etapa', 'audit.etl_log.filas_salida': 'Filas que escribió la etapa',
    'audit.etl_log.filas_rechazadas': 'Filas enviadas a cuarentena (audit.rechazos)',
    'audit.etl_log.estado': 'Resultado de la etapa: OK o ERROR', 'audit.etl_log.mensaje': 'Detalle de la etapa o mensaje de error',
    'audit.etl_log.inicio': 'Fecha y hora de inicio de la etapa', 'audit.etl_log.fin': 'Fecha y hora de término de la etapa',
    'audit.etl_log.segundos': 'Duración de la etapa en segundos',
    'audit.rechazos.entidad': 'Entidad rechazada: ventas, inventario o clientes', 'audit.rechazos.archivo': 'Archivo de origen del registro rechazado',
    'audit.rechazos.fila': 'Fila del archivo de origen', 'audit.rechazos.motivo': 'Código del motivo (DUPLICADO, PRECIO_INVALIDO, SKU_INEXISTENTE…)',
    'audit.rechazos.detalle': 'Explicación del rechazo', 'audit.rechazos.registro': 'Registro original sin datos personales (para revisión)',
    'audit.correcciones.fuente': 'Sistema donde se aplicó la corrección (POS, ECOMMERCE, LEGADO)',
    'audit.correcciones.regla': 'Regla de limpieza aplicada', 'audit.correcciones.dimension_calidad': 'Dimensión de calidad que corrige la regla',
    'audit.correcciones.capa': 'Capa donde se aplicó (Silver)', 'audit.correcciones.filas': 'Filas a las que se aplicó la regla',
    'audit.controles_calidad.control': 'Nombre del control de calidad', 'audit.controles_calidad.dimension_calidad': 'Dimensión de calidad que mide',
    'audit.controles_calidad.resultado': 'Resultado obtenido en la ejecución', 'audit.controles_calidad.ok': '1 = control aprobado · 0 = revisar',
    # capa semántica
    'sem.kpi.kpi': 'Nombre del indicador', 'sem.kpi.data_mart': 'Data mart que lo responde', 'sem.kpi.definicion': 'Definición de negocio',
    'sem.kpi.formula_sql': 'Fórmula SQL oficial del indicador', 'sem.kpi.grano': 'Nivel de detalle en que se calcula',
    'sem.diccionario.objeto': 'Tabla o vista consultada por la IA', 'sem.diccionario.columna': 'Columna descrita',
    'sem.diccionario.descripcion': 'Significado de negocio de la columna', 'sem.diccionario.sinonimos': 'Palabras con que el usuario puede referirse a ella',
    'sem.diccionario.regla': 'Regla para que la IA genere el SQL correcto',
    'sem.mapa_fuentes.sistema_origen': 'Sistema que genera el dato', 'sem.mapa_fuentes.area': 'Área dueña del sistema',
    'sem.mapa_fuentes.archivos': 'Archivos que entrega', 'sem.mapa_fuentes.formato': 'Formato y particularidades del archivo',
    'sem.mapa_fuentes.tabla_bronze': 'Tabla donde entra (copia cruda)', 'sem.mapa_fuentes.limpieza_silver': 'Limpieza aplicada en Silver',
    'sem.mapa_fuentes.integracion': 'Cómo se integra con las otras fuentes', 'sem.mapa_fuentes.destino_gold': 'Tablas Gold y data marts que alimenta',
    'sem.diccionario_datos.capa': 'Capa Medallion', 'sem.diccionario_datos.tabla': 'Tabla o vista', 'sem.diccionario_datos.columna': 'Columna',
    'sem.diccionario_datos.tipo_dato': 'Tipo de dato en la base', 'sem.diccionario_datos.rol': 'Rol en el modelo (PK, FK, medida, atributo…)',
    'sem.diccionario_datos.descripcion': 'Significado de la columna',
    # bronze (dato crudo, todo como texto)
    'bronze.pos_ventas.linea': 'Número de línea de la boleta', 'bronze.pos_ventas.fecha': 'Fecha en formatos mixtos (DD/MM/AAAA, AAAA/MM/DD, AAAA-MM-DD)',
    'bronze.pos_ventas.sku': 'SKU tal como lo digitó el POS (puede venir en minúsculas o con espacios)',
    'bronze.pos_ventas.rut_cliente': 'RUT del cliente en 3 formatos (dato personal: solo Bronze)', 'bronze.pos_ventas.canal': 'Canal escrito de distintas formas',
    'bronze.pos_ventas.precio_unitario': 'Precio como texto (a veces con coma decimal o N/A)', 'bronze.pos_ventas.descuento': 'Descuento en pesos',
    'bronze.pos_ventas.estado': 'OK, ANULADA o DEVUELTA',
    'bronze.web_pedidos.id_pedido': 'Número de pedido web', 'bronze.web_pedidos.item': 'Línea del pedido', 'bronze.web_pedidos.fecha_hora': 'Fecha-hora ISO del pedido',
    'bronze.web_pedidos.codigo_producto': 'Código propio del e-commerce (WEB-####)', 'bronze.web_pedidos.rut': 'RUT del cliente (dato personal: solo Bronze)',
    'bronze.web_pedidos.comuna_despacho': 'Comuna de despacho (código CUT)', 'bronze.web_pedidos.unidades': 'Unidades pedidas',
    'bronze.web_pedidos.precio': 'Precio con coma decimal', 'bronze.web_pedidos.descuento_pct': 'Descuento en porcentaje',
    'bronze.web_pedidos.cupon_campana': 'Código de campaña usado', 'bronze.web_pedidos.plataforma': 'WEB o APP', 'bronze.web_pedidos.estado_pedido': 'ENTREGADO, CANCELADO o DEVUELTO',
    'bronze.devoluciones.linea': 'Línea de la boleta devuelta', 'bronze.devoluciones.fecha_devolucion': 'Fecha de la devolución', 'bronze.devoluciones.motivo': 'Motivo declarado por el cliente',
    'bronze.stock_diario.fecha': 'Día del registro de stock', 'bronze.clientes_crm.rut': 'RUT en 3 formatos (dato personal: solo Bronze)',
    'bronze.clientes_crm.nombre': 'Nombre del cliente (dato personal: no pasa a Silver)', 'bronze.clientes_crm.email': 'Correo del cliente (dato personal: no pasa a Silver)',
    'bronze.sucursales.cod_comuna': 'Código CUT de la comuna de la sucursal', 'bronze.sucursales.zona_comercial': 'Zona comercial asignada',
    'bronze.equivalencias_productos.sistema': 'Sistema de origen (POS, ECOMMERCE, LEGADO)', 'bronze.equivalencias_productos.codigo_origen': 'Código del producto en ese sistema',
    'bronze.equivalencias_productos.sku': 'SKU corporativo equivalente',
    '_archivo': 'Archivo de origen (linaje)', '_fila': 'Fila del archivo de origen (linaje)', '_run_id': 'Ejecución que cargó la fila', '_cargado_en': 'Fecha y hora de carga',
}

ROL = {'PK': {'fecha_key@dim_fecha', 'producto_key@dim_producto', 'cliente_key@dim_cliente', 'geo_key@dim_geografia', 'sucursal_key@dim_sucursal',
              'canal_key@dim_canal', 'campana_key@dim_campana'},
       'NK': {'sku', 'cliente_hash', 'codigo_comuna', 'codigo_sucursal', 'codigo_campana'},
       'SCD': {'valido_desde', 'valido_hasta', 'es_actual', 'zona_comercial_actual', 'zona_comercial_anterior', 'fecha_cambio_zona'},
       'Linaje': {'lote_id', 'archivo_origen', 'fila_origen', 'run_id', 'origen', '_archivo', '_fila', '_run_id', '_cargado_en'},
       'DD': {'nro_boleta'}}


def rol_de(tabla, col, tipo):
    t = tabla.split('.')[-1].replace('gold_', '').replace('silver_', '')
    if f'{col}@{t}' in ROL['PK']:
        return 'PK'
    if col.endswith('_key'):
        return 'FK' if t.startswith('fact') or t == 'dim_sucursal' else 'Clave'
    if col in ROL['DD'] and t.startswith('fact'):
        return 'PK · Dimensión degenerada'
    if col == 'linea_boleta' and t.startswith('fact'):
        return 'PK'
    for r in ('NK', 'SCD', 'Linaje'):
        if col in ROL[r]:
            return {'NK': 'Clave natural', 'SCD': 'Control SCD', 'Linaje': 'Linaje'}[r]
    if (t.startswith(('fact', 'agg', 'dm')) or t.startswith('vw')) and re.search(r'(INT|DECIMAL|FLOAT|REAL|NUMERIC|BIGINT)', tipo.upper()) \
            and not col.endswith(('anio', 'mes', 'trimestre')) and col not in ('anio', 'mes', 'trimestre'):
        return 'Medida'
    return 'Atributo'


def columnas_bd():
    if MOTOR == 'mssql':
        c = leer("""SELECT TABLE_SCHEMA AS esquema, TABLE_NAME AS tabla, COLUMN_NAME AS columna,
                         DATA_TYPE + COALESCE('(' + CAST(CHARACTER_MAXIMUM_LENGTH AS VARCHAR) + ')', '') AS tipo_dato, ORDINAL_POSITION AS pos
                  FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_SCHEMA IN ('bronze', 'silver', 'gold', 'audit', 'sem')""")
        c['tabla'] = c['esquema'] + '.' + c['tabla']
        return c.sort_values(['esquema', 'tabla', 'pos'])
    filas = []
    for t in leer("SELECT name FROM sqlite_master WHERE type IN ('table','view') ORDER BY name")['name']:
        esquema = t.split('_')[0]
        if esquema not in ('bronze', 'silver', 'gold', 'audit', 'sem'):
            continue
        for _, r in pd.read_sql(text(f'PRAGMA table_info({t})'), engine).iterrows():
            filas.append((esquema, esquema + '.' + t[len(esquema) + 1:], r['name'], r['type'] or 'calculado (vista)', r['cid']))
    return pd.DataFrame(filas, columns=['esquema', 'tabla', 'columna', 'tipo_dato', 'pos'])


def construir_diccionario():
    c = columnas_bd()
    DESC_TABLA = {'audit': 'Auditoría del pipeline', 'sem': 'Capa semántica'}
    especifica = (c['tabla'] + '.' + c['columna']).map(DESC_TABLA_COL)
    c['descripcion'] = especifica.fillna(c['columna'].map(DESC_TABLA_COL)).fillna(c['columna'].map(DESC)) \
        .fillna(c['esquema'].map(DESC_TABLA).fillna('') + ' · ' + c['columna'].str.replace('_', ' '))
    c['rol'] = [rol_de(t, col, tp) for t, col, tp in zip(c['tabla'], c['columna'], c['tipo_dato'])]
    crudo = (c['esquema'] == 'bronze') & (c['rol'] != 'Linaje')
    c.loc[crudo, 'descripcion'] = 'Crudo (texto): ' + c.loc[crudo, 'descripcion']
    c['capa'] = c['esquema'].map({'bronze': 'Bronze', 'silver': 'Silver', 'gold': 'Gold', 'audit': 'Auditoría', 'sem': 'Semántica'})
    dic = c[['capa', 'tabla', 'columna', 'tipo_dato', 'rol', 'descripcion']].reset_index(drop=True)
    with engine.begin() as con:
        ejecutar('DELETE FROM sem.diccionario_datos', con)
        escribir(dic, 'sem.diccionario_datos', con)
    try:
        dic.to_excel(DIR_SALIDA_DIC / 'diccionario_datos.xlsx', index=False)
        destino = 'diccionario_datos.xlsx'
    except ImportError:
        dic.to_csv(DIR_SALIDA_DIC / 'diccionario_datos.csv', index=False, sep=';', encoding='utf-8-sig')
        destino = 'diccionario_datos.csv'
    return dic, destino


DIR_SALIDA_DIC = BASE / 'data' / 'salida'
DIR_SALIDA_DIC.mkdir(parents=True, exist_ok=True)
diccionario, archivo_dic = construir_diccionario()
print(f'Diccionario de datos: {diccionario.tabla.nunique()} tablas y vistas · {len(diccionario)} columnas → data/salida/{archivo_dic}')
display(diccionario.groupby(['capa', 'tabla']).size().rename('columnas').to_frame())
diccionario[diccionario.tabla.str.endswith('fact_ventas')]

## 10. Controles de calidad de la carga

Cada control queda registrado en `audit.controles_calidad` con su resultado. Si alguno crítico falla, el equipo lo ve antes de publicar el tablero.

In [ ]:
def controles():
    filas = []
    def add(control, dimension, resultado, ok):
        filas.append(dict(run_id=RUN_ID, lote_id='TODOS', control=control, dimension_calidad=dimension,
                          resultado=str(resultado), ok=int(bool(ok))))

    b = contar('bronze.pos_ventas') + contar('bronze.web_pedidos')
    s = contar('silver.ventas')
    r = int(leer("SELECT COUNT(*) AS n FROM audit.rechazos WHERE entidad = 'ventas'")['n'].iloc[0])
    add('Reconciliación ventas: Bronze = Silver + rechazados', 'Completitud', f'{b:,} = {s:,} + {r:,}', b == s + r)
    bi, si_ = contar('bronze.stock_diario'), contar('silver.inventario')
    ri = int(leer("SELECT COUNT(*) AS n FROM audit.rechazos WHERE entidad = 'inventario'")['n'].iloc[0])
    add('Reconciliación inventario: Bronze = Silver + rechazados', 'Completitud', f'{bi:,} = {si_:,} + {ri:,}', bi == si_ + ri)
    g = contar('gold.fact_ventas')
    add('Silver → Gold: todas las ventas limpias llegan al hecho', 'Completitud', f'{s:,} → {g:,}', s == g)
    dup = leer('SELECT COUNT(*) AS n FROM (SELECT nro_boleta, linea_boleta FROM gold.fact_ventas GROUP BY nro_boleta, linea_boleta HAVING COUNT(*) > 1) x')['n'].iloc[0]
    add('Unicidad del grano (nro_boleta, linea_boleta)', 'Unicidad', f'{dup} duplicados', dup == 0)
    for dim, sk in [('dim_producto', 'producto_key'), ('dim_cliente', 'cliente_key'), ('dim_sucursal', 'sucursal_key'),
                    ('dim_geografia', 'geo_key'), ('dim_fecha', 'fecha_key'), ('dim_campana', 'campana_key')]:
        h = leer(f'SELECT COUNT(*) AS n FROM gold.fact_ventas v LEFT JOIN gold.{dim} d ON d.{sk} = v.{sk} WHERE d.{sk} IS NULL')['n'].iloc[0]
        add(f'Claves huérfanas fact_ventas → {dim}', 'Integridad referencial', f'{h} huérfanas', h == 0)
    for dim, sk in [('dim_producto', 'producto_key'), ('dim_sucursal', 'sucursal_key'), ('dim_geografia', 'geo_key'), ('dim_fecha', 'fecha_key')]:
        h = leer(f'SELECT COUNT(*) AS n FROM gold.fact_inventario v LEFT JOIN gold.{dim} d ON d.{sk} = v.{sk} WHERE d.{sk} IS NULL')['n'].iloc[0]
        add(f'Claves huérfanas fact_inventario → {dim}', 'Integridad referencial', f'{h} huérfanas', h == 0)
    for dim, nk, sk in [('dim_producto', 'sku', 'producto_key'), ('dim_cliente', 'cliente_hash', 'cliente_key'), ('dim_geografia', 'codigo_comuna', 'geo_key')]:
        n = leer(f'SELECT COUNT(*) AS n FROM (SELECT {nk} FROM gold.{dim} WHERE {sk} > 0 GROUP BY {nk} HAVING SUM(es_actual) <> 1) x')['n'].iloc[0]
        add(f'SCD 2: exactamente 1 versión vigente por clave natural en {dim}', 'Consistencia', f'{n} claves con error', n == 0)
        sol = leer(f"""SELECT COUNT(*) AS n FROM gold.{dim} a JOIN gold.{dim} b ON a.{nk} = b.{nk} AND a.{sk} < b.{sk}
                       WHERE a.{sk} > 0 AND a.valido_desde < COALESCE(b.valido_hasta, '9999-12-31')
                         AND b.valido_desde < COALESCE(a.valido_hasta, '9999-12-31')""")['n'].iloc[0]
        add(f'SCD 2: vigencias sin solapamiento en {dim}', 'Consistencia', f'{sol} solapamientos', sol == 0)
    cs = leer('SELECT SUM(monto_neto) AS t FROM silver.ventas')['t'].iloc[0]
    cg = leer('SELECT SUM(monto_neto) AS t FROM gold.fact_ventas')['t'].iloc[0]
    add('Cuadre de venta neta Silver vs Gold', 'Precisión', f'{float(cs):,.0f} vs {float(cg):,.0f}', abs(float(cs) - float(cg)) < 1)
    reg = leer('SELECT COUNT(*) AS n FROM gold.fact_ventas WHERE ABS(monto_bruto - monto_descuento - monto_devuelto - monto_neto) > 0.01 OR monto_neto < 0')['n'].iloc[0]
    add('Regla única: monto_neto = bruto − descuento − devuelto y ≥ 0', 'Precisión', f'{reg} líneas fuera de regla', reg == 0)
    cdm = leer('SELECT SUM(venta_neta) AS t FROM gold.dm_comercial_ventas')['t'].iloc[0]
    cfz = leer('SELECT SUM(venta_neta) AS t FROM gold.dm_finanzas_margen')['t'].iloc[0]
    add('Data marts conformados: venta neta Comercial = Finanzas = hecho', 'Consistencia',
        f'{float(cdm):,.0f} / {float(cfz):,.0f} / {float(cg):,.0f}', abs(float(cdm) - float(cg)) < 1 and abs(float(cfz) - float(cg)) < 1)
    vt = leer('SELECT SUM(venta_neta) AS v, SUM(dias_en_quiebre) AS q FROM gold.vw_tablero_integrado')
    fq = leer('SELECT SUM(en_quiebre) AS q FROM gold.fact_inventario')['q'].iloc[0]
    add('Vista integrada sin duplicar: venta neta de la vista = fact_ventas', 'Consistencia',
        f'{float(vt.v.iloc[0]):,.0f} vs {float(cg):,.0f}', abs(float(vt.v.iloc[0]) - float(cg)) < 1)
    add('Vista integrada sin duplicar: días en quiebre de la vista = fact_inventario', 'Consistencia',
        f'{int(vt.q.iloc[0]):,} vs {int(fq):,}', int(vt.q.iloc[0]) == int(fq))
    # Coherencia de negocio entre los dos procesos: lo que recibe cada sucursal debe acompañar a lo que vende
    rep = leer("""SELECT codigo_sucursal, SUM(unidades_recibidas) AS rec, SUM(unidades_vendidas) AS vend
                   FROM gold.dm_operaciones_inventario GROUP BY codigo_sucursal""")
    rep['ratio'] = rep['rec'] / rep['vend']
    fuera = rep[(rep['ratio'] < 0.8) | (rep['ratio'] > 1.2)]
    add('Reposición coherente con la demanda: recibidas / vendidas entre 0,8 y 1,2 por sucursal', 'Consistencia',
        f'rango {rep.ratio.min():.2f}–{rep.ratio.max():.2f}' + (f' · fuera: {", ".join(fuera.codigo_sucursal)}' if len(fuera) else ''), len(fuera) == 0)
    sa = leer("SELECT COUNT(*) AS n FROM gold.fact_ventas v JOIN gold.dim_sucursal s ON s.sucursal_key = v.sucursal_key WHERE s.codigo_sucursal = 'SIN_ASIGNAR'")['n'].iloc[0]
    add('Ventas conservadas con sucursal SIN_ASIGNAR (informativo)', 'Completitud', f'{sa} líneas', True)
    hue = leer("""SELECT s.codigo_sucursal, COUNT(DISTINCT fi.fecha_key) AS dias FROM gold.fact_inventario fi
                  JOIN gold.dim_sucursal s ON s.sucursal_key = fi.sucursal_key GROUP BY s.codigo_sucursal""")
    falt = hue[hue['dias'] < hue['dias'].max()]
    add('Inventario: días sin registro por sucursal (oportunidad)', 'Oportunidad',
        '; '.join(f'{r.codigo_sucursal}: {int(hue["dias"].max() - r.dias)} días' for r in falt.itertuples()) or 'sin brechas', len(falt) == 0)
    cols = leer("""SELECT COUNT(*) AS n FROM gold.fact_ventas v JOIN gold.dim_fecha f ON f.fecha_key = v.fecha_key""")['n'].iloc[0]
    add('dim_fecha cubre todas las fechas de ventas', 'Completitud', f'{cols:,} de {g:,}', cols == g)
    # Privacidad: ninguna tabla Silver o Gold tiene columnas de RUT, nombre o correo
    if MOTOR == 'mssql':
        columnas = leer("""SELECT TABLE_SCHEMA + '.' + TABLE_NAME AS tabla, COLUMN_NAME AS columna FROM INFORMATION_SCHEMA.COLUMNS
                           WHERE TABLE_SCHEMA IN ('silver', 'gold', 'sem')""")
    else:
        tablas = leer("SELECT name FROM sqlite_master WHERE type IN ('table','view') AND (name LIKE 'silver_%' OR name LIKE 'gold_%' OR name LIKE 'sem_%')")['name']
        columnas = pd.DataFrame([(t, c) for t in tablas for c in leer(f'SELECT * FROM {t} LIMIT 0').columns], columns=['tabla', 'columna'])
    patron = r'(^|_)(rut|nombre|email|correo|telefono|direccion)(_|$)'
    pii = columnas[columnas['columna'].str.lower().str.contains(patron, regex=True) & (columnas['columna'] != 'nombre_mes')]
    add('Privacidad: sin RUT, nombre ni correo en Silver/Gold (Ley 19.628 / 21.719)', 'Seguridad',
        'ninguna columna' if pii.empty else ', '.join(pii.tabla + '.' + pii.columna), pii.empty)
    df = pd.DataFrame(filas)
    with engine.begin() as c:
        escribir(df, 'audit.controles_calidad', c)
    return df


resultado_controles = controles()
resultado_controles.assign(estado=resultado_controles['ok'].map({1: '✔ OK', 0: '✖ REVISAR'}))[['dimension_calidad', 'control', 'resultado', 'estado']]

El único control "no OK" esperado es el de **oportunidad del inventario**: la sucursal de Antofagasta no informó stock durante dos semanas de marzo.
El pipeline no inventa esos datos; lo detecta y lo deja visible para que Operaciones lo corrija en el origen.

In [ ]:
print('Registros rechazados (cuarentena) por motivo:')
leer("""SELECT entidad, motivo, COUNT(*) AS registros, MIN(detalle) AS descripcion
        FROM audit.rechazos GROUP BY entidad, motivo ORDER BY entidad, registros DESC""")

### Correcciones aplicadas por fuente y errores insertados vs detectados

Aquí se ve cómo se resolvieron los problemas que se repiten entre fuentes (por ejemplo, el mismo producto con distinto código en el POS, el e-commerce
y el sistema legado) y que lo que se insertó en la generación es lo que el pipeline detectó.

In [ ]:
print('Correcciones aplicadas en Silver, por regla y fuente (lotes 1 y 2):')
corr = leer("""SELECT regla, dimension_calidad, fuente, SUM(filas) AS filas
               FROM audit.correcciones WHERE run_id = :r AND lote_id IN ('L1-2025H1', 'L2-2025H2')
               GROUP BY regla, dimension_calidad, fuente""", r=RUN_ID)
display(corr.pivot_table(index=['dimension_calidad', 'regla'], columns='fuente', values='filas', aggfunc='sum', fill_value=0))

base = {'POS': int(leer("SELECT COUNT(*) AS n FROM bronze.pos_ventas WHERE _run_id = :r AND _archivo NOT LIKE '%legado%'", r=RUN_ID)['n'].iloc[0]),
        'E-commerce': int(leer('SELECT COUNT(*) AS n FROM bronze.web_pedidos WHERE _run_id = :r', r=RUN_ID)['n'].iloc[0])}
rech = leer("SELECT motivo, COUNT(*) AS n FROM audit.rechazos WHERE run_id = :r AND lote_id IN ('L1-2025H1', 'L2-2025H2') GROUP BY motivo", r=RUN_ID)
rech = dict(zip(rech.motivo, rech.n))
corr_fuente = corr.groupby(['regla', 'fuente'])['filas'].sum().to_dict()
FUENTE_AUDIT = {'POS': 'POS', 'E-commerce': 'ECOMMERCE', 'Legado 2018': 'LEGADO'}
def detectado(clave, fuente):
    tipo, valor = clave.split(':', 1)
    if tipo == 'rechazo':
        return int(rech.get(valor, 0))
    if tipo == 'correccion':                      # se cuenta solo en la fuente donde se insertó el error
        return int(corr_fuente.get((valor, FUENTE_AUDIT.get(fuente, fuente)), 0))
    dias = leer("""SELECT s.codigo_sucursal, COUNT(DISTINCT fi.fecha_key) AS dias FROM gold.fact_inventario fi
                    JOIN gold.dim_sucursal s ON s.sucursal_key = fi.sucursal_key GROUP BY s.codigo_sucursal""")
    return int(dias['dias'].max() - dias['dias'].min())            # días sin registro de stock
comparacion_errores = ERRORES_INSERTADOS.assign(filas_detectadas=[detectado(d, f) for d, f in zip(ERRORES_INSERTADOS['deteccion'], ERRORES_INSERTADOS['fuente'])])
comparacion_errores['pct_detectado'] = (comparacion_errores['filas_detectadas'] / comparacion_errores['fuente'].map(
    {'POS': base['POS'], 'E-commerce': base['E-commerce']}) * 100).round(2)
print(f'Filas de origen (lotes 1 y 2): POS {base["POS"]:,} · e-commerce {base["E-commerce"]:,}')
print('Nota: dos errores pueden caer en la misma fila; cada fila rechazada se cuenta solo por su primer motivo.')
comparacion_errores[['error', 'fuente', 'proporcion_insertada', 'filas_detectadas', 'pct_detectado', 'tratamiento']]

## 11. Evidencias

### 11.1 Linaje: de una cifra del data mart hasta la fila del archivo de origen

In [ ]:
ejemplo = leer("""SELECT v.nro_boleta, v.linea_boleta, v.monto_bruto, v.monto_descuento, v.monto_devuelto, v.monto_neto,
                         v.archivo_origen, v.fila_origen, v.lote_id, v.run_id
                  FROM gold.fact_ventas v WHERE v.devolucion = 1 AND v.archivo_origen LIKE 'ventas/pos_%' ORDER BY v.nro_boleta""").head(1)
print('1) Línea en gold.fact_ventas:'); display(ejemplo)
nro, lin = ejemplo.nro_boleta.iloc[0], int(ejemplo.linea_boleta.iloc[0])
print('2) La misma línea en silver.ventas (limpia):')
display(leer('SELECT nro_boleta, linea_boleta, fecha, sku, codigo_sucursal, canal, cantidad, precio_unitario, cantidad_devuelta, monto_neto FROM silver.ventas WHERE nro_boleta = :n AND linea_boleta = :l', n=nro, l=lin))
print('3) La fila original en bronze.pos_ventas (tal como llegó del POS):')
display(leer('SELECT nro_boleta, linea, fecha, sku, codigo_sucursal, rut_cliente, canal, cantidad, precio_unitario, descuento, _archivo, _fila FROM bronze.pos_ventas WHERE _archivo = :a AND _fila = :f',
             a=ejemplo.archivo_origen.iloc[0], f=int(ejemplo.fila_origen.iloc[0])))
print('4) La devolución que explica el monto devuelto:')
display(leer('SELECT nro_boleta, linea, fecha_devolucion, cantidad_devuelta, motivo, _archivo, _fila FROM bronze.devoluciones WHERE nro_boleta = :n AND linea = :l', n=nro, l=str(lin)))

### 11.2 Efecto de las técnicas SCD

**SCD 2 · Producto recategorizado.** En el lote 2 los audífonos y parlantes (subcategoría *Audio*) pasan de *Accesorios* a *Tecnología*, y *Electrohogar* se consolida en *Hogar*.
Con SCD 2, las ventas de enero a junio siguen en Accesorios y las de julio en adelante suman en Tecnología. La historia no se reescribe.

In [ ]:
display(leer("""SELECT producto_key, sku, producto, subcategoria, categoria, costo_unitario, valido_desde, valido_hasta, es_actual
                FROM gold.dim_producto WHERE sku IN ('AU-002', 'MI-016', 'TV-003') ORDER BY sku, producto_key"""))
leer("""SELECT CASE WHEN f.mes <= 6 THEN '1er semestre' ELSE '2do semestre' END AS periodo,
               p.categoria AS categoria_en_la_fecha_de_venta,
               SUM(v.monto_neto) AS venta_neta_audio
        FROM gold.fact_ventas v
        JOIN gold.dim_producto p ON p.producto_key = v.producto_key
        JOIN gold.dim_fecha f    ON f.fecha_key = v.fecha_key
        WHERE p.subcategoria = 'Audio' AND f.anio = 2025
        GROUP BY CASE WHEN f.mes <= 6 THEN '1er semestre' ELSE '2do semestre' END, p.categoria
        ORDER BY periodo""")

**SCD 2 · Dimensión espacial.** Las ventas de Chillán de 2018 quedan en la Región del Biobío si ocurrieron antes del 6 de septiembre de 2018,
y en la Región de Ñuble desde esa fecha (Ley 21.033). El *lookup point-in-time* asigna la versión correcta sin intervención manual.

In [ ]:
display(leer("SELECT geo_key, codigo_comuna, codigo_cut_vigente, comuna, provincia, region, valido_desde, valido_hasta, es_actual FROM gold.dim_geografia WHERE comuna = 'Chillán'"))
leer("""SELECT g.region AS region_en_la_fecha, gact.region AS region_actual, MIN(f.fecha) AS desde, MAX(f.fecha) AS hasta,
               COUNT(*) AS lineas, SUM(v.monto_neto) AS venta_neta
        FROM gold.fact_ventas v
        JOIN gold.dim_geografia g    ON g.geo_key = v.geo_key
        JOIN gold.dim_geografia gact ON gact.codigo_comuna = g.codigo_comuna AND gact.es_actual = 1
        JOIN gold.dim_fecha f        ON f.fecha_key = v.fecha_key
        WHERE f.anio = 2018 GROUP BY g.region, gact.region ORDER BY desde""")

**SCD 2 · Cliente.** Un cliente que sube de segmento tiene dos versiones; cada compra queda asociada al segmento que tenía ese día.

In [ ]:
cli_cambio = leer("""SELECT cliente_hash FROM gold.dim_cliente WHERE cliente_key > 0 GROUP BY cliente_hash HAVING COUNT(*) > 1""")['cliente_hash']
h = None
for cand in cli_cambio:
    n = leer("""SELECT COUNT(DISTINCT v.cliente_key) AS n FROM gold.fact_ventas v JOIN gold.dim_cliente c ON c.cliente_key = v.cliente_key
                WHERE c.cliente_hash = :h""", h=cand)['n'].iloc[0]
    if n == 2:
        h = cand
        break
display(leer('SELECT cliente_key, cliente_hash, segmento, comuna, region, valido_desde, valido_hasta, es_actual FROM gold.dim_cliente WHERE cliente_hash = :h', h=h)
        .assign(cliente_hash=lambda d: d.cliente_hash.str[:16] + '…'))
leer("""SELECT f.fecha, v.nro_boleta, c.segmento AS segmento_en_la_compra, v.monto_neto FROM gold.fact_ventas v
        JOIN gold.dim_cliente c ON c.cliente_key = v.cliente_key JOIN gold.dim_fecha f ON f.fecha_key = v.fecha_key
        WHERE c.cliente_hash = :h ORDER BY f.fecha""", h=h)

**SCD 1 y SCD 3 · Sucursal.** "Maipu" se corrigió a "Maipú" sobrescribiendo (no interesa conservar un error de tipeo).
Concepción y Chillán cambiaron de zona: SCD 3 guarda la zona actual y la anterior en la misma fila, lo que permite comparar ambas organizaciones en un mismo reporte.

In [ ]:
display(leer("""SELECT sucursal_key, codigo_sucursal, sucursal, tipo_sucursal, comuna, region, zona_comercial_actual, zona_comercial_anterior, fecha_cambio_zona
                FROM gold.dim_sucursal ORDER BY sucursal_key"""))
leer("""SELECT COALESCE(s.zona_comercial_anterior, s.zona_comercial_actual) AS zona_organizacion_anterior,
               s.zona_comercial_actual AS zona_organizacion_actual, SUM(v.monto_neto) AS venta_neta_2025
        FROM gold.fact_ventas v JOIN gold.dim_sucursal s ON s.sucursal_key = v.sucursal_key JOIN gold.dim_fecha f ON f.fecha_key = v.fecha_key
        WHERE f.anio = 2025 AND s.sucursal_key > 0
        GROUP BY COALESCE(s.zona_comercial_anterior, s.zona_comercial_actual), s.zona_comercial_actual ORDER BY venta_neta_2025 DESC""")

**Esquema estrella.** Ninguna dimensión depende de otra: `dim_geografia` se conecta **directo** a los dos hechos mediante `geo_key`
(versión SCD 2 vigente en la fecha del hecho), y `dim_sucursal` guarda comuna y región como atributos planos.
La consulta muestra el stock por región usando solo `fact_inventario → dim_geografia` (un JOIN por dimensión, sin pasar por la sucursal).

In [ ]:
leer("""SELECT g.region, COUNT(DISTINCT i.sucursal_key) AS sucursales, AVG(CAST(i.stock_cierre AS FLOAT)) AS stock_promedio_diario,
               SUM(i.en_quiebre) AS dias_producto_en_quiebre
        FROM gold.fact_inventario i JOIN gold.dim_geografia g ON g.geo_key = i.geo_key
        WHERE i.sucursal_key > 0 GROUP BY g.region ORDER BY g.region""")

### 11.3 Idempotencia: volver a ejecutar un lote no duplica datos

In [ ]:
antes = {t: contar(t) for t in ['bronze.pos_ventas', 'silver.ventas', 'gold.fact_ventas', 'gold.fact_inventario', 'gold.dim_producto', 'gold.dim_cliente']}
venta_antes = float(leer('SELECT SUM(monto_neto) AS t FROM gold.fact_ventas')['t'].iloc[0])
ejecutar_lote('L2-2025H2')            # se vuelve a procesar el lote 2 completo
construir_data_marts()
despues = {t: contar(t) for t in antes}
venta_despues = float(leer('SELECT SUM(monto_neto) AS t FROM gold.fact_ventas')['t'].iloc[0])
comparacion = pd.DataFrame({'antes': antes, 'despues': despues}, dtype=float)
comparacion.loc['venta neta total'] = [venta_antes, venta_despues]
comparacion['sin_duplicar'] = ((comparacion['antes'] - comparacion['despues']).abs() < 1).map({True: '✔', False: '✖'})
comparacion

Ninguna capa se duplica: **Bronze** reemplaza la copia de un archivo si ya se había cargado (por `_archivo`), **Silver** y los **hechos**
reemplazan las filas del mismo `lote_id`, y las dimensiones SCD 2 no crean versiones nuevas porque los atributos no cambiaron.
El registro de cada ingesta queda en `audit.etl_log` (el mensaje indica cuántas filas se reemplazaron).

### 11.4 Manejo de errores: un archivo corrupto no deja cargas a medias

Se simula que el POS envía un archivo con una columna faltante. El pipeline detecta el problema en Bronze, registra el error y **revierte el lote completo**.

In [ ]:
corrupto = pd.read_csv(DIR_FUENTE / 'ventas' / 'pos_ventas_2025_S2.csv', dtype=str).head(500).drop(columns=['precio_unitario'])
corrupto.to_csv(DIR_FUENTE / 'ventas' / 'pos_ventas_2026_S1.csv', index=False)
FUENTES_LOTE['L3-PRUEBA-ERROR'] = {**FUENTES_LOTE['L2-2025H2'], 'ventas_pos': ['ventas/pos_ventas_2026_S1.csv'],
                                   'vigencia_maestros': date(2026, 1, 1)}
filas_antes = contar('bronze.pos_ventas')
try:
    ejecutar_lote('L3-PRUEBA-ERROR')
except Exception as e:
    print('Error capturado por el orquestador →', type(e).__name__, ':', e)
print(f'Filas en bronze.pos_ventas antes: {filas_antes:,} · después: {contar("bronze.pos_ventas"):,}  (no quedó nada a medias)')
(DIR_FUENTE / 'ventas' / 'pos_ventas_2026_S1.csv').unlink()
del FUENTES_LOTE['L3-PRUEBA-ERROR']
leer("SELECT lote_id, etapa, objeto, estado, mensaje, inicio FROM audit.etl_log WHERE lote_id = 'L3-PRUEBA-ERROR' ORDER BY inicio")

### 11.5 Validación contra la verdad del negocio

Como los datos fueron generados, se conoce la venta neta "real". Se compara con lo que entrega el Data Warehouse después de limpiar.
La diferencia debe corresponder exactamente a las líneas rechazadas por errores del origen (registradas en `audit.rechazos`); los duplicados no cuentan porque eran copias.

In [ ]:
dev_total = pd.concat([pd.read_csv(DIR_FUENTE / 'ventas' / f'devoluciones_2025_S{k}.csv') for k in (1, 2)])
v = VERDAD.merge(dev_total.groupby(['nro_boleta', 'linea'], as_index=False)['cantidad_devuelta'].sum(), on=['nro_boleta', 'linea'], how='left')
v['cantidad_devuelta'] = np.where(v['estado'].isin(['ANULADA', 'DEVUELTA']), v['cantidad'], v['cantidad_devuelta'].fillna(0))
bruto = v['cantidad'] * v['precio_unitario']
v['neto'] = bruto - v['descuento'] - (bruto - v['descuento']) / v['cantidad'] * v['cantidad_devuelta']
real = v['neto'].sum()
dw = float(leer("SELECT SUM(v.monto_neto) AS t FROM gold.fact_ventas v JOIN gold.dim_fecha f ON f.fecha_key = v.fecha_key WHERE f.anio = 2025")['t'].iloc[0])
cargadas = leer("SELECT nro_boleta, linea_boleta FROM silver.ventas WHERE origen <> 'LEGADO'")
v = v.merge(cargadas.rename(columns={'linea_boleta': 'linea'}).assign(en_dw=1), on=['nro_boleta', 'linea'], how='left')
no_cargadas = v[v['en_dw'].isna()]
print(f'Venta neta 2025 según el negocio (datos generados) : ${real:,.0f}')
print(f'Venta neta 2025 en el Data Warehouse               : ${dw:,.0f}')
print(f'Diferencia                                         : ${real - dw:,.0f}  ({(real - dw) / real:.3%})')
print(f'Venta de las {len(no_cargadas)} líneas rechazadas en Silver : ${no_cargadas["neto"].sum():,.0f}')
print('→ La diferencia se explica completa por líneas rechazadas con motivo registrado (fecha imposible, cantidad o precio inválido, SKU inexistente).'
      if abs((real - dw) - no_cargadas['neto'].sum()) < 1 else '→ REVISAR: la diferencia no se explica solo por rechazos.')

### 11.6 Consultas de negocio que cruzan los data marts

In [ ]:
import matplotlib.pyplot as plt

mensual = leer("""SELECT mes, canal, SUM(venta_neta) AS venta_neta FROM gold.dm_comercial_ventas WHERE anio = 2025 GROUP BY mes, canal""")
pv = mensual.pivot(index='mes', columns='canal', values='venta_neta') / 1e6
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
pv.plot(ax=ax[0], marker='o', color=['#1f4a73', '#2f8c99'])
ax[0].set_title('Venta neta mensual 2025 por canal (MM$)')
ax[0].set_xlabel('Mes'); ax[0].set_ylabel('MM$'); ax[0].grid(alpha=0.3); ax[0].set_xticks(range(1, 13))
mg = leer("""SELECT categoria, SUM(margen_bruto) * 100.0 / SUM(venta_neta) AS margen_pct FROM gold.dm_finanzas_margen WHERE anio = 2025 GROUP BY categoria ORDER BY margen_pct""")
ax[1].barh(mg['categoria'], mg['margen_pct'], color='#2f8c99')
ax[1].set_title('Margen bruto % por categoría (2025)'); ax[1].grid(alpha=0.3, axis='x')
for i, val in enumerate(mg['margen_pct']):
    ax[1].text(val + 0.3, i, f'{val:.1f}%', va='center')
plt.tight_layout(); plt.show()

print('Pregunta integrada: ¿qué productos tuvieron más días sin stock justo en los meses con campañas, y cuánta venta movieron?')
leer("""SELECT t.sku, MAX(t.producto) AS producto, SUM(t.dias_en_quiebre) AS dias_en_quiebre,
               SUM(t.venta_neta_con_campana) AS venta_con_campana, SUM(t.venta_neta) AS venta_neta
        FROM gold.vw_tablero_integrado t
        WHERE t.anio = 2025 AND t.mes IN (3, 6, 7, 9, 11, 12)
        GROUP BY t.sku ORDER BY dias_en_quiebre DESC""").head(8)

### 11.7 Ejemplo de consulta para la IA generativa

Pregunta en lenguaje natural: *"¿Cuál fue la venta neta por región en el segundo semestre, solo e-commerce?"*.
El SQL que debe generar Copilot o Vanna.ai usa la vista semántica y la definición oficial del KPI; el checklist de validación (Anexo I de la EP1) se aplica sobre este resultado.

In [ ]:
sql_ia = """SELECT region_venta, SUM(venta_neta) AS venta_neta
FROM gold.vw_ia_ventas
WHERE anio = 2025 AND mes BETWEEN 7 AND 12 AND canal = 'E-commerce'
GROUP BY region_venta ORDER BY venta_neta DESC"""
print(sql_ia)
r = leer(sql_ia)
total_ref = float(leer("SELECT SUM(venta_neta) AS t FROM gold.dm_comercial_ventas WHERE anio = 2025 AND mes >= 7 AND canal = 'E-commerce'")['t'].iloc[0])
print(f'Validación (checklist): total de la respuesta {r.venta_neta.sum():,.0f} vs cálculo de referencia en el data mart {total_ref:,.0f} → '
      f'{"OK" if abs(r.venta_neta.sum() - total_ref) < 1 else "REVISAR"}')
r

## 12. Dataset limpio final

Todo lo que entró sucio y repartido en **18 archivos** de distintos sistemas sale como **un solo dataset limpio e integrado**:
tiendas, e-commerce, App y el histórico 2018 juntos, con fechas, SKU y canales homologados, la venta neta calculada con la regla única,
sin RUT ni nombres, y con la columna de linaje que indica de qué archivo y fila vino cada línea.
Se guarda en `data/salida/ventas_limpias_integradas.csv` (separador `;` y coma decimal, para abrirlo directo en Excel).

In [ ]:
DIR_SALIDA = BASE / 'data' / 'salida'
DIR_SALIDA.mkdir(parents=True, exist_ok=True)
limpio = leer("""
SELECT v.nro_boleta, v.linea_boleta, f.fecha, f.anio, f.mes, f.semana, s.codigo_sucursal, s.sucursal, s.tipo_sucursal,
       s.zona_comercial_actual AS zona_comercial, g.comuna AS comuna_venta, g.region AS region_venta, c.canal,
       p.sku, p.producto, p.categoria, p.subcategoria, p.marca, cp.campana, cl.segmento AS segmento_cliente,
       v.cantidad, v.precio_unitario, v.monto_bruto, v.monto_descuento, v.cantidad_devuelta, v.monto_devuelto,
       v.monto_neto, v.costo_neto, v.margen_bruto, v.archivo_origen, v.fila_origen
FROM gold.fact_ventas v
JOIN gold.dim_fecha f     ON f.fecha_key = v.fecha_key
JOIN gold.dim_sucursal s  ON s.sucursal_key = v.sucursal_key
JOIN gold.dim_geografia g ON g.geo_key = v.geo_key
JOIN gold.dim_canal c     ON c.canal_key = v.canal_key
JOIN gold.dim_producto p  ON p.producto_key = v.producto_key
JOIN gold.dim_campana cp  ON cp.campana_key = v.campana_key
JOIN gold.dim_cliente cl  ON cl.cliente_key = v.cliente_key""").sort_values(['fecha', 'nro_boleta', 'linea_boleta'])
ruta_limpio = DIR_SALIDA / 'ventas_limpias_integradas.csv'
limpio.to_csv(ruta_limpio, sep=';', decimal=',', index=False, encoding='utf-8-sig')

entrada = pd.DataFrame(json.loads((DIR_FUENTE / 'manifest.json').read_text(encoding='utf-8'))['archivos'])
print(f'ENTRADA: {len(entrada)} archivos sucios de distintos sistemas · {entrada.filas.sum():,} filas en total')
print(f'SALIDA : 1 archivo limpio e integrado → {ruta_limpio.relative_to(BASE)} · {len(limpio):,} líneas de venta')
print(f'         venta neta total: ${limpio.monto_neto.sum():,.0f} (2025 + histórico 2018; cuadra con gold.fact_ventas)')
display(limpio.groupby('canal').agg(lineas=('nro_boleta', 'size'), venta_neta=('monto_neto', 'sum')))
limpio.head(8)

## 13. Resumen de la ejecución y exportación

In [ ]:
resumen = leer("""SELECT lote_id, etapa, objeto, filas_entrada, filas_salida, filas_rechazadas, estado, segundos
                  FROM audit.etl_log WHERE run_id = :r ORDER BY inicio""", r=RUN_ID)
print(f'Etapas ejecutadas: {len(resumen)} · con error: {(resumen.estado == "ERROR").sum()} (la del lote de prueba)')
display(resumen.head(40))

try:                                   # Exportación opcional de los data marts a Parquet (consumo desde Power BI o Python)
    for t in TABLAS_DM:
        leer(f'SELECT * FROM {t}').to_parquet(DIR_GOLD / f'{t.split(".")[1]}.parquet', index=False)
    print('Data marts exportados a', DIR_GOLD)
except ImportError:
    print('PyArrow no instalado: se omite la exportación a Parquet.')

# Huella del resultado: resume las cifras finales en un código. Con los mismos archivos de entrada debe ser idéntica en cada ejecución.
metricas = {
    'fact_ventas_filas': contar('gold.fact_ventas'), 'fact_inventario_filas': contar('gold.fact_inventario'),
    'venta_neta': round(float(leer('SELECT SUM(monto_neto) AS t FROM gold.fact_ventas')['t'].iloc[0])),
    'margen_bruto': round(float(leer('SELECT SUM(margen_bruto) AS t FROM gold.fact_ventas')['t'].iloc[0])),
    'dias_en_quiebre': int(leer('SELECT SUM(en_quiebre) AS t FROM gold.fact_inventario')['t'].iloc[0]),
    'dim_producto': contar('gold.dim_producto'), 'dim_cliente': contar('gold.dim_cliente'), 'dim_geografia': contar('gold.dim_geografia'),
    'rechazos': int(leer("SELECT COUNT(*) AS n FROM (SELECT DISTINCT archivo, fila, motivo FROM audit.rechazos WHERE run_id = :r) x", r=RUN_ID)['n'].iloc[0]),
}
huella = hashlib.sha256(json.dumps(metricas, sort_keys=True).encode()).hexdigest()[:16]
display(pd.Series(metricas, name='valor').to_frame())
print(f'HUELLA DEL RESULTADO: {huella}  ← debe ser la misma en cada ejecución con el mismo dataset (reproducibilidad)')
log.info('Huella del resultado %s', huella)
log.info('Fin de la ejecución %s. Log en %s', RUN_ID, DIR_LOGS / f'pipeline_{RUN_ID}.log')